# Panel Builder – S&P 500 (2015–2025)

Este notebook construye el panel final firm-month utilizado en las estimaciones econométricas.

Input:
- data/raw/ (insumos Refinitiv descargados por ETL)

Output principal:
- data/clean/panel_master.parquet

Outputs auxiliares:
- data/clean/data_dictionary.csv
- data/clean/lineage_log.csv
- outputs/logs/panel_build_*.log
- outputs/qa/qa_report.md

El notebook NO estima modelos.

## 1) Inicialización del entorno de trabajo y verificación preliminar de insumos

### Objetivo

Esta etapa inicial define el entorno operativo del notebook de construcción del panel y verifica la disponibilidad de los principales archivos intermedios requeridos para el proceso de integración de datos.

Dado que este notebook consolida múltiples bloques previamente descargados y transformados, resulta fundamental establecer desde el inicio:

- una estructura robusta de paths;
- un sistema de logging reproducible;
- y controles preliminares sobre archivos críticos de entrada.

### Alcance

En particular, esta sección cumple cuatro funciones:

1. importar las librerías necesarias para el armado del panel;
2. detectar automáticamente el directorio raíz del proyecto;
3. definir la estructura de carpetas de trabajo y asegurar su existencia;
4. inspeccionar archivos intermedios relevantes, con énfasis en las variables de market power.

### Detección del directorio raíz

Para asegurar portabilidad, el notebook detecta el repositorio buscando marcadores estándar del proyecto (por ejemplo, `requirements.txt`, `.git` o `README.md`) hacia arriba desde el directorio actual. Esto permite ejecutar el notebook tanto desde la carpeta de notebooks como desde el root del repositorio, evitando dependencias rígidas de ubicación.

### Verificación preliminar de market power

Antes de construir el panel final, se inspecciona el archivo `market_power_monthly.parquet`, verificando:

- nombres de columnas disponibles;
- presencia de variables de market share;
- y proporción de valores faltantes en dichas variables.

Este control es importante porque las proxies de market power constituyen un bloque explicativo relevante dentro de la estrategia empírica, y su cobertura efectiva puede afectar tanto la comparabilidad entre modelos como el tamaño de muestra final.

### Logging y trazabilidad

Adicionalmente, se inicializa un archivo de log con timestamp, lo que permite registrar de manera ordenada la ejecución del notebook y facilitar auditorías, debugging posterior y trazabilidad reproducible del proceso de construcción del panel.

### Alcance interpretativo

Estos chequeos no modifican la lógica de construcción del panel, pero sí aseguran que los insumos estén disponibles y que el entorno de trabajo sea consistente antes de comenzar las etapas de merge, limpieza y validación estructural.

In [ ]:
# ==========================================================
# 1.1 IMPORTACIÓN DE LIBRERÍAS
# ==========================================================

from datetime import datetime
from pathlib import Path
from typing import Dict, List

import logging
import numpy as np
import pandas as pd

In [ ]:
# ==========================================================
# 1.2 DETECCIÓN DEL ROOT Y CONFIGURACIÓN DE PATHS
# ==========================================================

from pathlib import Path


def detect_project_root(markers=("requirements.txt", ".git", "README.md")) -> Path:
    """
    Detecta el directorio raíz del proyecto buscando marcadores estándar
    hacia arriba desde el directorio actual.
    Funciona tanto si el notebook corre desde /notebooks como desde el root.
    """
    here = Path.cwd().resolve()

    for parent in [here] + list(here.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent

    return here  # fallback


# ----------------------------------------------------------
# root del proyecto
# ----------------------------------------------------------
PROJECT_ROOT = detect_project_root()

RAW = PROJECT_ROOT / "data" / "raw"
CLEAN = PROJECT_ROOT / "data" / "clean"
INTERIM = PROJECT_ROOT / "data" / "intermediate"

OUTPUTS = PROJECT_ROOT / "outputs"
LOGS = OUTPUTS / "logs"
QA = OUTPUTS / "qa"

PANEL_PATH = CLEAN / "panel_master.parquet"


# ----------------------------------------------------------
# creación de directorios necesarios
# ----------------------------------------------------------
for p in [RAW, INTERIM, CLEAN, LOGS, QA]:
    p.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# chequeo básico de paths
# ----------------------------------------------------------
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW         =", RAW)
print("CLEAN       =", CLEAN)
print("OUTPUTS     =", OUTPUTS)


# ----------------------------------------------------------
# inspección preliminar de market power monthly
# ----------------------------------------------------------
mp = pd.read_parquet(INTERIM / "market_power_monthly.parquet")

print(mp.columns.tolist())

share_cols = [c for c in mp.columns if "market_share" in c]

print("\nVariables market share en market_power_monthly:")
print(share_cols)

print("\nMissingness:")
print(mp[share_cols].isna().mean().sort_values(ascending=False))

In [ ]:
# ==========================================================
# 1.3 CONFIGURACIÓN DE LOGGING Y AUDITORÍA DE INSUMOS
# ==========================================================

log_file = LOGS / f"panel_build_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

logging.info("Inicio Panel Builder")


# ----------------------------------------------------------
# verificación de directorios de trabajo
# ----------------------------------------------------------
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW         =", RAW, RAW.exists())
print("INTERIM     =", INTERIM, INTERIM.exists())
print("CLEAN       =", CLEAN, CLEAN.exists())


# ----------------------------------------------------------
# inventario rápido de archivos disponibles
# ----------------------------------------------------------
for folder in [RAW, INTERIM, CLEAN]:

    print(f"\n--- {folder} ---")

    if folder.exists():
        for f in sorted(folder.glob("*")):
            print(f.name)

## 4) Carga y auditoría preliminar de insumos base

### Objetivo

Esta sección carga los principales datasets necesarios para la construcción del panel maestro y realiza una auditoría preliminar de su estructura, cobertura e integridad básica.

Dado que el panel final surge de la combinación de múltiples fuentes con distinta granularidad y origen, este paso busca asegurar que los insumos críticos estén disponibles, no se encuentren vacíos y presenten un rango temporal razonable antes de avanzar con merges y transformaciones posteriores.

### Insumos principales

Los bloques de datos cargados en esta etapa incluyen:

- spreads de bonos;
- retornos diarios de equity;
- series de mercado;
- fundamentals trimestrales;
- metadata corporativa.

En la lógica general del notebook, estos constituyen los insumos mínimos para construir el panel firma–mes y derivar posteriormente las variables explicativas y de control.

### Estrategia de carga

La lectura de archivos se implementa de forma flexible, buscando cada dataset en un orden jerárquico de carpetas:

1. `data/clean`
2. `data/intermediate`
3. `data/raw`

Este criterio permite priorizar versiones ya procesadas cuando existen, preservando al mismo tiempo compatibilidad con corridas parciales del pipeline.

### Auditoría preliminar

Para cada dataset cargado se realizan controles descriptivos básicos:

- verificación de que el archivo no esté vacío;
- identificación de columnas clave potenciales;
- inspección del rango temporal observado;
- conteo de identificadores únicos y valores faltantes en variables identificadoras;
- visualización de las primeras observaciones.

Estos chequeos no sustituyen las validaciones específicas de construcción del panel, pero sí permiten detectar tempranamente problemas de cobertura, naming o estructura que luego afectarían la integración entre bloques.

### Alcance interpretativo

La función de esta sección es operativa y documental. No transforma los datos ni altera su contenido, sino que valida condiciones mínimas de uso para las etapas posteriores de limpieza, armonización de identificadores y merge de datasets.

In [ ]:
# ==========================================================
# 4 CARGA DE INSUMOS BASE
# ==========================================================

from pathlib import Path
from typing import Dict

import logging
import pandas as pd


# ==========================================================
# 4.1 HELPERS DE CARGA
# ==========================================================

def read_dataset(filename: str, folders: list[Path]) -> pd.DataFrame:
    """
    Busca un archivo en múltiples carpetas y lo lee según su extensión.
    Soporta archivos .parquet y .csv.
    """
    tried = []

    for folder in folders:
        path = folder / filename
        tried.append(str(path))

        if path.exists():
            print(f"Leyendo: {path}")

            if path.suffix == ".parquet":
                return pd.read_parquet(path)

            elif path.suffix == ".csv":
                return pd.read_csv(path)

    raise FileNotFoundError(
        f"No se encontró {filename} en ninguna de estas ubicaciones:\n" + "\n".join(tried)
    )


def audit_df(df: pd.DataFrame, name: str):
    """
    Auditoría preliminar de estructura, cobertura temporal e identificadores.
    """
    print(f"\n{'=' * 80}")
    print(f"AUDIT: {name}")
    print(f"{'=' * 80}")

    print("shape:", df.shape)

    key_candidates = [
        "issuer",
        "ticker",
        "ric",
        "ric_base",
        "isin",
        "cusip",
        "date",
        "period_end",
    ]

    present_keys = [c for c in key_candidates if c in df.columns]
    print("key-like cols:", present_keys)

    # ------------------------------------------------------
    # rango temporal
    # ------------------------------------------------------
    for dcol in ["date", "period_end"]:
        if dcol in df.columns:
            s = pd.to_datetime(df[dcol], errors="coerce")
            print(f"{dcol}: min={s.min()} max={s.max()} nulls={s.isna().sum()}")

    # ------------------------------------------------------
    # identificadores
    # ------------------------------------------------------
    for c in ["issuer", "ticker", "ric", "ric_base", "isin", "cusip"]:
        if c in df.columns:
            print(f"{c}: unique={df[c].nunique(dropna=True)} nulls={df[c].isna().sum()}")

    # ------------------------------------------------------
    # vista rápida
    # ------------------------------------------------------
    print("\nhead:")
    print(df.head(2))


# ==========================================================
# 4.2 CARGA DE INPUTS BASE
# ==========================================================

def load_base_inputs() -> Dict[str, pd.DataFrame]:
    """
    Carga los insumos principales para la construcción del panel.
    Orden de búsqueda: clean -> intermediate -> raw
    """
    data = {}

    search_order = [CLEAN, INTERIM, RAW]

    data["spreads_raw"] = read_dataset("oas_spreads_monthly_bond.parquet", search_order)
    data["equity_daily"] = read_dataset("equity_returns_daily.parquet", search_order)
    data["market_daily"] = read_dataset("market_prices_daily.parquet", search_order)
    data["fundamentals_q"] = read_dataset("fundamentals_extended_q.parquet", search_order)
    data["company_metadata"] = read_dataset("company_metadata.parquet", search_order)

    for name, df in data.items():
        assert not df.empty, f"{name} vacío"

        logging.info(f"{name} cargado: {df.shape}")
        print(f"✅ {name}: {df.shape}")

        audit_df(df, name)

    return data


# ==========================================================
# 4.3 EJECUCIÓN
# ==========================================================

raw_data = load_base_inputs()

In [ ]:
# ==========================================================
# 4.4 AUDITORÍA DETALLADA DE CLAVES E IDENTIFICADORES
# ==========================================================

def audit_keys_detail(df: pd.DataFrame, name: str):
    """
    Auditoría detallada de estructura, identificadores, fechas y duplicados.
    """
    print(f"\n{'#' * 100}")
    print(f"DETAIL AUDIT: {name}")
    print(f"{'#' * 100}")

    print("shape:", df.shape)

    print("\ncolumns:")
    print(list(df.columns))

    # ------------------------------------------------------
    # auditoría de identificadores
    # ------------------------------------------------------
    for c in [
        "issuer",
        "Issuer",
        "ticker",
        "Ticker",
        "ric",
        "ric_base",
        "bond_ric",
        "Instrument",
        "isin",
        "cusip",
    ]:
        if c in df.columns:
            print(f"\nColumn: {c}")
            print("  nulls :", df[c].isna().sum())
            print("  unique:", df[c].nunique(dropna=True))
            print("  sample:", df[c].dropna().astype(str).head(5).tolist())

    # ------------------------------------------------------
    # auditoría de fechas
    # ------------------------------------------------------
    for dcol in ["date", "period_end"]:
        if dcol in df.columns:
            s = pd.to_datetime(df[dcol], errors="coerce")
            print(f"\nDate col: {dcol}")
            print("  min   :", s.min())
            print("  max   :", s.max())
            print("  nulls :", s.isna().sum())

    # ------------------------------------------------------
    # chequeos de duplicados bajo claves razonables
    # ------------------------------------------------------
    candidate_keys = [
        ["Issuer", "date"],
        ["Ticker", "date"],
        ["ric", "date"],
        ["ticker", "date"],
        ["issuer", "date"],
        ["bond_ric", "date"],
        ["Instrument", "date"],
        ["ticker", "period_end"],
        ["issuer", "period_end"],
    ]

    print("\nDuplicate checks:")
    for keys in candidate_keys:
        if all(k in df.columns for k in keys):
            dup = df.duplicated(keys).sum()
            print(f"  duplicates by {keys}: {dup}")

    print("\nhead:")
    print(df.head(3))


# ----------------------------------------------------------
# ejecución de auditorías detalladas
# ----------------------------------------------------------
audit_keys_detail(raw_data["spreads_raw"], "spreads_raw")
audit_keys_detail(raw_data["equity_daily"], "equity_daily")
audit_keys_detail(raw_data["fundamentals_q"], "fundamentals_q")
audit_keys_detail(raw_data["company_metadata"], "company_metadata")

In [ ]:
# ==========================================================
# 4.5 CHEQUEOS ESPECÍFICOS DEL BLOQUE DE SPREADS
# ==========================================================

df = raw_data["spreads_raw"].copy()

# ----------------------------------------------------------
# 1) cantidad de bonos por issuer-date
# ----------------------------------------------------------
tmp = df.groupby(["Issuer", "date"])["bond_ric"].nunique()

print("\nBonos por issuer-date:")
print(tmp.describe())

# ----------------------------------------------------------
# 2) duplicados exactos por bond_ric-date
# ----------------------------------------------------------
print("\nDuplicados por bond_ric-date:")
print(df.duplicated(["bond_ric", "date"]).sum())

# ----------------------------------------------------------
# 3) fechas faltantes
# ----------------------------------------------------------
print("\nNaT en date:", df["date"].isna().sum())

# ----------------------------------------------------------
# 4) muestra de fechas observadas
# ----------------------------------------------------------
print("\nFechas únicas (sample):")
print(sorted(df["date"].dropna().unique())[:10])

## 4.1) Construcción de `bonds_monthly_spreads` a partir de insumos de spreads

### Objetivo

Este bloque construye el dataset operativo `bonds_monthly_spreads`, que constituye la base del bloque de spreads de crédito en el panel final.

El objetivo principal es armonizar la estructura de los datos provenientes del notebook de descargas con el formato requerido por el notebook de construcción del panel, garantizando compatibilidad sin necesidad de reestructurar el pipeline completo.

---

### Enfoque general

El procedimiento toma como insumo principal el archivo de spreads de bonos a frecuencia mensual y realiza una serie de transformaciones orientadas a:

- estandarizar identificadores;
- unificar nombres de variables;
- garantizar consistencia temporal;
- y completar columnas requeridas por el pipeline.

El resultado es un dataset limpio a nivel bono–mes, listo para ser utilizado en la agregación a nivel firma–mes.

---

### Construcción de la variable de spread

El spread de crédito se define como:

$$
spread_{i,t} = OAS_{i,t}
$$

donde $OAS_{i,t}$ corresponde al Option-Adjusted Spread del bono $i$ en el período $t$.

Esta elección implica que el spread refleja directamente el componente de riesgo crediticio ajustado por opciones embebidas, en línea con la práctica estándar en la literatura empírica de credit spreads.

---

### Armonización de variables

Se realiza una normalización de columnas basada en posibles variantes observadas en los datos originales:

- `issuer` y `ticker` se construyen a partir de múltiples candidatos (`Issuer`, `Ticker`, etc.);
- `bond_ric` se define como identificador único del bono;
- `date` se transforma a fin de mes para garantizar consistencia temporal.

Asimismo, se crean columnas requeridas por el pipeline aunque no estén disponibles en esta etapa:

- `ytm`
- `residual_maturity_years`
- `amount_issued`

Estas variables se inicializan como faltantes (`NaN`) cuando no pueden ser inferidas a partir de los insumos.

---

### Limpieza y validación

El dataset resultante se somete a una serie de filtros y controles:

- eliminación de observaciones con identificadores o spreads faltantes;
- eliminación de duplicados a nivel `bond_ric-date`;
- ordenamiento consistente por firma, bono y fecha;
- validación de rangos y valores de las variables clave.

---

### Generación de metadata de bonos

Se construye un archivo auxiliar `bonds_meta` que contiene información básica por bono:

- `issuer`
- `ticker`
- `bond_ric`
- (opcionalmente) `sector_source`

Este archivo permite asegurar que los merges posteriores no fallen en ausencia de metadata completa.

---

### Salidas

El bloque genera:

- `bonds_monthly_spreads.parquet` (dataset operativo)
- `bonds_meta.parquet` (metadata mínima de bonos)

Ambos archivos se guardan en disco para su uso en las etapas posteriores del pipeline.

---

### Alcance interpretativo

Este bloque no introduce transformaciones económicas adicionales sobre los spreads, sino que estandariza su representación.

En consecuencia, cualquier interpretación de los spreads en el análisis empírico refleja directamente la información contenida en el OAS observado, sin ajustes adicionales por liquidez, impuestos u otros factores.

In [ ]:
# ==========================================================
# 4.1 CONSTRUCCIÓN DE bonds_monthly_spreads (DESDE CLEAN)
# ==========================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ==========================================================
# 4.1.1 HELPERS
# ==========================================================

def standardize_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia nombres de columnas (espacios, formatos inconsistentes).
    """
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def to_eom(s: pd.Series) -> pd.Series:
    """
    Convierte fechas a fin de mes.
    """
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")


def read_any(path: Path) -> pd.DataFrame:
    """
    Lee archivo parquet o csv según extensión.
    """
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    elif path.suffix == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Formato no soportado: {path}")


def coalesce_first_nonnull(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    """
    Combina columnas candidatas tomando el primer valor no nulo.
    """
    valid = [c for c in candidates if c in df.columns]

    if not valid:
        return pd.Series(pd.NA, index=df.index, dtype="object")

    out = df[valid[0]].copy()

    for c in valid[1:]:
        out = out.fillna(df[c])

    return out


def ensure_expected_columns_from_clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Armoniza columnas y construye dataset operativo de spreads.
    """
    out = df.copy()

    # ------------------------------------------------------
    # identificadores
    # ------------------------------------------------------
    out["issuer"] = coalesce_first_nonnull(
        out, ["issuer", "Issuer", "Issuer_x", "Issuer_y"]
    ).astype("string").str.strip()

    out["ticker"] = coalesce_first_nonnull(
        out, ["ticker", "Ticker", "Ticker_x", "Ticker_y"]
    ).astype("string").str.strip()

    if "bond_ric" not in out.columns:
        if "Instrument" in out.columns:
            out["bond_ric"] = out["Instrument"]
        else:
            raise KeyError("No se encontró bond_ric ni Instrument.")

    if "date" not in out.columns:
        raise KeyError("No se encontró columna date.")

    # ------------------------------------------------------
    # spread
    # ------------------------------------------------------
    if "spread_bps" not in out.columns:
        if "oas" in out.columns:
            out["spread_bps"] = out["oas"]
        elif "oas_bps" in out.columns:
            out["spread_bps"] = out["oas_bps"]
        else:
            raise KeyError("No se encontró spread_bps ni oas/oas_bps.")

    # ------------------------------------------------------
    # variables opcionales
    # ------------------------------------------------------
    for col in ["ytm", "residual_maturity_years", "amount_issued"]:
        if col not in out.columns:
            out[col] = np.nan

    # ------------------------------------------------------
    # tipos
    # ------------------------------------------------------
    out["bond_ric"] = out["bond_ric"].astype("string").str.strip()
    out["date"] = to_eom(out["date"])

    for c in ["spread_bps", "ytm", "residual_maturity_years", "amount_issued"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # limpiar strings inválidos
    for c in ["issuer", "ticker", "bond_ric"]:
        out[c] = out[c].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    # ------------------------------------------------------
    # selección final
    # ------------------------------------------------------
    out = (
        out[
            [
                "issuer",
                "ticker",
                "bond_ric",
                "date",
                "spread_bps",
                "ytm",
                "residual_maturity_years",
                "amount_issued",
            ]
        ]
        .dropna(subset=["issuer", "bond_ric", "date", "spread_bps"])
        .drop_duplicates(subset=["bond_ric", "date"])
        .sort_values(["issuer", "bond_ric", "date"])
        .reset_index(drop=True)
    )

    return out


# ==========================================================
# 4.1.2 LECTURA DESDE CLEAN
# ==========================================================

CLEAN_BONDS_PARQ = CLEAN / "bonds_monthly_spreads.parquet"
CLEAN_BONDS_CSV = CLEAN_BONDS_PARQ.with_suffix(".csv")

OUT_BONDS_PARQ = INTERIM / "bonds_monthly_spreads.parquet"
OUT_BONDS_CSV = OUT_BONDS_PARQ.with_suffix(".csv")

if CLEAN_BONDS_PARQ.exists():
    source_used = CLEAN_BONDS_PARQ
elif CLEAN_BONDS_CSV.exists():
    source_used = CLEAN_BONDS_CSV
else:
    raise FileNotFoundError("No se encontró bonds_monthly_spreads en CLEAN.")

raw_clean = read_any(source_used)
raw_clean = standardize_cols(raw_clean)

print(f"Fuente utilizada: {source_used}")
print("Columnas originales:", raw_clean.columns.tolist())


# ==========================================================
# 4.1.3 CONSTRUCCIÓN DATASET OPERATIVO
# ==========================================================

bonds_monthly_spreads = ensure_expected_columns_from_clean(raw_clean)


# ==========================================================
# 4.1.4 EXPORT
# ==========================================================

bonds_monthly_spreads.to_parquet(OUT_BONDS_PARQ, index=False)
bonds_monthly_spreads.to_csv(OUT_BONDS_CSV, index=False)

print("Archivo guardado en INTERIM:")
print(OUT_BONDS_PARQ)


# ==========================================================
# 4.1.5 QA
# ==========================================================

print("Filas:", len(bonds_monthly_spreads))
print("Bond RIC únicos:", bonds_monthly_spreads["bond_ric"].nunique())
print("Issuer únicos:", bonds_monthly_spreads["issuer"].nunique())
print("Rango fechas:",
      bonds_monthly_spreads["date"].min(),
      "->",
      bonds_monthly_spreads["date"].max())

## 5) Construcción de spreads a nivel firma–mes

### Objetivo

Este bloque transforma la información de spreads de crédito desde nivel bono–mes a nivel firma–mes, generando la variable dependiente principal utilizada en las estimaciones econométricas.

Dado que múltiples bonos pueden coexistir para una misma firma en un período determinado, es necesario consolidar esta información en una única observación por firma y mes.

---

### Enfoque general

El procedimiento parte del dataset `bonds_monthly_spreads` previamente construido y realiza:

- limpieza adicional de identificadores y tipos de datos;
- verificación de integridad de las variables clave;
- agregación de variables a nivel `issuer-date`.

El resultado es un panel firma–mes con métricas representativas del costo de financiamiento corporativo en el mercado de bonos.

---

### Estrategia de agregación

Para cada firma $i$ en el período $t$, se construyen las siguientes variables:

#### Spread promedio

$$
spread_{i,t} = \frac{\sum_{j \in i} w_{j,t} \cdot spread_{j,t}}{\sum_{j \in i} w_{j,t}}
$$

donde:

- $j$ indexa bonos de la firma $i$,
- $w_{j,t}$ corresponde al monto emitido (`amount_issued`) cuando está disponible.

En ausencia de información de pesos, se utiliza un promedio simple:

$$
spread_{i,t} = \frac{1}{N_{i,t}} \sum_{j \in i} spread_{j,t}
$$

---

#### Yield to maturity promedio

Se construye de forma análoga, utilizando ponderación por monto cuando es posible.

---

#### Madurez residual promedio

Se calcula como promedio (ponderado o simple) de la madurez residual de los bonos activos de la firma.

---

#### Número de bonos

Se incluye como:

$$
n\_bonds_{i,t} = |\{j \in i\}|
$$

lo que permite capturar la intensidad de observaciones disponibles por firma.

---

### Limpieza y validación

Previo a la agregación, el dataset es sometido a:

- eliminación de observaciones con identificadores faltantes;
- eliminación de duplicados a nivel `bond_ric-date`;
- estandarización de tipos de datos;
- inspección de valores faltantes en variables clave.

Posteriormente, se valida que:

- no existan duplicados a nivel `issuer-date`;
- el rango temporal sea consistente;
- las distribuciones de las variables agregadas sean razonables.

---

### Salida

El bloque genera el dataset:

- `bonds_monthly_spreads_firmlevel.parquet`

que constituye la base del bloque de spreads dentro del panel final.

---

### Alcance interpretativo

La agregación implica una pérdida de información a nivel instrumento individual, pero permite construir una medida sintética del costo de financiamiento de cada firma en cada período.

En consecuencia, los resultados empíricos deben interpretarse como relaciones a nivel firma–mes, y no a nivel bono individual.

In [ ]:
# ==========================================================
# 4.2 CONSTRUCCIÓN DE SPREADS A NIVEL FIRMA–MES
# ==========================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ==========================================================
# 4.2.1 HELPERS
# ==========================================================

def load_any_existing(candidates: list[Path]) -> tuple[pd.DataFrame, Path]:
    """
    Carga el primer archivo existente entre múltiples candidatos.
    """
    for p in candidates:
        if p.exists():
            if p.suffix == ".parquet":
                return pd.read_parquet(p), p
            elif p.suffix == ".csv":
                return pd.read_csv(p), p

    raise FileNotFoundError(
        "No se encontró ninguno de los archivos candidatos:\n" +
        "\n".join(str(p) for p in candidates)
    )


def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Convierte columnas a formato numérico de forma robusta.
    """
    out = df.copy()

    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out


# ==========================================================
# 4.2.2 CARGA DE SPREADS
# ==========================================================

BONDS_SPREADS_CANDIDATES = [
    INTERIM / "bonds_monthly_spreads.parquet",
    CLEAN   / "bonds_monthly_spreads.parquet",
    RAW     / "bonds_monthly_spreads.parquet",
    INTERIM / "bonds_monthly_spreads.csv",
    CLEAN   / "bonds_monthly_spreads.csv",
    RAW     / "bonds_monthly_spreads.csv",
]

bonds, bonds_source_path = load_any_existing(BONDS_SPREADS_CANDIDATES)
bonds = bonds.copy()

print(f"\nLeyendo bonds desde: {bonds_source_path}")


# ==========================================================
# 4.2.3 LIMPIEZA BÁSICA
# ==========================================================

bonds["date"] = pd.to_datetime(bonds["date"], errors="coerce")
bonds["issuer"] = bonds["issuer"].astype("string").str.strip()
bonds["bond_ric"] = bonds["bond_ric"].astype("string").str.strip()

if "ticker" in bonds.columns:
    bonds["ticker"] = bonds["ticker"].astype("string").str.strip()

bonds = coerce_numeric(
    bonds,
    ["spread_bps", "ytm", "residual_maturity_years", "amount_issued"]
)


# ==========================================================
# 4.2.4 QA PRELIMINAR
# ==========================================================

print("\n=== QA previo ===")
print("Columnas:", bonds.columns.tolist())

if "residual_maturity_years" in bonds.columns:
    print("\nResumen residual_maturity_years:")
    print(bonds["residual_maturity_years"].describe())

if "ytm" in bonds.columns:
    print("\nResumen ytm:")
    print(bonds["ytm"].describe())


# ==========================================================
# 4.2.5 LIMPIEZA FINAL
# ==========================================================

bonds = (
    bonds
    .dropna(subset=["issuer", "bond_ric", "date", "spread_bps"])
    .drop_duplicates(subset=["bond_ric", "date"])
    .sort_values(["issuer", "bond_ric", "date"])
    .copy()
)


# ==========================================================
# 4.2.6 DETECCIÓN DE PESOS
# ==========================================================

amount_candidates = [
    c for c in bonds.columns
    if "amount" in c.lower() and ("issued" in c.lower() or "issue" in c.lower())
]

if not amount_candidates:
    amount_candidates = [c for c in bonds.columns if "amount" in c.lower()]

amount_col = amount_candidates[0] if amount_candidates else None

print("\namount_col detectada:", amount_col)


# ==========================================================
# 4.2.7 FUNCIÓN DE AGREGACIÓN
# ==========================================================

def agg_func(df: pd.DataFrame) -> pd.Series:

    # pesos
    if amount_col and amount_col in df.columns:
        w = pd.to_numeric(df[amount_col], errors="coerce")
    else:
        w = pd.Series(np.nan, index=df.index)

    out = {}

    # spread
    if w.notna().sum() > 0 and w.sum() > 0:
        out["spread_mean_bps"] = np.average(df["spread_bps"], weights=w)
    else:
        out["spread_mean_bps"] = df["spread_bps"].mean()

    # ytm
    if "ytm" in df.columns:
        y = pd.to_numeric(df["ytm"], errors="coerce")

        if w.notna().sum() > 0 and w.sum() > 0 and y.notna().sum() > 0:
            mask = y.notna() & w.notna()
            out["ytm_mean"] = np.average(y[mask], weights=w[mask]) if mask.sum() > 0 else y.mean()
        else:
            out["ytm_mean"] = y.mean()
    else:
        out["ytm_mean"] = np.nan

    # maturidad
    if "residual_maturity_years" in df.columns:
        m = pd.to_numeric(df["residual_maturity_years"], errors="coerce")

        if w.notna().sum() > 0 and w.sum() > 0 and m.notna().sum() > 0:
            mask = m.notna() & w.notna()
            out["residual_maturity_mean"] = np.average(m[mask], weights=w[mask]) if mask.sum() > 0 else m.mean()
        else:
            out["residual_maturity_mean"] = m.mean()
    else:
        out["residual_maturity_mean"] = np.nan

    # cantidad de bonos
    out["n_bonds"] = df["bond_ric"].nunique()

    return pd.Series(out)


# ==========================================================
# 4.2.8 AGREGACIÓN FIRMA–MES
# ==========================================================

bonds_monthly_spreads_firm = (
    bonds
    .groupby(["issuer", "date"], as_index=False)
    .apply(agg_func)
    .reset_index(drop=True)
    .sort_values(["issuer", "date"])
)


# ==========================================================
# 4.2.9 QA FINAL
# ==========================================================

print("\n=== QA firm-level ===")
print("Shape:", bonds_monthly_spreads_firm.shape)
print("Duplicados issuer-date:", bonds_monthly_spreads_firm.duplicated(["issuer", "date"]).sum())


# ==========================================================
# 4.2.10 EXPORT
# ==========================================================

OUT_FIRM_PARQ = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"
OUT_FIRM_CSV  = INTERIM / "bonds_monthly_spreads_firmlevel.csv"

bonds_monthly_spreads_firm.to_parquet(OUT_FIRM_PARQ, index=False)
bonds_monthly_spreads_firm.to_csv(OUT_FIRM_CSV, index=False)

print("\nFirm-level spreads guardado en:")
print(OUT_FIRM_PARQ)

### 5.1) QA spreads

In [ ]:
# =====================================================
# QA básica de spreads (sanity check) – MISMA CELDA QUE TENÍAS
# =====================================================
df = bonds.copy()

print("\n--- Estadísticas generales ---")
print(df["spread_bps"].describe())

print("\n--- NAs en spread_bps ---")
print(df["spread_bps"].isna().mean())

print("\n--- Conteo de bonos únicos ---")
print("n bond_ric:", df["bond_ric"].nunique())

print("\n--- Conteo de emisores únicos ---")
print("n issuer:", df["issuer"].nunique())

print("\n--- Rango temporal ---")
print(df["date"].min(), "→", df["date"].max())

print("\n--- Checks por issuer-date (pre-aggregate) ---")
tmp = df.groupby(["issuer", "date"]).size()
print("issuer-date grupos:", tmp.shape[0])
print("max filas por issuer-date:", tmp.max())


--- Estadísticas generales ---
count      327629.0
mean     143.300153
std      111.914424
min        -20800.1
25%            94.7
50%           138.3
75%           182.8
max          6427.7
Name: spread_bps, dtype: Float64

--- NAs en spread_bps ---
0.0

--- Conteo de bonos únicos ---
n bond_ric: 5805

--- Conteo de emisores únicos ---
n issuer: 271

--- Rango temporal ---
2015-01-31 00:00:00 → 2025-12-31 00:00:00

--- Checks por issuer-date (pre-aggregate) ---
issuer-date grupos: 28701
max filas por issuer-date: 724


## 6) Construcción del mapping `issuer → ticker → sector`

### Objetivo

Este bloque construye una tabla de correspondencia entre emisores (`issuer`), tickers y sector económico, con el fin de facilitar merges posteriores a nivel firma y estandarizar identificadores entre bloques de datos heterogéneos.

El output final es un mapping conservador, deduplicado por emisor, que resume para cada `issuer` un ticker representativo y, cuando está disponible, un sector asociado.

---

### Fuentes utilizadas

La construcción del mapping combina dos fuentes principales:

1. **Base de bonos filtrada**  
   Se utiliza para recuperar la relación entre `issuer` y `ticker`.

2. **Base de market power mensual**  
   Se utiliza para recuperar la relación entre `ticker` y `sector` (o, en su defecto, `gics_sector_name`).

De este modo, el procedimiento implementa una estrategia en dos pasos:

$$
issuer \rightarrow ticker \rightarrow sector
$$

---

### Metodología

#### 1. Estandarización de identificadores

Se normalizan los nombres de columnas y se limpian strings para eliminar:

- espacios sobrantes;
- valores vacíos;
- representaciones textuales de faltantes;
- y sufijos propios de identificadores tipo RIC.

En particular, se construye una variable `ticker_base` a partir del ticker original, eliminando sufijos de mercado y aplicando normalizaciones simples. Esto permite aumentar la compatibilidad entre fuentes que no usan exactamente la misma convención de ticker.

#### 2. Construcción del bloque `issuer → ticker`

A partir de la base de bonos, se retienen observaciones con `issuer` y `ticker_base` válidos. Luego se evalúa:

- cuántos tickers aparecen por emisor;
- cuántos emisores aparecen por ticker;
- y el grado de ambigüedad de la correspondencia.

#### 3. Construcción del bloque `ticker → sector`

A partir de la base de market power, se recupera el sector económico por ticker. Se prioriza la columna `sector`, y si no existe, se utiliza `gics_sector_name`.

Posteriormente se construye una versión deduplicada a nivel `ticker_base`.

#### 4. Merge y construcción final

El mapping final se construye mediante un merge de tipo many-to-one:

$$
issuer\_ticker \;\; \xrightarrow[\text{merge on } ticker\_base]{} \;\; issuer\_ticker\_sector
$$

Finalmente, se deduplica a nivel `issuer`, conservando una única asignación representativa por emisor.

---

### Salidas

El bloque genera los archivos:

- `data/clean/issuer_ticker_sector.parquet`
- `data/clean/issuer_ticker_sector.csv`

---

### Validaciones incluidas

Se realizan los siguientes controles:

- existencia de las fuentes requeridas;
- presencia de columnas mínimas (`issuer`, `ticker`, `sector`);
- número de tickers por emisor;
- número de emisores por ticker;
- número de sectores por ticker;
- porcentaje de faltantes en `ticker` y `sector`;
- deduplicación final por `issuer`.

---

### Alcance interpretativo

El mapping construido en este bloque es una tabla operativa de correspondencia y no una clasificación económica exhaustiva. En particular:

- la asignación de ticker por emisor es conservadora y toma una única correspondencia representativa;
- la asignación sectorial depende de la compatibilidad entre fuentes;
- y la variable `sector` debe interpretarse como una etiqueta útil para merges y controles, más que como una reconstrucción completa de la estructura corporativa del emisor.

In [ ]:
# ==========================================================
# 6 MAPPING issuer -> ticker -> sector
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ==========================================================
# 6.1 HELPERS
# ==========================================================

def _normalize_colname(c: str) -> str:
    """
    Normaliza nombres de columnas.
    """
    return str(c).strip().lower().replace(" ", "_")


def _clean_str(s: pd.Series) -> pd.Series:
    """
    Limpia strings y homogeneiza faltantes.
    """
    s = s.astype("string").str.strip()
    return s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})


def _base_ticker(s: pd.Series) -> pd.Series:
    """
    Construye una versión base del ticker, eliminando sufijos
    de mercado y normalizando separadores.
    """
    s = _clean_str(s).str.upper()

    # quitar sufijos tipo RIC equity
    s = s.str.replace(r"\..*$", "", regex=True)

    # normalizaciones útiles
    s = s.str.replace("/", ".", regex=False)
    s = s.str.replace("-", ".", regex=False)

    return s


# ==========================================================
# 6.2 FUENTE 1: issuer -> ticker DESDE BONOS
# ==========================================================

BONDS_FILE = PROJECT_ROOT / "data" / "inputs" / "new_bonds" / "bonds_sp500_filtered.parquet"

if not BONDS_FILE.exists():
    raise FileNotFoundError(f"No encontré {BONDS_FILE}")

print(f"Fuente bonds: {BONDS_FILE.name}")

bonds_raw = pd.read_parquet(BONDS_FILE).copy()
bonds_raw.columns = [_normalize_colname(c) for c in bonds_raw.columns]

required_bonds = ["issuer", "ticker"]
missing_bonds = [c for c in required_bonds if c not in bonds_raw.columns]

if missing_bonds:
    raise KeyError(f"Faltan columnas en bonds_sp500_filtered: {missing_bonds}")

issuer_map_raw = bonds_raw[["issuer", "ticker"]].copy()
issuer_map_raw["issuer"] = _clean_str(issuer_map_raw["issuer"])
issuer_map_raw["ticker"] = _clean_str(issuer_map_raw["ticker"])
issuer_map_raw["ticker_base"] = _base_ticker(issuer_map_raw["ticker"])

issuer_map_raw = issuer_map_raw.dropna(subset=["issuer", "ticker_base"]).copy()

print("\n=== issuer_map_raw (preview) ===")
print(issuer_map_raw.head(10))

print("\n=== issuer_map_raw (QA) ===")
print("Filas:", len(issuer_map_raw))
print("Issuer únicos:", issuer_map_raw["issuer"].nunique())
print("Ticker únicos:", issuer_map_raw["ticker"].nunique())
print("Ticker_base únicos:", issuer_map_raw["ticker_base"].nunique())

issuer_ticker_counts = (
    issuer_map_raw[["issuer", "ticker_base"]]
    .dropna()
    .drop_duplicates()
    .groupby("issuer")["ticker_base"]
    .nunique()
)

print("\nTicker por issuer:")
print(issuer_ticker_counts.describe())
print("Issuers con más de 1 ticker:", (issuer_ticker_counts > 1).sum())

ticker_issuer_counts = (
    issuer_map_raw[["issuer", "ticker_base"]]
    .dropna()
    .drop_duplicates()
    .groupby("ticker_base")["issuer"]
    .nunique()
)

print("\nIssuer por ticker:")
print(ticker_issuer_counts.describe())
print("Tickers con más de 1 issuer:", (ticker_issuer_counts > 1).sum())


# ==========================================================
# 6.3 FUENTE 2: ticker -> sector DESDE MARKET POWER
# ==========================================================

MP_PARQ = INTERIM / "market_power_monthly.parquet"
MP_CSV = INTERIM / "market_power_monthly.csv"

if MP_PARQ.exists():
    mp_raw = pd.read_parquet(MP_PARQ).copy()
    mp_path = MP_PARQ
elif MP_CSV.exists():
    mp_raw = pd.read_csv(MP_CSV).copy()
    mp_path = MP_CSV
else:
    raise FileNotFoundError("No encontré market_power_monthly ni en parquet ni csv")

print(f"\nFuente market power: {mp_path.name}")

mp_raw.columns = [str(c).strip() for c in mp_raw.columns]

if "ticker" not in mp_raw.columns:
    raise KeyError("market_power_monthly debe tener columna 'ticker'")

if "sector" in mp_raw.columns:
    sector_col = "sector"
elif "gics_sector_name" in mp_raw.columns:
    sector_col = "gics_sector_name"
else:
    raise KeyError("No encontré ni 'sector' ni 'gics_sector_name' en market_power_monthly")

ticker_sector_raw = mp_raw[["ticker", sector_col]].copy()
ticker_sector_raw = ticker_sector_raw.rename(columns={sector_col: "sector"})

ticker_sector_raw["ticker"] = _clean_str(ticker_sector_raw["ticker"])
ticker_sector_raw["ticker_base"] = _base_ticker(ticker_sector_raw["ticker"])
ticker_sector_raw["sector"] = _clean_str(ticker_sector_raw["sector"])

ticker_sector_raw = ticker_sector_raw.dropna(subset=["ticker_base", "sector"]).copy()

print("\n=== ticker_sector_raw (preview) ===")
print(ticker_sector_raw.head(10))

print("\n=== ticker_sector_raw (QA) ===")
print("Filas:", len(ticker_sector_raw))
print("Ticker únicos:", ticker_sector_raw["ticker"].nunique())
print("Ticker_base únicos:", ticker_sector_raw["ticker_base"].nunique())
print("Sector únicos:", ticker_sector_raw["sector"].nunique())

ticker_sector_counts = (
    ticker_sector_raw[["ticker_base", "sector"]]
    .dropna()
    .drop_duplicates()
    .groupby("ticker_base")["sector"]
    .nunique()
)

print("\nSector por ticker:")
print(ticker_sector_counts.describe())
print("Tickers con más de 1 sector:", (ticker_sector_counts > 1).sum())


# ==========================================================
# 6.4 INTERSECCIÓN ENTRE FUENTES
# ==========================================================

issuer_bases = set(issuer_map_raw["ticker_base"].dropna().unique())
sector_bases = set(ticker_sector_raw["ticker_base"].dropna().unique())
common_bases = issuer_bases.intersection(sector_bases)

print("\n=== INTERSECCIÓN issuer_map_raw vs ticker_sector_raw ===")
print("Ticker_base en bonds:", len(issuer_bases))
print("Ticker_base en market_power:", len(sector_bases))
print("Comunes:", len(common_bases))


# ==========================================================
# 6.5 CONSTRUCCIÓN CONSERVADORA DEL MAPPING FINAL
# ==========================================================

issuer_ticker = (
    issuer_map_raw
    .sort_values(["issuer", "ticker_base", "ticker"])
    .groupby("issuer", as_index=False)
    .agg({
        "ticker": lambda s: s.dropna().iloc[0] if s.dropna().shape[0] > 0 else pd.NA,
        "ticker_base": lambda s: s.dropna().iloc[0] if s.dropna().shape[0] > 0 else pd.NA,
    })
)

ticker_sector = (
    ticker_sector_raw
    .sort_values(["ticker_base", "ticker", "sector"])
    .groupby("ticker_base", as_index=False)
    .agg({
        "sector": lambda s: s.dropna().iloc[0] if s.dropna().shape[0] > 0 else pd.NA,
    })
)

issuer_map_final = issuer_ticker.merge(
    ticker_sector,
    on="ticker_base",
    how="left",
    validate="m:1",
)

print("\n=== issuer_map_final (preview) ===")
print(issuer_map_final.head(10))

print("\n=== issuer_map_final (QA) ===")
print("Filas:", len(issuer_map_final))
print("Issuer únicos:", issuer_map_final["issuer"].nunique())
print("Ticker únicos:", issuer_map_final["ticker"].nunique())
print("Ticker_base únicos:", issuer_map_final["ticker_base"].nunique())
print("ticker NA%:", issuer_map_final["ticker"].isna().mean())
print("sector NA%:", issuer_map_final["sector"].isna().mean())
print("Duplicados issuer:", issuer_map_final.duplicated(["issuer"]).sum())


# ==========================================================
# 6.6 EXPORT
# ==========================================================

OUT_MAP_PARQ = CLEAN / "issuer_ticker_sector.parquet"
OUT_MAP_CSV = CLEAN / "issuer_ticker_sector.csv"

issuer_map_final.to_parquet(OUT_MAP_PARQ, index=False)
issuer_map_final.to_csv(OUT_MAP_CSV, index=False)

print("\nMapping guardado en:")
print(f"- {OUT_MAP_PARQ}")
print(f"- {OUT_MAP_CSV}")

## 7) Enriquecimiento del panel de spreads firma–mes con ticker y sector

### Objetivo

Este bloque incorpora información adicional de identificación y clasificación sectorial al panel de spreads ya agregado a nivel firma–mes.

En particular, se agrega para cada emisor:

- un ticker representativo;
- una etiqueta sectorial operativa.

El objetivo es enriquecer el panel para facilitar merges posteriores con otros bloques de datos y permitir controles descriptivos o sectoriales, sin alterar las definiciones originales de spreads.

---

### Insumos

Se utilizan dos datasets previamente construidos:

1. `bonds_monthly_spreads_firmlevel.parquet`  
   Panel de spreads agregado a nivel `issuer-date`.

2. `issuer_ticker_sector.parquet`  
   Mapping operativo entre emisor, ticker y sector.

---

### Metodología

El procedimiento realiza un merge de tipo **left join** sobre la clave `issuer`, preservando como base el panel de spreads firma–mes.

Formalmente:

$$
\text{spreads\_firm\_enriched} = \text{spreads\_firm} \;\; \text{LEFT JOIN} \;\; \text{issuer\_map}
$$

donde el join se realiza únicamente sobre el identificador `issuer`.

Esta estrategia implica que:

- todas las observaciones del panel de spreads deben preservarse;
- el mapping se incorpora solo cuando existe correspondencia válida;
- y las variables `ticker` y `sector` pueden quedar faltantes en los casos sin match.

---

### Restricciones de diseño

Este bloque se construye bajo tres principios explícitos:

1. **no modificar la definición de spreads**;  
2. **no alterar el número de observaciones del panel firma–mes**;  
3. **mantener unicidad de la clave `issuer-date`**.

---

### Validaciones incluidas

Se implementan los siguientes controles:

- comparación del número de filas antes y después del merge;
- porcentaje de valores faltantes en `ticker` y `sector`;
- chequeo de duplicados en la combinación `issuer-date`;
- conteo de emisores sin match en el mapping.

Estos controles permiten verificar que el enriquecimiento del panel no introduzca multiplicación de filas ni inconsistencias de granularidad.

---

### Salida

El bloque genera:

- `data/intermediate/spreads_firm_enriched.parquet`
- `data/intermediate/spreads_firm_enriched.csv`

---

### Alcance interpretativo

Las variables `ticker` y `sector` añadidas en este paso cumplen una función auxiliar de identificación y clasificación. No forman parte de la definición económica del spread ni modifican la construcción de la variable dependiente.

En consecuencia, cualquier cambio en cobertura de `ticker` o `sector` afecta la riqueza informativa del panel, pero no la medición del spread firma–mes en sí misma.

In [ ]:
# ==========================================================
# 7 MERGE spreads firm-level + mapping issuer -> ticker/sector
# ==========================================================

import pandas as pd


# ==========================================================
# 7.1 CARGA DEL PANEL DE SPREADS FIRMA–MES
# ==========================================================

SPREADS_FIRM_PARQ = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"
SPREADS_FIRM_CSV = INTERIM / "bonds_monthly_spreads_firmlevel.csv"

if SPREADS_FIRM_PARQ.exists():
    spreads_firm = pd.read_parquet(SPREADS_FIRM_PARQ)
elif SPREADS_FIRM_CSV.exists():
    spreads_firm = pd.read_csv(SPREADS_FIRM_CSV)
else:
    raise FileNotFoundError("No encontré bonds_monthly_spreads_firmlevel en INTERIM.")

spreads_firm["date"] = pd.to_datetime(spreads_firm["date"], errors="coerce")
spreads_firm["issuer"] = spreads_firm["issuer"].astype("string").str.strip()

print("Spreads firm-level shape:", spreads_firm.shape)


# ==========================================================
# 7.2 CARGA DEL MAPPING issuer -> ticker -> sector
# ==========================================================

MAP_PARQ = CLEAN / "issuer_ticker_sector.parquet"
MAP_CSV = CLEAN / "issuer_ticker_sector.csv"

if MAP_PARQ.exists():
    issuer_map = pd.read_parquet(MAP_PARQ)
elif MAP_CSV.exists():
    issuer_map = pd.read_csv(MAP_CSV)
else:
    raise FileNotFoundError("No encontré issuer_ticker_sector en CLEAN.")

issuer_map["issuer"] = issuer_map["issuer"].astype("string").str.strip()

print("Mapping shape:", issuer_map.shape)


# ==========================================================
# 7.3 MERGE LEFT
# ==========================================================

n_before = spreads_firm.shape[0]

spreads_firm_enriched = spreads_firm.merge(
    issuer_map,
    on="issuer",
    how="left",
    validate="m:1",
)

n_after = spreads_firm_enriched.shape[0]


# ==========================================================
# 7.4 CHEQUEOS DE INTEGRIDAD
# ==========================================================

print("\n=== QA Merge ===")
print("Filas antes:", n_before)
print("Filas después:", n_after)
print("Δ filas:", n_after - n_before)

assert n_before == n_after, "El merge cambió el número de filas. Revisar claves."

if "ticker" in spreads_firm_enriched.columns:
    print("Ticker NA%:", spreads_firm_enriched["ticker"].isna().mean())

if "sector" in spreads_firm_enriched.columns:
    print("Sector NA%:", spreads_firm_enriched["sector"].isna().mean())

dup = spreads_firm_enriched.duplicated(subset=["issuer", "date"]).sum()
print("Duplicados issuer-date:", dup)

assert dup == 0, "Hay duplicados issuer-date tras el merge."

print(
    "\nIssuers sin match en mapping:",
    spreads_firm_enriched["ticker"].isna().sum()
    if "ticker" in spreads_firm_enriched.columns else
    "ticker no existe"
)

print("\nPreview:")
print(spreads_firm_enriched.head())


# ==========================================================
# 7.5 EXPORT
# ==========================================================

OUT_ENRICHED_PARQ = INTERIM / "spreads_firm_enriched.parquet"
OUT_ENRICHED_CSV = INTERIM / "spreads_firm_enriched.csv"

spreads_firm_enriched.to_parquet(OUT_ENRICHED_PARQ, index=False)
spreads_firm_enriched.to_csv(OUT_ENRICHED_CSV, index=False)

print("\nspreads_firm_enriched guardado en:")
print("-", OUT_ENRICHED_PARQ)
print("-", OUT_ENRICHED_CSV)

## 8) Estimación rolling del CAPM: beta e idiosyncratic volatility

### Objetivo

El presente bloque tiene como objetivo construir medidas de riesgo accionario a nivel firma–mes, a partir de información diaria de retornos de acciones y del mercado. En particular, se estiman dos variables clave:

- $\beta_{i,t}^{(252)}$: sensibilidad del retorno de la firma $i$ al mercado, estimada en una ventana móvil de 252 días;
- $\sigma_{\varepsilon,i,t}^{(252)}$: volatilidad idiosincrática, definida como el desvío estándar de los residuos del modelo CAPM en dicha ventana.

Estas variables constituyen controles fundamentales en la literatura de spreads crediticios, al capturar tanto el riesgo sistemático como el componente no diversificable específico de cada firma.

---

### Metodología de estimación

Para cada firma $i$, se estima de forma rolling el siguiente modelo CAPM utilizando retornos diarios:

$$
r_{i,t} = \alpha_i + \beta_i r_{m,t} + \varepsilon_{i,t}
$$

donde:

- $r_{i,t}$ es el retorno diario de la acción;
- $r_{m,t}$ es el retorno diario del mercado (S&P 500);
- $\varepsilon_{i,t}$ representa el componente idiosincrático.

La estimación se realiza sobre ventanas móviles de 252 observaciones (aproximadamente un año bursátil), imponiendo un mínimo de observaciones válidas equivalente al 50% de la ventana.

A partir de cada ventana se obtienen:

- $\hat{\beta}_{i,t}$: coeficiente estimado del factor de mercado;
- $\hat{\sigma}_{\varepsilon,i,t}$: volatilidad idiosincrática, calculada como:

$$
\hat{\sigma}_{\varepsilon,i,t} = \sqrt{252} \cdot \text{sd}(\hat{\varepsilon}_{i,t})
$$

lo cual permite anualizar la medida.

---

### Construcción de las variables finales

Las estimaciones se realizan inicialmente a frecuencia diaria. Posteriormente, se construyen las variables a nivel mensual mediante:

- agregación por firma–mes;
- selección de la **última observación disponible dentro de cada mes**.

Esto garantiza consistencia temporal con el panel principal de análisis (firma–mes), evitando interpolaciones o promedios que podrían introducir sesgos.

---

### Controles de calidad y validaciones

Durante el proceso se implementan múltiples validaciones:

- verificación de duplicados en la clave (`ticker`, `date`);
- control de cobertura de datos (retornos de firma y mercado);
- chequeo de valores faltantes y proporción de observaciones válidas;
- exclusión de ventanas con varianza nula del retorno de mercado;
- requerimiento mínimo de observaciones por ventana;
- validación de la unicidad de observaciones en el panel mensual final.

Estas verificaciones aseguran la robustez de las estimaciones y la consistencia del dataset resultante.

---

### Consideraciones de implementación

La estimación rolling del CAPM se implementa utilizando `numpy` (en particular, `np.linalg.lstsq`) en lugar de `statsmodels`. Esta decisión responde a criterios de:

- reproducibilidad del entorno;
- reducción de dependencias externas;
- mayor control sobre el flujo de cálculo.

---

### Alcance interpretativo

Las variables construidas capturan:

- $\beta_{i,t}$: exposición sistemática al riesgo de mercado;
- $\sigma_{\varepsilon,i,t}$: riesgo idiosincrático no explicado por el mercado.

Ambas medidas son relevantes para el análisis de spreads crediticios, ya que permiten distinguir entre componentes de riesgo sistemático y específico en la determinación del costo de financiamiento corporativo.

In [ ]:
# ==========================================================
# 8. ROLLING CAPM -> ESTIMACIÓN DE BETA E IVOL (SIN STATSMODELS)
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ==========================================================
# 8.1 HELPERS DE INPUT Y ESTANDARIZACIÓN
# ==========================================================

def read_dataset(filename_candidates, folders):
    """
    Busca y lee un dataset dentro de múltiples carpetas y posibles nombres.
    Prioriza formato parquet; fallback a csv.
    """
    for folder in folders:
        for fname in filename_candidates:
            path = folder / fname
            if path.exists():
                print(f"📂 Leyendo: {path}")
                if path.suffix == ".parquet":
                    return pd.read_parquet(path)
                elif path.suffix == ".csv":
                    return pd.read_csv(path)

    tried = [str(folder / fname) for folder in folders for fname in filename_candidates]
    raise FileNotFoundError("No se encontró ninguno de estos archivos:\n" + "\n".join(tried))


def standardize_columns(df):
    """
    Limpia nombres de columnas (strip + string).
    """
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


# ==========================================================
# 8.2 FUNCIÓN DE ESTIMACIÓN ROLLING CAPM (NUMPY)
# ==========================================================

def rolling_capm_numpy(sub, ret_col="ret", mkt_col="mkt_ret", window=252, min_obs=None):
    """
    Estima beta e idiosyncratic volatility usando CAPM rolling.
    Implementación basada en numpy (lstsq).
    """

    if min_obs is None:
        min_obs = int(window * 0.5)

    sub = sub.sort_values("date").copy()

    # Conversión a numérico
    sub[ret_col] = pd.to_numeric(sub[ret_col], errors="coerce")
    sub[mkt_col] = pd.to_numeric(sub[mkt_col], errors="coerce")
    sub = sub.replace([np.inf, -np.inf], np.nan)

    # Si no alcanza la ventana mínima → no se estima
    if len(sub) < window:
        return pd.DataFrame(columns=["date", "beta_252", "ivol_252"])

    results = []

    y_all = sub[ret_col].astype(float).to_numpy()
    x_all = sub[mkt_col].astype(float).to_numpy()
    d_all = pd.to_datetime(sub["date"]).to_numpy()

    # Loop rolling
    for i in range(window, len(sub) + 1):

        y = y_all[i - window:i]
        x = x_all[i - window:i]
        d = d_all[i - 1]

        # Filtrado de observaciones válidas
        valid = np.isfinite(y) & np.isfinite(x)

        if valid.sum() < min_obs:
            continue

        yv = y[valid]
        xv = x[valid]

        # Evitar problemas de identificación (varianza ~ 0)
        if np.var(xv) < 1e-10:
            continue

        X = np.column_stack([np.ones(len(xv)), xv])

        try:
            coef, _, _, _ = np.linalg.lstsq(X, yv, rcond=None)
            alpha, beta = coef

            resid = yv - (alpha + beta * xv)

            # Volatilidad idiosincrática anualizada
            if len(resid) > 1:
                ivol = np.std(resid, ddof=1) * np.sqrt(252)
            else:
                ivol = np.nan

        except Exception:
            continue

        results.append({
            "date": d,
            "beta_252": float(beta),
            "ivol_252": float(ivol) if np.isfinite(ivol) else np.nan
        })

    return pd.DataFrame(results)


# ==========================================================
# 8.3 CARGA DE RETORNOS ACCIONARIOS (EQUITY)
# ==========================================================

search_order = [CLEAN, INTERIM, RAW]

equity = read_dataset(
    [
        "equity_returns_daily.parquet",
        "equity_returns_daily.csv",
        "equity_market_returns.parquet",
        "equity_market_returns.csv",
    ],
    search_order
)

equity = standardize_columns(equity)

print("\nColumnas equity (raw):", equity.columns.tolist())


# Normalización de nombres de columnas
rename_eq = {}

for c in equity.columns:
    cl = c.lower().strip()

    if cl == "date":
        rename_eq[c] = "date"

    elif cl in {"ticker", "instrument", "ric", "source_ric"}:
        rename_eq[c] = "ticker"

    elif cl in {"ret", "return", "daily_return", "equity_ret",
                "ret_log", "log_return", "log_ret"}:
        rename_eq[c] = "ret"

    elif cl == "issuer":
        rename_eq[c] = "issuer"

equity = equity.rename(columns=rename_eq).copy()


# Validación de columnas mínimas
required_eq = ["date", "ticker", "ret"]
missing_eq = [c for c in required_eq if c not in equity.columns]

if missing_eq:
    raise KeyError(f"Faltan columnas en equity: {missing_eq}. Disponibles: {equity.columns.tolist()}")


# Limpieza de tipos
equity["date"] = pd.to_datetime(equity["date"], errors="coerce")
equity["ticker"] = equity["ticker"].astype("string").str.strip()
equity["ret"] = pd.to_numeric(equity["ret"], errors="coerce")

if "issuer" in equity.columns:
    equity["issuer"] = equity["issuer"].astype("string").str.strip()

equity = equity.dropna(subset=["date", "ticker"]).sort_values(["ticker", "date"]).copy()


# QA input equity
print("\n=== QA equity input ===")
print("Shape:", equity.shape)
print("Tickers:", equity["ticker"].nunique())
print("Rango fechas:", equity["date"].min(), "->", equity["date"].max())
print("Duplicados ticker-date:", equity.duplicated(["ticker", "date"]).sum())
print("Ret NA%:", round(equity["ret"].isna().mean() * 100, 2))


# ==========================================================
# 8.4 CARGA DE RETORNOS DE MERCADO
# ==========================================================

market = read_dataset(
    [
        "market_sp500.parquet",
        "market_sp500.csv",
        "market_returns_daily.parquet",
        "market_returns_daily.csv",
        "market_prices_daily.parquet",
        "market_prices_daily.csv",
    ],
    search_order
)

market = standardize_columns(market)

print("\nColumnas market (raw):", market.columns.tolist())


# Normalización columnas market
rename_mkt = {}

for c in market.columns:
    cl = c.lower().strip()

    if cl == "date":
        rename_mkt[c] = "date"

    elif cl in {"mkt_ret", "market_ret", "ret", "return",
                "mkt_ret_log", "market_ret_log", "ret_log", "log_return", "log_ret"}:
        rename_mkt[c] = "mkt_ret"

    elif cl in {"close", "price", "px_last", "market_price",
                "index_level", "mkt_close", "market_close", "close_index"}:
        rename_mkt[c] = "price"

market = market.rename(columns=rename_mkt).copy()


# Construcción de retorno de mercado si no existe
if "mkt_ret" not in market.columns:
    if "price" in market.columns:
        market["price"] = pd.to_numeric(market["price"], errors="coerce")
        market["mkt_ret"] = np.log(market["price"] / market["price"].shift(1))
    else:
        raise KeyError(
            f"No encontré ni 'mkt_ret' ni precios. Columnas: {market.columns.tolist()}"
        )

market["date"] = pd.to_datetime(market["date"], errors="coerce")
market["mkt_ret"] = pd.to_numeric(market["mkt_ret"], errors="coerce")
market = market.dropna(subset=["date"]).sort_values("date").copy()


# QA market
print("\n=== QA market input ===")
print("Shape:", market.shape)
print("Rango fechas:", market["date"].min(), "->", market["date"].max())
print("Duplicados date:", market.duplicated(["date"]).sum())
print("mkt_ret NA%:", round(market["mkt_ret"].isna().mean() * 100, 2))


# ==========================================================
# 8.5 MERGE EQUITY + MARKET
# ==========================================================

df = equity.merge(
    market[["date", "mkt_ret"]],
    on="date",
    how="left",
    validate="m:1"
)

df = df.sort_values(["ticker", "date"]).copy()


# QA post-merge
print("\n=== Diagnóstico post-merge equity + market ===")
print("Shape total:", df.shape)
print("Tickers:", df["ticker"].nunique())
print("Rango fechas:", df["date"].min(), "->", df["date"].max())

print("\nNulos en columnas clave:")
print(df[["date", "ticker", "ret", "mkt_ret"]].isna().sum())


# ==========================================================
# 8.6 ESTIMACIÓN ROLLING CAPM POR TICKER
# ==========================================================

ROLL_WINDOW = 252
results = []

for ticker, sub in df.groupby("ticker"):

    res = rolling_capm_numpy(
        sub,
        ret_col="ret",
        mkt_col="mkt_ret",
        window=ROLL_WINDOW
    )

    if res.empty:
        continue

    res["ticker"] = ticker

    if "issuer" in sub.columns:
        issuer_vals = sub["issuer"].dropna().astype("string").unique()
        res["issuer"] = issuer_vals[0] if len(issuer_vals) > 0 else pd.NA

    results.append(res)

if len(results) == 0:
    raise ValueError("No se pudieron calcular betas/ivol. Revisar cobertura.")

equity_features = pd.concat(results, ignore_index=True)
equity_features["date"] = pd.to_datetime(equity_features["date"], errors="coerce")


# ==========================================================
# 8.7 MENSUALIZACIÓN (FIRMA–MES)
# ==========================================================

equity_features["month"] = equity_features["date"].dt.to_period("M")

equity_features = equity_features.sort_values(["ticker", "date"]).copy()

equity_features_monthly = (
    equity_features
    .groupby(["ticker", "month"], as_index=False)
    .tail(1)
    .copy()
)

equity_features_monthly["date"] = equity_features_monthly["month"].dt.to_timestamp("M")
equity_features_monthly = equity_features_monthly.drop(columns=["month"])


# ==========================================================
# 8.8 EXPORT
# ==========================================================

OUT_EQ_PARQ = INTERIM / "equity_features_monthly.parquet"
OUT_EQ_CSV  = INTERIM / "equity_features_monthly.csv"

equity_features_monthly.to_parquet(OUT_EQ_PARQ, index=False)
equity_features_monthly.to_csv(OUT_EQ_CSV, index=False)

print("\n✅ equity_features_monthly guardado en:")
print("-", OUT_EQ_PARQ)
print("-", OUT_EQ_CSV)

In [ ]:
print("\nTickers con >=252 obs válidas:", (coverage_by_ticker["n_both_ok"] >= 252).sum())
print("Tickers con >=126 obs válidas:", (coverage_by_ticker["n_both_ok"] >= 126).sum())


Tickers con >=252 obs válidas: 501
Tickers con >=126 obs válidas: 502


In [ ]:
print(equity_features_monthly.shape)
print(equity_features_monthly["ivol_252"].describe())
print("NA%:", equity_features_monthly["ivol_252"].isna().mean())

(57822, 4)
count    57822.000000
mean         0.250004
std          0.109104
min          0.060697
25%          0.180954
50%          0.222997
75%          0.288375
max          1.669013
Name: ivol_252, dtype: float64
NA%: 0.0


## 9) Alternativas de volatilidad

Este bloque construye medidas alternativas de volatilidad a frecuencia mensual con el objetivo de capturar distintas dimensiones del riesgo agregado relevante para el análisis de spreads crediticios. En particular, se incorporan una medida de volatilidad sectorial basada en ETFs sectoriales y una medida de volatilidad agregada del mercado basada en el S&P 500. Ambas variables pueden utilizarse como controles complementarios en las especificaciones empíricas del panel firma–mes.

---

## 9.1 Volatilidad sectorial implícita en retornos de ETFs sectoriales

### Objetivo

El objetivo de este subbloque es construir una medida mensual de volatilidad sectorial a partir de precios diarios de ETFs sectoriales. Esta variable busca aproximar el nivel de incertidumbre o riesgo agregado específico de cada sector económico, de modo de complementar las medidas firm-level de riesgo accionario previamente estimadas.

La construcción se realiza a nivel `sector_ric` y luego se lleva a frecuencia mensual para facilitar su integración con el panel principal de análisis.

---

### Insumos

Se utiliza el archivo:

- `data/raw/sector_etfs_daily.csv`

con las columnas:

- `date`
- `price`
- `sector_ric`

donde `sector_ric` identifica el ETF representativo de cada sector.

---

### Metodología de construcción

En primer lugar, se ordena la información por sector y fecha, y se construyen retornos logarítmicos diarios a partir del precio del ETF sectorial:

$$
r_{s,t} = \log(P_{s,t}) - \log(P_{s,t-1})
$$

donde:

- $P_{s,t}$ es el precio del ETF del sector $s$ en el día $t$;
- $r_{s,t}$ es el retorno diario logarítmico correspondiente.

A continuación, para cada `sector_ric`, se calcula una medida de volatilidad rolling diaria como el desvío estándar de los retornos dentro de una ventana móvil de 60 ruedas bursátiles, requiriendo un mínimo de 45 observaciones válidas:

$$
\text{ivol\_sector\_daily}_{s,t}
=
\text{sd}\left(r_{s,t-59}, \dots, r_{s,t}\right)
$$

con la restricción de que la estimación solo se computa cuando existen al menos 45 retornos observados dentro de la ventana.

Finalmente, la serie diaria se agrega a frecuencia mensual mediante el promedio de la volatilidad diaria observada dentro de cada mes calendario para cada sector. La fecha mensual se fija al fin de mes.

---

### Validaciones y controles

Se implementan los siguientes controles:

- verificación de existencia de las columnas requeridas;
- conversión y validación de tipos (`date`, `price`, `sector_ric`);
- eliminación de observaciones faltantes en variables clave;
- chequeo de cobertura temporal;
- cuantificación de valores faltantes en la variable final;
- revisión de estadísticos descriptivos y valores extremos;
- identificación de sectores con mayor y menor volatilidad promedio.

Estos controles permiten asegurar trazabilidad y plausibilidad económica de la serie construida.

---

### Salida

El subbloque genera el archivo:

- `data/intermediate/ivol_sector_monthly_60d.parquet`

con la variable:

- `ivol_sector`

a nivel `sector_ric`–mes.

---

### Alcance interpretativo

La variable `ivol_sector` captura una medida de volatilidad realizada a nivel sectorial, basada en el comportamiento diario de ETFs representativos. Su interpretación económica es la de un proxy de riesgo agregado sectorial, útil para controlar por shocks comunes dentro de cada industria o segmento del mercado accionario.

Dado que esta medida se construye a partir de ETFs y no de retornos de bonos individuales, su rol en el análisis es complementario. No modifica la medición del spread crediticio, sino que aporta información contextual sobre el entorno de riesgo en el que opera cada firma.

---

## 9.2 Volatilidad agregada del mercado (S&P 500)

### Objetivo

Este subbloque construye una medida mensual de volatilidad agregada del mercado a partir de información diaria del S&P 500. Su objetivo es capturar variaciones en la incertidumbre general del mercado accionario, que pueden afectar de manera transversal el precio del riesgo y, en consecuencia, los spreads de bonos corporativos.

---

### Procedimiento

En primer lugar, se cargan precios o retornos diarios del índice de mercado. Si el archivo de entrada ya contiene retornos, estos se utilizan directamente. En caso contrario, los retornos logarítmicos diarios se construyen a partir de precios del índice:

$$
r_{m,t} = \log(P_{m,t}) - \log(P_{m,t-1})
$$

Luego, se estima una medida diaria de volatilidad rolling del mercado como el desvío estándar de los retornos en una ventana móvil de 60 ruedas:

$$
\text{mkt\_vol\_60d}_{t}
=
\text{sd}\left(r_{m,t-59}, \dots, r_{m,t}\right)
$$

Finalmente, la serie diaria se mensualiza conservando la última observación disponible de cada mes, de modo de mantener consistencia temporal con el resto de las variables del panel firma–mes.

---

### Output

El bloque genera:

- `mkt_vol_monthly_60d.parquet`

con la variable:

- `mkt_vol_60d`

---

### Alcance interpretativo

La variable `mkt_vol_60d` resume el nivel de volatilidad reciente del mercado accionario agregado. Su inclusión en la estrategia empírica permite controlar por shocks de riesgo sistémico que afectan simultáneamente a múltiples firmas y sectores.

A diferencia de la volatilidad sectorial, esta medida no distingue heterogeneidad entre industrias, sino que representa el componente agregado de incertidumbre financiera observable en el mercado.

In [ ]:
# ==========================================================
# 9.1 ALTERNATIVA: VOLATILIDAD SECTORIAL (ETF SECTORIAL)
# Input:
#   - sector_etfs_daily.csv con columnas: date, price, sector_ric
# Output:
#   - ivol_sector_monthly_{WINDOW}d.parquet
# ==========================================================

import pandas as pd
import numpy as np


# ==========================================================
# 9.1.1 PARÁMETROS
# ==========================================================

WINDOW = 60
MIN_OBS = 45


# ==========================================================
# 9.1.2 CARGA DE DATOS
# ==========================================================

path = RAW / "sector_etfs_daily.csv"

if not path.exists():
    raise FileNotFoundError(f"No encontré {path}")

etf = pd.read_csv(path)

required = {"date", "price", "sector_ric"}
if not required.issubset(etf.columns):
    raise ValueError(f"Faltan columnas. Requeridas: {required}. Tiene: {list(etf.columns)}")


# ==========================================================
# 9.1.3 LIMPIEZA Y ESTANDARIZACIÓN
# ==========================================================

etf["date"] = pd.to_datetime(etf["date"], errors="coerce")
etf["price"] = pd.to_numeric(etf["price"], errors="coerce")
etf["sector_ric"] = etf["sector_ric"].astype(str)

etf = etf.dropna(subset=["date", "price", "sector_ric"]).copy()
etf = etf.sort_values(["sector_ric", "date"])


# ==========================================================
# 9.1.4 CONSTRUCCIÓN DE RETORNOS DIARIOS POR SECTOR
# ==========================================================

etf["ret"] = (
    etf.groupby("sector_ric")["price"]
       .apply(lambda s: np.log(s).diff())
       .reset_index(level=0, drop=True)
)


# ==========================================================
# 9.1.5 VOLATILIDAD SECTORIAL DIARIA (ROLLING STD)
# ==========================================================

etf["ivol_sector_daily"] = (
    etf.groupby("sector_ric")["ret"]
       .rolling(WINDOW, min_periods=MIN_OBS)
       .std()
       .reset_index(level=0, drop=True)
)


# ==========================================================
# 9.1.6 MENSUALIZACIÓN
# ==========================================================

etf["month"] = etf["date"].dt.to_period("M").dt.to_timestamp("M")

ivol_sector_monthly = (
    etf.groupby(["sector_ric", "month"], as_index=False)
       .agg(ivol_sector=("ivol_sector_daily", "mean"))
       .rename(columns={"month": "date"})
)


# ==========================================================
# 9.1.7 EXPORTACIÓN
# ==========================================================

out_path = INTERIM / f"ivol_sector_monthly_{WINDOW}d.parquet"
ivol_sector_monthly.to_parquet(out_path, index=False)

print("✅ IVOL sectorial mensual guardada en:", out_path)
print(ivol_sector_monthly.head())


# ==========================================================
# 9.1.8 DIAGNÓSTICO Y CONTROL DE CALIDAD
# ==========================================================

df = ivol_sector_monthly.copy()

print("\n--- Cobertura ---")
print("Sectores (sector_ric):", df["sector_ric"].nunique())
print("Obs:", len(df))
print("Rango:", df["date"].min(), "->", df["date"].max())

print("\n--- Missingness ---")
print("NaN ivol_sector:", df["ivol_sector"].isna().sum())

print("\n--- Estadísticos ---")
print(df["ivol_sector"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

print("\n--- Checks de plausibilidad ---")
print("IVOL <= 0:", (df["ivol_sector"] <= 0).sum())
p95 = df["ivol_sector"].quantile(0.95)
print("IVOL p95:", round(float(p95), 6))

print("\n--- Sectores más volátiles (promedio) ---")
print(df.groupby("sector_ric")["ivol_sector"].mean().sort_values(ascending=False).head(10))

print("\n--- Sectores menos volátiles (promedio) ---")
print(df.groupby("sector_ric")["ivol_sector"].mean().sort_values().head(10))

print("\n--- Ejemplo: meses con IVOL extrema ---")
print(
    df.sort_values("ivol_sector", ascending=False)
      .head(10)[["sector_ric", "date", "ivol_sector"]]
)

## 9.2 Volatilidad agregada del mercado (S&P 500)

### Objetivo

El objetivo de este subbloque es construir una medida mensual de volatilidad agregada del mercado accionario a partir de información diaria del índice S&P 500. Esta variable permite capturar variaciones en el nivel de incertidumbre sistémica del mercado, las cuales pueden influir de manera transversal sobre los spreads de bonos corporativos.

---

### Insumos

Se utilizan datasets de mercado que pueden contener:

- precios diarios del índice S&P 500; o
- retornos diarios previamente calculados.

Las posibles fuentes incluyen:

- `market_sp500.*`
- `market_returns_daily.*`
- `market_prices_daily.*`

---

### Metodología de construcción

En primer lugar, se normalizan las columnas del dataset para identificar correctamente las variables de fecha y retornos del mercado.

Si el dataset contiene retornos diarios, estos se utilizan directamente. En caso contrario, los retornos logarítmicos se construyen a partir de precios:

$$
r_{m,t} = \log(P_{m,t}) - \log(P_{m,t-1})
$$

donde:

- $P_{m,t}$ es el nivel del índice en el día $t$;
- $r_{m,t}$ es el retorno logarítmico diario del mercado.

A continuación, se calcula la volatilidad diaria del mercado como el desvío estándar de los retornos en una ventana móvil de 60 ruedas bursátiles, requiriendo un mínimo de 45 observaciones válidas:

$$
\text{mkt\_vol\_daily}_{t}
=
\text{sd}\left(r_{m,t-59}, \dots, r_{m,t}\right)
$$

Finalmente, la serie se lleva a frecuencia mensual seleccionando la **última observación disponible dentro de cada mes**, de modo de mantener consistencia temporal con el panel firma–mes.

---

### Validaciones y controles

Se implementan los siguientes controles:

- verificación de la existencia de la columna `date`;
- identificación correcta de retornos o construcción desde precios;
- eliminación de valores no finitos (`NaN`, `inf`);
- chequeo de cobertura temporal;
- análisis descriptivo de la serie resultante;
- detección de valores extremos;
- conteo de valores faltantes.

Estos controles permiten asegurar la calidad y coherencia de la serie de volatilidad agregada.

---

### Salida

El bloque genera:

- `data/intermediate/mkt_vol_monthly_60d.parquet`
- `data/intermediate/mkt_vol_monthly_60d.csv`

con la variable:

- `mkt_vol_60d`

---

### Alcance interpretativo

La variable `mkt_vol_60d` representa una medida de volatilidad realizada del mercado accionario agregado. Su rol en el análisis empírico es capturar shocks de riesgo sistémico que afectan simultáneamente a todas las firmas.

A diferencia de la volatilidad sectorial, esta medida no incorpora heterogeneidad entre industrias, sino que refleja el nivel general de incertidumbre financiera observable en el mercado.

In [ ]:
# ==========================================================
# 9.2 VOLATILIDAD DEL MERCADO (S&P 500)
# Rolling std de retornos diarios
# Output: mkt_vol_monthly_{WINDOW}d.parquet
# ==========================================================

import pandas as pd
import numpy as np


# ==========================================================
# 9.2.1 PARÁMETROS
# ==========================================================

WINDOW = 60
MIN_OBS = 45


# ==========================================================
# 9.2.2 HELPERS
# ==========================================================

def read_dataset(filename_candidates, folders):
    """
    Busca y carga un dataset dentro de múltiples carpetas y formatos.
    """
    for folder in folders:
        for fname in filename_candidates:
            path = folder / fname
            if path.exists():
                print(f"📂 Leyendo: {path}")
                if path.suffix == ".parquet":
                    return pd.read_parquet(path)
                elif path.suffix == ".csv":
                    return pd.read_csv(path)

    tried = [str(folder / fname) for folder in folders for fname in filename_candidates]
    raise FileNotFoundError("No se encontró ninguno de estos archivos:\n" + "\n".join(tried))


def standardize_columns(df):
    """
    Limpia nombres de columnas.
    """
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


# ==========================================================
# 9.2.3 CARGA DE DATOS DE MERCADO
# ==========================================================

search_order = [CLEAN, INTERIM, RAW]

mkt = read_dataset(
    [
        "market_sp500.parquet",
        "market_sp500.csv",
        "market_returns_daily.parquet",
        "market_returns_daily.csv",
        "market_prices_daily.parquet",
        "market_prices_daily.csv",
    ],
    search_order
)

mkt = standardize_columns(mkt)

print("\nColumnas market:", mkt.columns.tolist())


# ==========================================================
# 9.2.4 NORMALIZACIÓN DE COLUMNAS
# ==========================================================

rename_mkt = {}

for c in mkt.columns:
    cl = c.lower().strip()

    if cl == "date":
        rename_mkt[c] = "date"

    elif cl in {"mkt_ret", "market_ret", "ret", "return",
                "mkt_ret_log", "market_ret_log", "ret_log", "log_return", "log_ret"}:
        rename_mkt[c] = "mkt_ret"

    elif cl in {"close", "price", "px_last", "market_price", "index_level",
                "mkt_close", "market_close", "close_index"}:
        rename_mkt[c] = "price"

mkt = mkt.rename(columns=rename_mkt).copy()

if "date" not in mkt.columns:
    raise ValueError(f"El dataset de mercado debe tener columna 'date'. Tiene: {list(mkt.columns)}")

mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce").dt.normalize()
mkt = mkt.sort_values("date").copy()


# ==========================================================
# 9.2.5 CONSTRUCCIÓN / VALIDACIÓN DE RETORNOS
# ==========================================================

if "mkt_ret" in mkt.columns:
    mkt["mkt_ret"] = pd.to_numeric(mkt["mkt_ret"], errors="coerce")

elif "price" in mkt.columns:
    mkt["price"] = pd.to_numeric(mkt["price"], errors="coerce")
    mkt["mkt_ret"] = np.log(mkt["price"] / mkt["price"].shift(1))

else:
    raise ValueError(
        "No encontré ni retorno ni precios en market.\n"
        f"Columnas disponibles: {list(mkt.columns)}"
    )

mkt = mkt.replace([np.inf, -np.inf], np.nan)
mkt = mkt.dropna(subset=["date", "mkt_ret"]).copy()

print("\nPreview market normalizado:")
print(mkt[["date", "mkt_ret"]].head())


# ==========================================================
# 9.2.6 VOLATILIDAD ROLLING DIARIA
# ==========================================================

mkt["mkt_vol_daily"] = (
    mkt["mkt_ret"]
    .rolling(window=WINDOW, min_periods=MIN_OBS)
    .std()
)


# ==========================================================
# 9.2.7 MENSUALIZACIÓN (ÚLTIMA OBS DEL MES)
# ==========================================================

mkt["month"] = mkt["date"].dt.to_period("M")

mkt_vol_monthly = (
    mkt.sort_values("date")
       .groupby("month", as_index=False)
       .tail(1)
       .copy()
)

mkt_vol_monthly["date"] = mkt_vol_monthly["month"].dt.to_timestamp("M")

mkt_vol_monthly = mkt_vol_monthly[["date", "mkt_vol_daily"]].rename(
    columns={"mkt_vol_daily": f"mkt_vol_{WINDOW}d"}
)

col = f"mkt_vol_{WINDOW}d"


# ==========================================================
# 9.2.8 EXPORTACIÓN
# ==========================================================

out_parquet = INTERIM / f"mkt_vol_monthly_{WINDOW}d.parquet"
out_csv     = INTERIM / f"mkt_vol_monthly_{WINDOW}d.csv"

mkt_vol_monthly.to_parquet(out_parquet, index=False)
mkt_vol_monthly.to_csv(out_csv, index=False)


# ==========================================================
# 9.2.9 QA Y DIAGNÓSTICO
# ==========================================================

df = mkt_vol_monthly.copy()

print("\n✅ mkt_vol mensual guardada en:")
print("-", out_parquet)
print("-", out_csv)

print("\nPreview:")
print(df.head(10))

print("\nRango:", df["date"].min(), "->", df["date"].max())

print("\nStats:")
print(df[col].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

print("\nNaNs:", df[col].isna().sum())
print("p95:", df[col].quantile(0.95))
print("max:", df[col].max())

print("\nTop 10 meses más volátiles:")
print(df.sort_values(col, ascending=False).head(10))

## 10) Fundamentals trimestrales a frecuencia mensual

### Objetivo

Este bloque transforma variables contables observadas a frecuencia trimestral en una base mensual a nivel firma, con el fin de compatibilizar los fundamentals con el panel principal de análisis firma–mes.

La lógica económica de este paso consiste en asignar a cada mes la última información contable trimestral disponible para la firma, preservando así una estructura temporal coherente con la frecuencia de observación de los spreads. Sobre esa base mensualizada, se construyen luego los ratios financieros utilizados como controles en la estrategia empírica.

---

### Insumos

Se utiliza como insumo principal el archivo:

- `fundamentals_extended_q.parquet`  
  o su equivalente en formato `.csv`.

Este dataset contiene fundamentals trimestrales por firma, identificados por `ticker` y `date`, junto con variables contables en distintas nomenclaturas posibles según la extracción de Refinitiv.

---

### Metodología de construcción

En primer lugar, se cargan los fundamentals trimestrales y se normalizan las columnas clave de identificación, en particular `ticker` y `date`. Luego, la base se ordena por firma y fecha, y se eliminan observaciones inválidas en esas variables.

A continuación, la expansión de frecuencia trimestral a mensual se realiza **firma por firma**. Para cada `ticker`, se construye una grilla mensual desde el primer hasta el último mes observado, fijando las fechas al cierre de mes. Sobre esa grilla, los valores trimestrales se propagan hacia adelante mediante `forward fill`, de modo que cada mes hereda la última información contable disponible.

Formalmente, si $X_{i,t_q}$ representa un fundamental trimestral observado para la firma $i$ en la fecha trimestral $t_q$, entonces la versión mensualizada asigna:

$$
X_{i,m} = X_{i,t_q}
\quad \text{para todo mes } m \text{ posterior a } t_q \text{ hasta la siguiente actualización trimestral.}
$$

Este procedimiento permite obtener una serie mensual escalonada de fundamentals, consistente con la disponibilidad discreta de la información contable.

---

### Variables derivadas

Una vez expandida la base a frecuencia mensual, se construyen las variables derivadas del panel manteniendo las definiciones originales del notebook. Entre ellas se incluyen, según disponibilidad de columnas:

- `leverage`: deuda total sobre activos totales;
- `log_assets`: logaritmo de activos totales;
- `cash_to_assets`: efectivo sobre activos totales;
- `current_ratio`: activos corrientes sobre pasivos corrientes;
- `roa`: resultado neto sobre activos totales;
- `rollover_12m`: deuda de corto plazo sobre deuda total.

Estas variables se construyen utilizando nombres alternativos de columnas cuando existen diferencias de nomenclatura en la fuente original.

---

### Validaciones y controles

Se implementan los siguientes controles:

- verificación de existencia de columnas clave (`ticker`, `date`);
- chequeo de duplicados en la combinación `ticker-date`;
- revisión del rango temporal y cantidad de firmas cubiertas;
- control de valores faltantes en la base mensual resultante;
- evaluación específica de missingness en variables financieras derivadas;
- validación de cobertura mensual por firma;
- reemplazo de valores infinitos generados por divisiones por valores faltantes.

Estos controles buscan asegurar que la expansión temporal no genere duplicaciones, inconsistencias de clave ni ratios inválidos.

---

### Nota de implementación

La expansión de quarterly a monthly se realiza firma por firma mediante una función específica, en lugar de utilizar directamente combinaciones de `groupby(...).resample(...).reset_index()`. Esta decisión evita problemas de duplicación de etiquetas y preserva la trazabilidad del proceso dentro del pipeline.

---

### Salida

El bloque genera:

- `data/intermediate/fundamentals_monthly_enriched.parquet`
- `data/intermediate/fundamentals_monthly_enriched.csv`

---

### Alcance interpretativo

La mensualización de fundamentals no introduce información nueva, sino que redistribuye temporalmente la información contable observada trimestralmente para hacerla compatible con el panel firma–mes. En consecuencia, la variación mensual de estas variables no refleja actualizaciones contables mensuales efectivas, sino la persistencia de la última información trimestral disponible.

Este supuesto es estándar en paneles de frecuencia mensual que integran información contable y financiera, pero debe tenerse en cuenta al interpretar la dinámica temporal de los controles contables dentro de las regresiones.

In [ ]:
# ==========================================================
# 10. FUNDAMENTALS TRIMESTRALES A FRECUENCIA MENSUAL
# ==========================================================

import pandas as pd
import numpy as np


# ==========================================================
# 10.1 HELPERS
# ==========================================================

def read_dataset(filename_candidates, folders):
    """
    Busca y carga un dataset a partir de una lista de nombres posibles
    y un conjunto de carpetas candidatas.
    """
    for folder in folders:
        for fname in filename_candidates:
            path = folder / fname
            if path.exists():
                print(f"📂 Leyendo: {path}")
                if path.suffix == ".parquet":
                    return pd.read_parquet(path)
                elif path.suffix == ".csv":
                    return pd.read_csv(path)

    tried = [str(folder / fname) for folder in folders for fname in filename_candidates]
    raise FileNotFoundError("No se encontró ninguno de estos archivos:\n" + "\n".join(tried))


def standardize_columns(df):
    """
    Estandariza nombres de columnas eliminando espacios sobrantes.
    """
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def quarterly_to_monthly_ffill(df, firm_col="ticker", date_col="date"):
    """
    Expande fundamentals trimestrales a frecuencia mensual
    usando forward fill dentro de cada firma.
    """
    parts = []

    for firm, sub in df.groupby(firm_col):
        sub = sub.sort_values(date_col).copy()
        sub = sub.dropna(subset=[date_col])

        if sub.empty:
            continue

        sub = sub.drop_duplicates(subset=[date_col], keep="last").copy()
        sub = sub.set_index(date_col)

        monthly_index = pd.date_range(
            start=sub.index.min().to_period("M").to_timestamp("M"),
            end=sub.index.max().to_period("M").to_timestamp("M"),
            freq="M"
        )

        sub_monthly = sub.reindex(sub.index.union(monthly_index)).sort_index().ffill()
        sub_monthly = sub_monthly.loc[monthly_index].copy()
        sub_monthly[firm_col] = firm
        sub_monthly = sub_monthly.reset_index().rename(columns={"index": date_col})

        parts.append(sub_monthly)

    if len(parts) == 0:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)
    return out


# ==========================================================
# 10.2 CARGA DE FUNDAMENTALS TRIMESTRALES
# ==========================================================

search_order = [CLEAN, INTERIM, RAW]

fundamentals_q = read_dataset(
    [
        "fundamentals_extended_q.parquet",
        "fundamentals_extended_q.csv",
    ],
    search_order
)

fundamentals_q = standardize_columns(fundamentals_q)

print("\nColumnas fundamentals:", fundamentals_q.columns.tolist())


# ==========================================================
# 10.3 NORMALIZACIÓN DE COLUMNAS CLAVE
# ==========================================================

rename_fund = {}

for c in fundamentals_q.columns:
    cl = c.lower().strip()

    if cl == "date":
        rename_fund[c] = "date"

    elif cl in {"ticker", "instrument"}:
        rename_fund[c] = "ticker"

    elif cl in {"ric", "source_ric"} and "ticker" not in fundamentals_q.columns:
        rename_fund[c] = "ticker"

fundamentals_q = fundamentals_q.rename(columns=rename_fund).copy()

required_fund = ["date", "ticker"]
missing_fund = [c for c in required_fund if c not in fundamentals_q.columns]

if missing_fund:
    raise KeyError(
        f"Faltan columnas clave en fundamentals: {missing_fund}. "
        f"Disponibles: {fundamentals_q.columns.tolist()}"
    )

fundamentals_q["date"] = pd.to_datetime(fundamentals_q["date"], errors="coerce")
fundamentals_q["ticker"] = fundamentals_q["ticker"].astype("string").str.strip()

fundamentals_q = (
    fundamentals_q
    .dropna(subset=["date", "ticker"])
    .sort_values(["ticker", "date"])
    .copy()
)

print("\n=== QA fundamentals_q ===")
print("Shape:", fundamentals_q.shape)
print("N firms:", fundamentals_q["ticker"].nunique())
print("Date range:", fundamentals_q["date"].min(), "->", fundamentals_q["date"].max())
print("Duplicados ticker-date:", fundamentals_q.duplicated(subset=["ticker", "date"]).sum())


# ==========================================================
# 10.4 EXPANSIÓN TRIMESTRAL -> MENSUAL
# ==========================================================

fund_monthly = quarterly_to_monthly_ffill(
    fundamentals_q,
    firm_col="ticker",
    date_col="date"
)

if fund_monthly.empty:
    raise ValueError("La expansión trimestral a mensual devolvió un DataFrame vacío.")

fund_monthly = fund_monthly.sort_values(["ticker", "date"]).copy()


# ==========================================================
# 10.5 CONSTRUCCIÓN DE VARIABLES DERIVADAS
# Mantiene las definiciones originales del notebook
# ==========================================================

# Coerción numérica de columnas potencialmente relevantes
num_candidates = [
    # Versión corta
    "total_debt", "total_assets", "cash", "current_assets", "current_liabilities",
    "net_income", "long_term_debt", "short_term_debt", "revenue", "ebit", "interest_expense",

    # Versión larga / nomenclatura Refinitiv
    "total debt - actual",
    "total assets",
    "cash & short term investments",
    "total current assets",
    "total current liabilities",
    "debt - long-term - total",
    "short-term debt & current portion of long-term debt",
    "revenue from business activities - total",
    "interest expense - broker estimate",
]

for c in num_candidates:
    if c in fund_monthly.columns:
        fund_monthly[c] = pd.to_numeric(fund_monthly[c], errors="coerce")


# ==========================================================
# 10.5.1 RESOLUCIÓN DE NOMBRES EFECTIVOS
# ==========================================================

COL_TOTAL_DEBT = None
if "total_debt" in fund_monthly.columns:
    COL_TOTAL_DEBT = "total_debt"
elif "total debt - actual" in fund_monthly.columns:
    COL_TOTAL_DEBT = "total debt - actual"

COL_TOTAL_ASSETS = None
if "total_assets" in fund_monthly.columns:
    COL_TOTAL_ASSETS = "total_assets"
elif "total assets" in fund_monthly.columns:
    COL_TOTAL_ASSETS = "total assets"

COL_CASH = None
if "cash" in fund_monthly.columns:
    COL_CASH = "cash"
elif "cash & short term investments" in fund_monthly.columns:
    COL_CASH = "cash & short term investments"

COL_CURR_ASSETS = None
if "current_assets" in fund_monthly.columns:
    COL_CURR_ASSETS = "current_assets"
elif "total current assets" in fund_monthly.columns:
    COL_CURR_ASSETS = "total current assets"

COL_CURR_LIAB = None
if "current_liabilities" in fund_monthly.columns:
    COL_CURR_LIAB = "current_liabilities"
elif "total current liabilities" in fund_monthly.columns:
    COL_CURR_LIAB = "total current liabilities"

COL_NET_INCOME = "net_income" if "net_income" in fund_monthly.columns else None

COL_ST_DEBT = None
if "short_term_debt" in fund_monthly.columns:
    COL_ST_DEBT = "short_term_debt"
elif "short-term debt & current portion of long-term debt" in fund_monthly.columns:
    COL_ST_DEBT = "short-term debt & current portion of long-term debt"


# ==========================================================
# 10.5.2 CONSTRUCCIÓN DE RATIOS
# ==========================================================

# leverage
if COL_TOTAL_DEBT and COL_TOTAL_ASSETS:
    fund_monthly["leverage"] = fund_monthly[COL_TOTAL_DEBT] / fund_monthly[COL_TOTAL_ASSETS]

# log_assets
if COL_TOTAL_ASSETS:
    fund_monthly["log_assets"] = np.log(fund_monthly[COL_TOTAL_ASSETS])
    fund_monthly.loc[fund_monthly[COL_TOTAL_ASSETS] <= 0, "log_assets"] = np.nan

# cash_to_assets
if COL_CASH and COL_TOTAL_ASSETS:
    fund_monthly["cash_to_assets"] = fund_monthly[COL_CASH] / fund_monthly[COL_TOTAL_ASSETS]

# current_ratio
if COL_CURR_ASSETS and COL_CURR_LIAB:
    fund_monthly["current_ratio"] = fund_monthly[COL_CURR_ASSETS] / fund_monthly[COL_CURR_LIAB]

# roa
if COL_NET_INCOME and COL_TOTAL_ASSETS:
    fund_monthly["roa"] = fund_monthly[COL_NET_INCOME] / fund_monthly[COL_TOTAL_ASSETS]

# rollover_12m
if COL_ST_DEBT and COL_TOTAL_DEBT:
    fund_monthly["rollover_12m"] = fund_monthly[COL_ST_DEBT] / fund_monthly[COL_TOTAL_DEBT]
    fund_monthly.loc[fund_monthly[COL_TOTAL_DEBT] <= 0, "rollover_12m"] = np.nan
    print(f"✔ rollover_12m construido usando: {COL_ST_DEBT} / {COL_TOTAL_DEBT}")
else:
    print("⚠️ No se pudo construir rollover_12m.")
    print("COL_ST_DEBT =", COL_ST_DEBT)
    print("COL_TOTAL_DEBT =", COL_TOTAL_DEBT)

# Limpieza de infinitos generados por divisiones
fund_monthly = fund_monthly.replace([np.inf, -np.inf], np.nan)


# ==========================================================
# 10.6 QA DEL OUTPUT MENSUAL
# ==========================================================

print("\n=== QA fund_monthly ===")
print("Shape:", fund_monthly.shape)
print("N firms:", fund_monthly["ticker"].nunique())
print("Date range:", fund_monthly["date"].min(), "->", fund_monthly["date"].max())

dup = fund_monthly.duplicated(subset=["ticker", "date"]).sum()
print("Duplicados ticker-date:", dup)
assert dup == 0, "Hay duplicados ticker-date en fundamentals."

print("\nMissing por columna (%):")
print((fund_monthly.isna().mean() * 100).sort_values(ascending=False).head(20))

key_vars = [
    c for c in [
        "leverage",
        "log_assets",
        "cash_to_assets",
        "current_ratio",
        "roa",
        "rollover_12m",
    ]
    if c in fund_monthly.columns
]

if key_vars:
    print("\nMissing variables clave (%):")
    print((fund_monthly[key_vars].isna().mean() * 100).sort_values(ascending=False))

if "rollover_12m" in fund_monthly.columns:
    print("\nQA rollover_12m:")
    print(fund_monthly["rollover_12m"].describe())
    print("% NaN rollover_12m:", fund_monthly["rollover_12m"].isna().mean() * 100)

print("\nCobertura por firma (mensual):")
coverage = (
    fund_monthly.groupby("ticker")
    .size()
    .describe()
)
print(coverage)

print("\nPreview:")
print(fund_monthly.head())


# ==========================================================
# 10.7 EXPORT INTERMEDIATE
# ==========================================================

OUT_FUND_PARQ = INTERIM / "fundamentals_monthly_enriched.parquet"
OUT_FUND_CSV  = INTERIM / "fundamentals_monthly_enriched.csv"

fund_monthly.to_parquet(OUT_FUND_PARQ, index=False)
fund_monthly.to_csv(OUT_FUND_CSV, index=False)

print("\n✅ fundamentals_monthly_enriched guardado en:")
print("-", OUT_FUND_PARQ)
print("-", OUT_FUND_CSV)

## 11) Poder de mercado (market share): preservación de múltiples variantes

### Objetivo

Este bloque construye y preserva múltiples proxies de poder de mercado a partir de la fuente específica descargada para market power. La lógica de esta etapa no consiste en imponer ex ante una única medida definitiva de market share, sino en conservar simultáneamente distintas alternativas plausibles para que la selección final de la proxy principal se realice en el notebook de modelos econométricos.

Este enfoque permite mantener flexibilidad analítica y evaluar, en una instancia posterior, cuál de las medidas disponibles presenta mejor cobertura, mayor variabilidad útil y mejor desempeño empírico en las regresiones principales y de robustez.

---

### Insumos

Se utiliza una base específica de fundamentals con información de ventas y, cuando están disponibles, variables preconstruidas de market share. El bloque busca automáticamente el primer archivo existente entre distintas rutas y formatos posibles, incluyendo variantes en `RAW` e `INTERIM`.

La base de entrada puede contener, entre otras, las siguientes variables relevantes:

- identificador de firma (`ticker`, `ric`, `ric_base` o `instrument`);
- variable temporal (`date` u otras equivalentes);
- ventas (`revenue` u otros nombres alternativos);
- clasificaciones GICS:
  - `gics_sector_name`
  - `gics_industry_group_name`
  - `gics_industry_name`
  - `gics_subindustry_name`
- proxies ya descargadas en la fuente, tales como:
  - `market_share`
  - `market_share_raw`
  - `market_share_w`

---

### Metodología de construcción

En primer lugar, se carga la base específica de market power y se normalizan las columnas de fecha, identificador de firma y ventas. La fecha se lleva a frecuencia mensual, fijando las observaciones al cierre de mes.

Luego, se preservan las proxies de market share ya disponibles en el archivo fuente, cuando existen. En particular:

- `market_share_raw`: market share ya calculado en la fuente original;
- `market_share_w`: versión alternativa también disponible o reconstruida sobre ventas totales mensuales de la muestra.

Adicionalmente, se recalculan distintas variantes de market share según el nivel de agregación sectorial disponible. Para cada firma $i$ en el período $t$, la fórmula general es:

$$
\text{market share}_{i,t}^{(g)}
=
\frac{\text{revenue}_{i,t}}
{\sum_{j \in g,t} \text{revenue}_{j,t}}
$$

donde $g$ representa el grupo de referencia utilizado para la agregación. Según disponibilidad de columnas, se preservan las siguientes variantes:

- `market_share_sector`: calculado dentro de `gics_sector_name`;
- `market_share_industry_group`: calculado dentro de `gics_industry_group_name`;
- `market_share_industry`: calculado dentro de `gics_industry_name`;
- `market_share_subindustry`: calculado dentro de `gics_subindustry_name`.

Asimismo, se construyen y conservan variables auxiliares de ventas agregadas por nivel, tales como:

- `revenue`
- `total_sales_all`
- `sector_sales`
- `industry_group_sales`
- `industry_sales`
- `subindustry_sales`

Estas variables mejoran la trazabilidad del cálculo y permiten revisar posteriormente la lógica de construcción de cada proxy.

---

### Resolución de duplicados y estructura final

Dado que la fuente de market power puede contener más de una observación por firma y mes, el bloque implementa un control explícito de duplicados en la clave `ticker-date`. En caso de observarse múltiples registros para una misma firma en el mismo mes, se conserva la última observación disponible luego del ordenamiento correspondiente.

El resultado final es una base mensual con una única observación por `ticker-date`, lista para mergear con el panel principal.

---

### Ventaja metodológica

La principal ventaja de esta estrategia es que evita fijar prematuramente una única definición de poder de mercado. En cambio, permite que la elección final de la proxy principal se realice en la etapa econométrica, donde podrá evaluarse comparativamente cada alternativa en términos de:

- cobertura efectiva dentro de la muestra;
- dispersión y variabilidad temporal;
- sensibilidad a outliers;
- capacidad explicativa en las regresiones principales;
- robustez frente a definiciones alternativas de mercado relevante.

---

### Validaciones y controles

Se implementan los siguientes controles:

- identificación y estandarización de variables clave (`date`, `ticker`, `revenue`);
- verificación de cobertura de las clasificaciones GICS;
- detección de duplicados en la clave `ticker-date`;
- revisión de valores faltantes por columna;
- chequeos de plausibilidad sobre las variables de market share:
  - valores faltantes,
  - valores negativos,
  - valores mayores a 1,
  - percentiles altos y máximos;
- análisis de cobertura por firma.

Estos controles permiten verificar que las proxies construidas sean internamente consistentes y aptas para su uso posterior en modelos econométricos.

---

### Output

El bloque genera:

- `INTERIM/market_power_monthly.parquet`
- `INTERIM/market_power_monthly.csv`

con todas las variantes disponibles de market share en formato mensual listo para mergear con el panel.

---

### Alcance interpretativo

Las variables preservadas en este bloque representan proxies empíricas de poder de mercado construidas a partir de participación en ventas dentro de distintos niveles de agregación. En consecuencia, su interpretación depende críticamente de cómo se define el mercado relevante: muestra completa, sector, industry group, industria o subindustria.

Por lo tanto, estas medidas no deben interpretarse como una noción estructural única de market power, sino como aproximaciones alternativas cuya conveniencia empírica deberá evaluarse posteriormente en función de criterios de cobertura, estabilidad y desempeño econométrico.

In [ ]:
# ==========================================================
# 11. PODER DE MERCADO (MARKET SHARE) – MENSUAL
# Construcción y preservación de todas las proxies disponibles
# ==========================================================

import pandas as pd
import numpy as np


# ==========================================================
# 11.1 HELPERS
# ==========================================================

def read_first_existing(candidates):
    """
    Lee el primer archivo existente dentro de una lista de rutas candidatas.
    """
    for path in candidates:
        if path.exists():
            print(f"📂 Leyendo: {path}")
            if path.suffix == ".parquet":
                return pd.read_parquet(path), path
            elif path.suffix == ".csv":
                return pd.read_csv(path), path

    tried = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(f"No se encontró ninguno de estos archivos:\n{tried}")


def first_existing_col(df, candidates):
    """
    Devuelve la primera columna disponible dentro de una lista de candidatas.
    """
    return next((c for c in candidates if c in df.columns), None)


def add_share_from_level(df, level_col, sales_col, out_col):
    """
    Calcula market share dentro del nivel de agregación indicado:
    share_{i,t} = revenue_{i,t} / sum_j revenue_{j,t} en ese nivel.
    """
    if level_col not in df.columns or sales_col not in df.columns:
        return pd.Series(np.nan, index=df.index, name=out_col)

    sales = pd.to_numeric(df[sales_col], errors="coerce")

    totals = (
        df.groupby([level_col, "date"], dropna=False)[sales_col]
          .transform("sum")
    )
    totals = pd.to_numeric(totals, errors="coerce")

    out = sales / totals
    out = out.replace([np.inf, -np.inf], np.nan)
    out.name = out_col

    return out


# ==========================================================
# 11.2 CARGA DE LA BASE ESPECÍFICA DE MARKET POWER
# ==========================================================

mp_candidates = [
    RAW / "fundamentals_with_market_share_sp500.csv",
    RAW / "fundamentals_with_market_share_sp500.parquet",
    RAW / "fundamentals_with_market_share.csv",
    RAW / "fundamentals_with_market_share.parquet",
    INTERIM / "fundamentals_with_market_share_sp500.csv",
    INTERIM / "fundamentals_with_market_share_sp500.parquet",
    INTERIM / "fundamentals_with_market_share.csv",
    INTERIM / "fundamentals_with_market_share.parquet",
]

fund, src_path = read_first_existing(mp_candidates)
fund.columns = [str(c).strip() for c in fund.columns]

print("\nArchivo fuente:", src_path)
print("\nColumnas disponibles:")
print(fund.columns.tolist())


# ==========================================================
# 11.3 NORMALIZACIÓN DE FECHA
# ==========================================================

date_candidates = [
    "date", "period_end", "periodend", "fiscal_date", "report_date",
    "month", "year_month", "datadate"
]
date_col = first_existing_col(fund, date_candidates)

if date_col is None:
    raise KeyError(
        "No encontré columna temporal en la base de market share.\n"
        f"Archivo usado: {src_path}\n"
        f"Busqué: {date_candidates}\n"
        f"Columnas disponibles: {fund.columns.tolist()}"
    )

if date_col != "date":
    fund = fund.rename(columns={date_col: "date"})

fund["date"] = pd.to_datetime(fund["date"], errors="coerce")
fund["date"] = fund["date"].dt.to_period("M").dt.to_timestamp("M")


# ==========================================================
# 11.4 NORMALIZACIÓN DEL IDENTIFICADOR DE FIRMA
# ==========================================================

ticker_candidates = ["ticker", "ric", "ric_base", "instrument"]
ticker_col = first_existing_col(fund, ticker_candidates)

if ticker_col is None:
    raise KeyError(f"No encontré columna ticker/ric. Columnas: {fund.columns.tolist()}")

if ticker_col != "ticker":
    fund = fund.rename(columns={ticker_col: "ticker"})

fund["ticker"] = fund["ticker"].astype("string").str.strip()

print("\nColumna usada como identificador de firma:", ticker_col)


# ==========================================================
# 11.5 NORMALIZACIÓN DE VENTAS
# ==========================================================

sales_candidates = ["revenue", "sales", "net_sales", "total_revenue", "rev", "sales_revenue"]
sales_col = first_existing_col(fund, sales_candidates)

if sales_col is not None and sales_col != "revenue":
    fund = fund.rename(columns={sales_col: "revenue"})
    sales_col = "revenue"

if "revenue" not in fund.columns:
    raise KeyError(
        "No encontré columna de ventas para construir market share. "
        f"Columnas disponibles: {fund.columns.tolist()}"
    )

fund["revenue"] = pd.to_numeric(fund["revenue"], errors="coerce")


# ==========================================================
# 11.6 VERIFICACIÓN DE NIVELES GICS DISPONIBLES
# ==========================================================

gics_cols = [
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]

print("\nCobertura GICS disponible:")
for c in gics_cols:
    if c in fund.columns:
        print(f"{c}: True | NA% = {round(fund[c].isna().mean() * 100, 2)}")
    else:
        print(f"{c}: False")


# ==========================================================
# 11.7 PRESERVAR SHARES YA DISPONIBLES EN LA FUENTE
# ==========================================================

if "market_share" in fund.columns:
    fund["market_share_raw"] = pd.to_numeric(fund["market_share"], errors="coerce")

if "market_share_raw" in fund.columns:
    fund["market_share_raw"] = pd.to_numeric(fund["market_share_raw"], errors="coerce")

if "market_share_w" in fund.columns:
    fund["market_share_w"] = pd.to_numeric(fund["market_share_w"], errors="coerce")


# ==========================================================
# 11.8 CONSTRUCCIÓN DE PROXIES DE MARKET SHARE
# ==========================================================

# 11.8.1 Share sobre ventas totales de toda la muestra en cada mes
total_sales_all = fund.groupby("date", dropna=False)["revenue"].transform("sum")
total_sales_all = pd.to_numeric(total_sales_all, errors="coerce")

fund["market_share_w"] = pd.to_numeric(fund["revenue"], errors="coerce") / total_sales_all
fund["market_share_w"] = fund["market_share_w"].replace([np.inf, -np.inf], np.nan)

# 11.8.2 Share dentro de sector
if "gics_sector_name" in fund.columns:
    fund["market_share_sector"] = add_share_from_level(
        fund,
        "gics_sector_name",
        "revenue",
        "market_share_sector"
    )

# 11.8.3 Share dentro de industry group
if "gics_industry_group_name" in fund.columns:
    fund["market_share_industry_group"] = add_share_from_level(
        fund,
        "gics_industry_group_name",
        "revenue",
        "market_share_industry_group"
    )

# 11.8.4 Share dentro de industry
if "gics_industry_name" in fund.columns:
    fund["market_share_industry"] = add_share_from_level(
        fund,
        "gics_industry_name",
        "revenue",
        "market_share_industry"
    )

# 11.8.5 Share dentro de subindustry
if "gics_subindustry_name" in fund.columns:
    fund["market_share_subindustry"] = add_share_from_level(
        fund,
        "gics_subindustry_name",
        "revenue",
        "market_share_subindustry"
    )


# ==========================================================
# 11.9 VARIABLES AUXILIARES DE VENTAS POR NIVEL
# ==========================================================

fund["total_sales_all"] = total_sales_all

if "gics_sector_name" in fund.columns:
    fund["sector_sales"] = fund.groupby(
        ["gics_sector_name", "date"], dropna=False
    )["revenue"].transform("sum")

if "gics_industry_group_name" in fund.columns:
    fund["industry_group_sales"] = fund.groupby(
        ["gics_industry_group_name", "date"], dropna=False
    )["revenue"].transform("sum")

if "gics_industry_name" in fund.columns:
    fund["industry_sales"] = fund.groupby(
        ["gics_industry_name", "date"], dropna=False
    )["revenue"].transform("sum")

if "gics_subindustry_name" in fund.columns:
    fund["subindustry_sales"] = fund.groupby(
        ["gics_subindustry_name", "date"], dropna=False
    )["revenue"].transform("sum")


# ==========================================================
# 11.10 COLUMNA SECTOR BASE PARA MERGES SIMPLES
# ==========================================================

if "gics_sector_name" in fund.columns:
    fund["sector"] = fund["gics_sector_name"]
else:
    fund["sector"] = pd.NA


# ==========================================================
# 11.11 SELECCIÓN DE COLUMNAS A EXPORTAR
# ==========================================================

cols_out = ["ticker", "date", "sector"]

optional_cols = [
    "revenue",
    "total_sales_all",
    "sector_sales",
    "industry_group_sales",
    "industry_sales",
    "subindustry_sales",
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
    "market_share_raw",
    "market_share_w",
    "market_share_sector",
    "market_share_industry_group",
    "market_share_industry",
    "market_share_subindustry",
]

cols_out += [c for c in optional_cols if c in fund.columns]

mp_out = fund[cols_out].copy()
mp_out = mp_out.dropna(subset=["ticker", "date"]).copy()


# ==========================================================
# 11.12 QA PRE-DEDUP
# ==========================================================

print("\n=== QA pre-dedup market power ===")
print("Shape:", mp_out.shape)
print("N firms:", mp_out["ticker"].nunique())
print("Rango:", mp_out["date"].min(), "->", mp_out["date"].max())

dup = mp_out.duplicated(subset=["ticker", "date"]).sum()
print("Duplicados ticker-date:", dup)

if dup > 0:
    dup_rows = mp_out[mp_out.duplicated(subset=["ticker", "date"], keep=False)].copy()
    print("\nPreview duplicados:")
    print(dup_rows.head(10))


# ==========================================================
# 11.13 RESOLUCIÓN DE DUPLICADOS TICKER-DATE
# ==========================================================

if dup > 0:
    mp_out = (
        mp_out.sort_values(["ticker", "date"])
              .drop_duplicates(subset=["ticker", "date"], keep="last")
              .copy()
    )


# ==========================================================
# 11.14 QA FINAL
# ==========================================================

print("\n=== QA final market power ===")
print("Shape:", mp_out.shape)
print("N firms:", mp_out["ticker"].nunique())
print("Rango:", mp_out["date"].min(), "->", mp_out["date"].max())
print("Duplicados ticker-date:", mp_out.duplicated(subset=["ticker", "date"]).sum())

print("\nMissing por columna (%):")
print((mp_out.isna().mean() * 100).sort_values(ascending=False))

share_vars = [c for c in mp_out.columns if c.startswith("market_share")]

print("\nVariables de market share guardadas:")
print(share_vars)

for c in share_vars:
    print(f"\n--- QA {c} ---")
    print("NA%:", round(mp_out[c].isna().mean() * 100, 2))
    print("< 0:", int((mp_out[c] < 0).sum()))
    print("> 1:", int((mp_out[c] > 1).sum()))
    print("p95:", mp_out[c].quantile(0.95))
    print("max:", mp_out[c].max())

print("\nCobertura por firma:")
print(mp_out.groupby("ticker").size().describe())

print("\nTop 10 market_share (primeras variables disponibles):")
preview_cols = ["ticker", "date", "sector"] + share_vars[:4]

if len(share_vars) > 0:
    print(mp_out.sort_values(share_vars[0], ascending=False).head(10)[preview_cols])
else:
    print(mp_out.head(10))


# ==========================================================
# 11.15 EXPORT INTERMEDIATE
# ==========================================================

OUT_MP_PARQ = INTERIM / "market_power_monthly.parquet"
OUT_MP_CSV  = INTERIM / "market_power_monthly.csv"

mp_out.to_parquet(OUT_MP_PARQ, index=False)
mp_out.to_csv(OUT_MP_CSV, index=False)

print("\n✅ market_power_monthly guardado en:")
print("-", OUT_MP_PARQ)
print("-", OUT_MP_CSV)

## 12) Factor de crédito agregado (alternativa CRC a partir de índices iBoxx)

### Objetivo

Este bloque construye una medida alternativa de exposición firm-level a shocks agregados de crédito utilizando índices mensuales iBoxx. El objetivo es aproximar una dimensión de sensibilidad crediticia de cada emisor a partir de la reacción de sus spreads ante movimientos agregados del mercado de crédito.

A diferencia de una medida contemporánea de spread, esta construcción busca capturar una **beta de crédito** estimada en ventana móvil, es decir, una exposición dinámica de cada firma a factores agregados de nivel y pendiente del mercado crediticio.

---

### Insumos

Se utilizan los siguientes archivos:

- `data/intermediate/bonds_monthly_spreads_firmlevel.parquet`
- `data/raw/iboxx_credit_indices_monthly.parquet`

o sus equivalentes en otros directorios candidatos cuando corresponda.

El primer archivo aporta la serie mensual de spreads agregados a nivel `issuer-date`, mientras que el segundo contiene índices mensuales iBoxx a partir de los cuales se construyen los shocks agregados de crédito.

---

### Construcción de los factores agregados de crédito

A partir de las series iBoxx, se construyen shocks mensuales utilizando variaciones logarítmicas. En particular, se definen dos factores principales.

El primero es un shock de nivel agregado del mercado de crédito:

$$
\text{credit\_level}_t = -\Delta \log(\text{iboxx\_ig\_liquid}_t)
$$

Esta convención de signo implica que una caída del índice se interpreta como un episodio de tightening o deterioro de las condiciones crediticias, por lo que el shock toma valores positivos en esos casos.

El segundo factor busca capturar movimientos en la pendiente de la curva de crédito:

$$
\text{credit\_slope}_t =
\left[-\Delta \log(\text{iboxx\_overall\_10y\_plus}_t)\right]
-
\left[-\Delta \log(\text{iboxx\_overall\_1\_5y}_t)\right]
$$

De este modo, `credit_slope` resume variaciones relativas entre el tramo largo y el tramo corto de la curva crediticia.

Además de estos factores legacy, el bloque conserva otras series auxiliares derivadas de los índices iBoxx, tales como niveles logarítmicos, cambios de pendiente y rezagos, con el fin de facilitar extensiones posteriores del análisis econométrico.

---

### Variable dependiente firm-level

Para cada emisor se construye la variación mensual del spread promedio en puntos básicos:

$$
\Delta \text{spread}_{i,t}
=
\text{spread\_mean\_bps}_{i,t}
-
\text{spread\_mean\_bps}_{i,t-1}
$$

Esta variable, denominada `d_spread`, constituye la variable dependiente de las regresiones rolling utilizadas para estimar la exposición de cada firma a shocks agregados de crédito.

---

### Metodología de estimación

Para cada `issuer`, se estima una regresión rolling OLS con ventana de 36 meses y un mínimo de 24 observaciones válidas. La especificación es:

$$
\Delta \text{spread}_{i,t}
=
\alpha_i
+
\beta^{L}_{i,t}\,\text{credit\_level}_t
+
\beta^{S}_{i,t}\,\text{credit\_slope}_t
+
\varepsilon_{i,t}
$$

donde:

- $\beta^{L}_{i,t}$ corresponde a `crc_level_beta`;
- $\beta^{S}_{i,t}$ corresponde a `crc_slope_beta`.

Para cada ventana se guardan:

- `crc_level_beta`
- `crc_slope_beta`
- `crc_r2`
- `crc_nobs`

Estas métricas permiten caracterizar tanto la magnitud de la exposición estimada como la calidad de ajuste de cada regresión rolling.

---

### Validaciones y controles

Se implementan los siguientes controles:

- verificación de existencia de `spread_mean_bps` en el panel firm-level;
- identificación automática de las tres series iBoxx requeridas;
- control de valores no positivos en los índices antes de aplicar logaritmos;
- revisión de cobertura temporal y missingness de los factores;
- chequeo de cobertura de emisores con suficiente cantidad de observaciones;
- análisis descriptivo de `crc_level_beta`, `crc_slope_beta`, `crc_r2` y `crc_nobs`;
- detección de valores extremos mediante percentiles.

Adicionalmente, el bloque intenta mapear `issuer` a `ticker` utilizando fuentes auxiliares disponibles en `INTERIM`, con el fin de facilitar merges posteriores con el panel principal.

---

### Output

El bloque genera:

- `data/intermediate/crc_alt_iboxx_betas_monthly.parquet`
- `data/intermediate/crc_alt_iboxx_betas_monthly.csv`

y adicionalmente exporta una base ampliada de factores agregados de crédito:

- `data/intermediate/credit_factors_iboxx_extended.parquet`
- `data/intermediate/credit_factors_iboxx_extended.csv`

---

### Alcance interpretativo

Las variables `crc_level_beta` y `crc_slope_beta` deben interpretarse como medidas de sensibilidad de los spreads firm-level a shocks agregados del mercado de crédito. No representan niveles de spread ni factores estructurales permanentes, sino coeficientes estimados en ventanas móviles que resumen la exposición reciente de cada emisor a variaciones del entorno crediticio agregado.

En consecuencia, estas betas pueden variar en el tiempo como resultado de cambios en la composición de la deuda, en la percepción de riesgo del emisor o en la propia dinámica del mercado de crédito agregado. Su utilidad principal dentro de la estrategia empírica es capturar heterogeneidad en la transmisión de shocks crediticios comunes a nivel firma–mes.

In [ ]:
# ==========================================================
# 12. CRC ALTERNATIVA (iBoxx): BETAS FIRM-LEVEL A SHOCKS AGREGADOS
# Lee:  iboxx_credit_indices_monthly.parquet
# Output: crc_alt_iboxx_betas_monthly.parquet
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm


# ==========================================================
# 12.1 HELPERS
# ==========================================================

def read_first_existing(candidates):
    """
    Lee el primer archivo existente dentro de una lista de rutas candidatas.
    """
    for path in candidates:
        if path.exists():
            print(f"📂 Leyendo: {path}")
            if path.suffix == ".parquet":
                return pd.read_parquet(path), path
            elif path.suffix == ".csv":
                return pd.read_csv(path), path

    tried = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(f"No se encontró ninguno de estos archivos:\n{tried}")


def to_eom(s):
    """
    Convierte una serie temporal a fin de mes.
    """
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")


def pick_series(cols, candidates):
    """
    Identifica una serie iBoxx por coincidencia flexible de nombres.
    """
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        for lc, orig in cols_l.items():
            if cand in lc:
                return orig
    return None


# ==========================================================
# 12.2 CARGA DE SPREADS FIRM-LEVEL
# ==========================================================

SP_FIRM = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"

if not SP_FIRM.exists():
    raise FileNotFoundError(f"No encontré {SP_FIRM}")

sp = pd.read_parquet(SP_FIRM).copy()
sp["date"] = to_eom(sp["date"])
sp = sp.sort_values(["issuer", "date"]).copy()

if "spread_mean_bps" not in sp.columns:
    raise KeyError("bonds_monthly_spreads_firmlevel.parquet debe tener 'spread_mean_bps'.")

# Variación mensual del spread por issuer
sp["d_spread"] = sp.groupby("issuer")["spread_mean_bps"].diff()


# ==========================================================
# 12.3 CARGA DE ÍNDICES iBoxx MENSUALES
# ==========================================================

iboxx_candidates = [
    RAW / "iboxx_credit_indices_monthly.parquet",
    RAW / "iboxx_credit_indices_monthly.csv",
    INTERIM / "iboxx_credit_indices_monthly.parquet",
    INTERIM / "iboxx_credit_indices_monthly.csv",
    CLEAN / "iboxx_credit_indices_monthly.parquet",
    CLEAN / "iboxx_credit_indices_monthly.csv",
]

iboxx, iboxx_path = read_first_existing(iboxx_candidates)
iboxx.columns = [str(c).strip() for c in iboxx.columns]
iboxx["date"] = to_eom(iboxx["date"])

if "value" not in iboxx.columns and "close" in iboxx.columns:
    iboxx = iboxx.rename(columns={"close": "value"})

req_cols = {"date", "series", "value"}
miss = req_cols - set(iboxx.columns)

if miss:
    raise ValueError(f"Faltan columnas en iBoxx: {miss}")

wide = iboxx.pivot(index="date", columns="series", values="value").sort_index()

print("\nSeries disponibles en iBoxx wide:")
print(list(wide.columns))


# ==========================================================
# 12.4 IDENTIFICACIÓN FLEXIBLE DE SERIES REQUERIDAS
# ==========================================================

ig_col = pick_series(
    wide.columns,
    ["iboxx_ig_liquid", "ig liquid", "investment grade", "liquid ig"]
)

s1_col = pick_series(
    wide.columns,
    ["iboxx_overall_1_5y", "1-5", "1_5", "1 to 5", "1-5y"]
)

s10_col = pick_series(
    wide.columns,
    ["iboxx_overall_10y_plus", "10y+", "10+", "10 plus", "10y plus"]
)

picked = {
    "iboxx_ig_liquid": ig_col,
    "iboxx_overall_1_5y": s1_col,
    "iboxx_overall_10y_plus": s10_col,
}

print("\nSeries seleccionadas:")
print(picked)

if any(v is None for v in picked.values()):
    raise ValueError(
        "No pude identificar automáticamente las 3 series requeridas.\n"
        f"Disponibles: {list(wide.columns)}"
    )

wide = wide[[ig_col, s1_col, s10_col]].copy()
wide.columns = ["iboxx_ig_liquid", "iboxx_overall_1_5y", "iboxx_overall_10y_plus"]

print("\nPreview iBoxx wide:")
print(wide.head())

print("\nCobertura de series iBoxx:")
print((wide.isna().mean() * 100).round(2))
print("Rango:", wide.index.min(), "->", wide.index.max())


# ==========================================================
# 12.5 CONSTRUCCIÓN DE FACTORES AGREGADOS DE CRÉDITO
# ==========================================================

# Validación para aplicar logaritmos
if (wide <= 0).any().any():
    bad_cols = wide.columns[(wide <= 0).any()].tolist()
    raise ValueError(
        f"Hay valores no positivos en series iBoxx, no puedo tomar logs. "
        f"Series afectadas: {bad_cols}"
    )

# Niveles logarítmicos
log_wide = np.log(wide)

# Alternativa 1: nivel agregado del mercado de crédito
credit_market_level_log = log_wide["iboxx_ig_liquid"]

# Alternativa 2: pendiente de la curva en nivel
credit_curve_level = (
    log_wide["iboxx_overall_10y_plus"] - log_wide["iboxx_overall_1_5y"]
)

# Cambio mensual de la pendiente en nivel
credit_curve_change = credit_curve_level.diff()

# Shocks mensuales (cambios log)
ret = log_wide.diff()

# Shock bruto del índice agregado
credit_market_return = ret["iboxx_ig_liquid"]

# Shock orientado como tightening:
# si cae el índice, sube esta variable
credit_tightening_shock = -credit_market_return

# Shocks por tramo
credit_short_return = ret["iboxx_overall_1_5y"]
credit_long_return = ret["iboxx_overall_10y_plus"]

# Shock de pendiente
credit_curve_shock = (-credit_long_return) - (-credit_short_return)

# Rezagos listos para análisis posterior
credit_tightening_shock_l1 = credit_tightening_shock.shift(1)
credit_curve_shock_l1 = credit_curve_shock.shift(1)
credit_market_level_log_l1 = credit_market_level_log.shift(1)
credit_curve_level_l1 = credit_curve_level.shift(1)
credit_curve_change_l1 = credit_curve_change.shift(1)

# Compatibilidad hacia atrás con CRC actual
credit_level = credit_tightening_shock
credit_slope = credit_curve_shock

# Consolidado de factores
factors = pd.DataFrame({
    "date": wide.index,

    # nombres legacy
    "credit_level": credit_level.values,
    "credit_slope": credit_slope.values,

    # nombres más descriptivos
    "credit_tightening_shock": credit_tightening_shock.values,
    "credit_curve_shock": credit_curve_shock.values,

    # componentes intermedios
    "credit_market_return": credit_market_return.values,
    "credit_short_return": credit_short_return.values,
    "credit_long_return": credit_long_return.values,

    # alternativas en nivel
    "credit_market_level_log": credit_market_level_log.values,
    "credit_curve_level": credit_curve_level.values,
    "credit_curve_change": credit_curve_change.values,

    # rezagos
    "credit_tightening_shock_l1": credit_tightening_shock_l1.values,
    "credit_curve_shock_l1": credit_curve_shock_l1.values,
    "credit_market_level_log_l1": credit_market_level_log_l1.values,
    "credit_curve_level_l1": credit_curve_level_l1.values,
    "credit_curve_change_l1": credit_curve_change_l1.values,
})

factors = factors.sort_values("date").reset_index(drop=True)

print("\nPreview factores de crédito:")
print(factors.head())

print("\nCobertura / missing de factores:")
print((factors.isna().mean() * 100).round(2))

print("\nResumen factores principales:")
print(
    factors[
        [
            "credit_level",
            "credit_slope",
            "credit_tightening_shock",
            "credit_curve_shock",
            "credit_market_level_log",
            "credit_curve_level",
            "credit_curve_change",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

print("\nCorrelación entre factores principales:")
print(
    factors[
        [
            "credit_level",
            "credit_slope",
            "credit_tightening_shock",
            "credit_curve_shock",
            "credit_market_level_log",
            "credit_curve_level",
            "credit_curve_change",
        ]
    ].corr()
)


# ==========================================================
# 12.6 ESTIMACIÓN DE BETAS ROLLING POR ISSUER
# ==========================================================

WINDOW = 36
MIN_OBS = 24

rows = []

for issuer, g in sp.groupby("issuer"):
    tmp = (
        g[["date", "d_spread"]]
        .merge(factors, on="date", how="inner")
        .dropna(subset=["d_spread", "credit_level", "credit_slope"])
        .sort_values("date")
    )

    if len(tmp) < MIN_OBS:
        continue

    for i in range(MIN_OBS, len(tmp) + 1):
        sub = tmp.iloc[max(0, i - WINDOW):i]

        if len(sub) < MIN_OBS:
            continue

        X = sm.add_constant(sub[["credit_level", "credit_slope"]])
        y = sub["d_spread"]

        try:
            res = sm.OLS(y, X).fit()
        except Exception:
            continue

        rows.append({
            "issuer": issuer,
            "date": sub["date"].iloc[-1],
            "crc_level_beta": float(res.params.get("credit_level", np.nan)),
            "crc_slope_beta": float(res.params.get("credit_slope", np.nan)),
            "crc_r2": float(res.rsquared),
            "crc_nobs": int(res.nobs),
        })

crc_alt = pd.DataFrame(rows)

if crc_alt.empty:
    raise ValueError("crc_alt quedó vacío. Revisá cobertura de spreads e iBoxx.")

crc_alt = crc_alt.sort_values(["issuer", "date"]).copy()


# ==========================================================
# 12.7 MAPEO ISSUER -> TICKER
# ==========================================================

issuer_map_parts = []

BONDS_MONTHLY_PARQ = INTERIM / "bonds_monthly_spreads.parquet"
BONDS_MONTHLY_CSV = INTERIM / "bonds_monthly_spreads.csv"

if BONDS_MONTHLY_PARQ.exists():
    bonds_map = pd.read_parquet(BONDS_MONTHLY_PARQ)
elif BONDS_MONTHLY_CSV.exists():
    bonds_map = pd.read_csv(BONDS_MONTHLY_CSV)
else:
    bonds_map = None

if bonds_map is not None:
    bonds_map.columns = [str(c).strip() for c in bonds_map.columns]
    if {"issuer", "ticker"}.issubset(bonds_map.columns):
        bonds_map["issuer"] = bonds_map["issuer"].astype("string").str.strip()
        bonds_map["ticker"] = bonds_map["ticker"].astype("string").str.strip()
        tmp_map = bonds_map[["issuer", "ticker"]].dropna().drop_duplicates()
        issuer_map_parts.append(tmp_map)
        print("\nMapping issuer→ticker desde bonds_monthly_spreads:")
        print("filas:", len(tmp_map), "| issuers únicos:", tmp_map["issuer"].nunique())

BASE_PANEL_PARQ = INTERIM / "panel_master_pre_filters.parquet"
BASE_PANEL_CSV = INTERIM / "panel_master_pre_filters.csv"

if BASE_PANEL_PARQ.exists():
    panel_map = pd.read_parquet(BASE_PANEL_PARQ)
elif BASE_PANEL_CSV.exists():
    panel_map = pd.read_csv(BASE_PANEL_CSV)
else:
    panel_map = None

if panel_map is not None:
    panel_map.columns = [str(c).strip() for c in panel_map.columns]
    if {"issuer", "ticker"}.issubset(panel_map.columns):
        panel_map["issuer"] = panel_map["issuer"].astype("string").str.strip()
        panel_map["ticker"] = panel_map["ticker"].astype("string").str.strip()
        tmp_map = panel_map[["issuer", "ticker"]].dropna().drop_duplicates()
        issuer_map_parts.append(tmp_map)
        print("\nMapping issuer→ticker desde panel_master_pre_filters:")
        print("filas:", len(tmp_map), "| issuers únicos:", tmp_map["issuer"].nunique())

if issuer_map_parts:
    issuer_map = pd.concat(issuer_map_parts, ignore_index=True).drop_duplicates()
    crc_alt = crc_alt.merge(issuer_map, on="issuer", how="left")
else:
    print("\n⚠ No encontré fuentes para mapear issuer→ticker.")


# ==========================================================
# 12.8 EXPORTACIÓN DE OUTPUTS
# ==========================================================

OUT_PARQ = INTERIM / "crc_alt_iboxx_betas_monthly.parquet"
OUT_CSV = INTERIM / "crc_alt_iboxx_betas_monthly.csv"

crc_alt.to_parquet(OUT_PARQ, index=False)
crc_alt.to_csv(OUT_CSV, index=False)

# Factores ampliados para uso posterior en panel / econometría
FACTORS_PARQ = INTERIM / "credit_factors_iboxx_extended.parquet"
FACTORS_CSV = INTERIM / "credit_factors_iboxx_extended.csv"

factors.to_parquet(FACTORS_PARQ, index=False)
factors.to_csv(FACTORS_CSV, index=False)

print("\n✅ CRC alternativa guardada en:")
print("-", OUT_PARQ)
print("-", OUT_CSV)

print("\n✅ Factores de crédito ampliados guardados en:")
print("-", FACTORS_PARQ)
print("-", FACTORS_CSV)


# ==========================================================
# 12.9 DIAGNÓSTICO
# ==========================================================

print("\n" + "=" * 78)
print("SECCIÓN 12.9 — DIAGNÓSTICO CRC ALTERNATIVA (iBoxx)")
print("=" * 78)

print("\n[12.9.1] Inputs: spreads firm-level e iBoxx")
print("-" * 78)
print("Spreads firm-level (sp):")
print("  filas:", len(sp), "| issuers:", sp["issuer"].nunique())
print("  rango:", sp["date"].min(), "->", sp["date"].max())
print("  spread_mean_bps NaN%:", sp["spread_mean_bps"].isna().mean())
print("  d_spread NaN% (esperable por diff):", sp["d_spread"].isna().mean())

print("\niBoxx (wide):")
print("  rango:", wide.index.min(), "->", wide.index.max())
print("  NaN% por serie (level):")
print((wide.isna().mean() * 100).round(2))

print("\n[12.9.2] Factores")
print("-" * 78)
print("Factores: filas:", len(factors), "| rango:", factors["date"].min(), "->", factors["date"].max())

print("\nStats factores principales:")
print(
    factors[
        [
            "credit_level",
            "credit_slope",
            "credit_tightening_shock",
            "credit_curve_shock",
            "credit_market_level_log",
            "credit_curve_level",
            "credit_curve_change",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

print("\nCorrelación entre factores:")
print(
    factors[
        [
            "credit_level",
            "credit_slope",
            "credit_tightening_shock",
            "credit_curve_shock",
            "credit_market_level_log",
            "credit_curve_level",
            "credit_curve_change",
        ]
    ].corr()
)

print("\n[12.9.3] Cobertura CRC estimada")
print("-" * 78)
print("  filas:", len(crc_alt), "| issuers:", crc_alt["issuer"].nunique())

if "ticker" in crc_alt.columns:
    print("  tickers:", crc_alt["ticker"].nunique(dropna=True))

print("  rango:", crc_alt["date"].min(), "->", crc_alt["date"].max())

obs_per_issuer = crc_alt.groupby("issuer").size()

print("\nObs por issuer (resumen):")
print(obs_per_issuer.describe()[["min", "25%", "50%", "mean", "75%", "max"]])

print("\nDistribución crc_nobs:")
print(crc_alt["crc_nobs"].value_counts().sort_index())

print("\n[12.9.4] Calidad del ajuste (R²)")
print("-" * 78)
print(crc_alt["crc_r2"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\n[12.9.5] Outliers (P1/P99)")
print("-" * 78)

def outlier_block(df, col):
    p1, p99 = df[col].quantile([0.01, 0.99])
    share = ((df[col] < p1) | (df[col] > p99)).mean()
    print(f"{col}: P1={p1:,.4f} | P99={p99:,.4f} | % fuera={share*100:.2f}%")

outlier_block(crc_alt, "crc_level_beta")
outlier_block(crc_alt, "crc_slope_beta")

## 13) Construcción de `ticker_to_etf`

### Objetivo

Este bloque recrea el mapping `ticker_to_etf` a partir de la base de fundamentals con market share, con el propósito de generar un archivo auxiliar reproducible que permita vincular cada firma con un ETF sectorial.

La finalidad principal de este paso es preservar compatibilidad con la lógica del notebook anterior y facilitar el uso de proxies sectoriales basadas en ETFs en etapas posteriores del pipeline.

---

### Insumos

Se utiliza como fuente una base de fundamentals con información de market share y clasificación sectorial, buscando automáticamente entre las siguientes variantes:

- `fundamentals_with_market_share_sp500`
- `fundamentals_with_market_share`

en formatos `.parquet` o `.csv`, tanto en `RAW` como en `INTERIM`.

---

### Metodología de construcción

En primer lugar, se identifica una columna de instrumento (`ric`, `ric_base` o `instrument`) y una columna de sector económico (`gics_sector_name` u otras equivalentes disponibles).

A partir del identificador `ric`, se construye el `ticker` extrayendo el componente previo al separador `"."`, normalizándolo en mayúsculas y eliminando espacios innecesarios. Esta convención replica la lógica operativa utilizada en el notebook previo para obtener un identificador firma–sector homogéneo.

Luego, dado que una firma puede aparecer múltiples veces en la fuente, se asigna a cada `ticker` un único sector GICS utilizando el valor modal observado en la base. En caso de empate o ausencia de una moda estricta, se conserva el primer valor disponible. Formalmente, para cada firma $i$, se define:

$$
\text{gics\_sector}_i = \text{mode}\left(\text{sector}_{i,t}\right)
$$

donde el operador modo resume la categoría sectorial más frecuente observada para ese ticker en la fuente utilizada.

A continuación, se mapea cada sector GICS a un ETF sectorial estándar mediante una tabla fija de correspondencias. Entre ellas:

- Energy $\rightarrow$ `XLE`
- Materials $\rightarrow$ `XLB`
- Industrials $\rightarrow$ `XLI`
- Consumer Discretionary $\rightarrow$ `XLY`
- Consumer Staples $\rightarrow$ `XLP`
- Health Care $\rightarrow$ `XLV`
- Financials $\rightarrow$ `XLF`
- Information Technology $\rightarrow$ `XLK`
- Communication Services $\rightarrow$ `XLC`
- Utilities $\rightarrow$ `XLU`
- Real Estate $\rightarrow$ `XLRE`

Finalmente, el bloque genera una tabla auxiliar con las siguientes columnas:

- `ticker`
- `gics_sector`
- `sector_etf`
- `sector_ric`

---

### Validaciones y controles

Se implementan los siguientes controles:

- verificación de existencia de columna RIC e información sectorial;
- chequeo de cobertura de `ric`, `ticker` y `gics_sector`;
- validación de unicidad de `ticker` en la tabla final;
- cuantificación de valores faltantes en `sector_etf`;
- análisis de distribución de firmas por ETF;
- identificación de sectores sin mapping explícito.

Estos controles permiten asegurar que el archivo auxiliar resultante sea internamente consistente y utilizable en merges posteriores.

---

### Output

El bloque genera:

- `RAW/ticker_to_etf.parquet`
- `RAW/ticker_to_etf.csv`

---

### Alcance interpretativo

El archivo `ticker_to_etf` constituye un mapping auxiliar entre firmas y ETFs sectoriales. No representa una relación económica estimada, sino una asignación operativa construida a partir de la clasificación sectorial predominante de cada ticker.

En consecuencia, su función es instrumental dentro del pipeline: facilitar merges con información de ETFs sectoriales y mantener compatibilidad con notebooks previos. Su validez depende de la estabilidad del sector asignado a cada firma y de la adecuación del ETF seleccionado como proxy representativa de dicho sector.

In [ ]:
# ==========================================================
# 13. CONSTRUCCIÓN DE ticker_to_etf
# ==========================================================

import pandas as pd
import numpy as np


# ==========================================================
# 13.1 HELPERS
# ==========================================================

def read_first_existing(candidates):
    """
    Lee el primer archivo existente dentro de una lista de rutas candidatas.
    """
    for path in candidates:
        if path.exists():
            print(f"📂 Leyendo: {path}")
            if path.suffix == ".parquet":
                return pd.read_parquet(path), path
            elif path.suffix == ".csv":
                return pd.read_csv(path), path

    tried = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(f"No se encontró ninguno de estos archivos:\n{tried}")


def mode_or_first(x: pd.Series):
    """
    Devuelve la moda de la serie; si no existe una moda usable,
    devuelve el primer valor no nulo.
    """
    x = x.dropna()

    if x.empty:
        return pd.NA

    m = x.mode()
    if len(m) > 0:
        return m.iloc[0]

    return x.iloc[0]


# ==========================================================
# 13.2 CARGA DE LA BASE CON INFORMACIÓN SECTORIAL
# ==========================================================

fms_candidates = [
    RAW / "fundamentals_with_market_share_sp500.parquet",
    RAW / "fundamentals_with_market_share_sp500.csv",
    RAW / "fundamentals_with_market_share.parquet",
    RAW / "fundamentals_with_market_share.csv",
    INTERIM / "fundamentals_with_market_share_sp500.parquet",
    INTERIM / "fundamentals_with_market_share_sp500.csv",
    INTERIM / "fundamentals_with_market_share.parquet",
    INTERIM / "fundamentals_with_market_share.csv",
]

fms, fms_path = read_first_existing(fms_candidates)
fms.columns = [str(c).strip() for c in fms.columns]

print("\nColumnas disponibles en fundamentals_with_market_share:")
print(fms.columns.tolist())


# ==========================================================
# 13.3 DETECCIÓN DE COLUMNAS CLAVE
# ==========================================================

ric_candidates = ["ric", "ric_base", "instrument"]
ric_col = next((c for c in ric_candidates if c in fms.columns), None)

sector_candidates = [
    "gics_sector_name",
    "gics_sector",
    "sector",
    "sector_name",
    "trbc_sector",
]
sector_col = next((c for c in sector_candidates if c in fms.columns), None)

if ric_col is None:
    raise KeyError(
        f"No encontré una columna RIC en fundamentals_with_market_share. "
        f"Busqué: {ric_candidates}. Tiene: {fms.columns.tolist()}"
    )

if sector_col is None:
    raise KeyError(
        f"No encontré una columna de sector en fundamentals_with_market_share. "
        f"Busqué: {sector_candidates}. Tiene: {fms.columns.tolist()}"
    )


# ==========================================================
# 13.4 CONSTRUCCIÓN DE TICKER DESDE RIC
# ==========================================================

fms["ric"] = fms[ric_col].astype("string").str.strip()

fms["ticker"] = (
    fms["ric"]
    .str.split(".").str[0]
    .str.upper()
    .str.strip()
)

fms["gics_sector"] = fms[sector_col].astype("string").str.strip()

print("\n=== QA base ticker/sector ===")
print("Shape:", fms.shape)
print("RIC no nulos:", fms["ric"].notna().sum())
print("Ticker no nulos:", fms["ticker"].notna().sum())
print("Sectores no nulos:", fms["gics_sector"].notna().sum())


# ==========================================================
# 13.5 ASIGNACIÓN DE UN ÚNICO SECTOR POR TICKER
# ==========================================================

sector_per_ticker = (
    fms.dropna(subset=["ticker", "gics_sector"])
       .groupby("ticker", as_index=False)["gics_sector"]
       .agg(mode_or_first)
)


# ==========================================================
# 13.6 MAPEO GICS -> ETF SECTORIAL
# ==========================================================

GICS_TO_ETF = {
    "Energy": "XLE",
    "Materials": "XLB",
    "Industrials": "XLI",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Health Care": "XLV",
    "Financials": "XLF",
    "Information Technology": "XLK",
    "Communication Services": "XLC",
    "Utilities": "XLU",
    "Real Estate": "XLRE",
}

sector_per_ticker["sector_etf"] = sector_per_ticker["gics_sector"].map(GICS_TO_ETF)
sector_per_ticker["sector_ric"] = sector_per_ticker["sector_etf"]

ticker_to_etf = sector_per_ticker[
    ["ticker", "gics_sector", "sector_etf", "sector_ric"]
].copy()


# ==========================================================
# 13.7 QA DEL OUTPUT
# ==========================================================

print("\n=== QA ticker_to_etf ===")
print("Shape:", ticker_to_etf.shape)
print("Tickers únicos:", ticker_to_etf["ticker"].nunique())
print("Duplicados ticker:", ticker_to_etf.duplicated(["ticker"]).sum())
print("Sectores GICS únicos:", ticker_to_etf["gics_sector"].nunique())

print("\nCobertura sector_etf:")
print("NA% sector_etf:", round(ticker_to_etf["sector_etf"].isna().mean() * 100, 2))

print("\nDistribución por ETF:")
print(ticker_to_etf["sector_etf"].value_counts(dropna=False))

unmapped = ticker_to_etf[ticker_to_etf["sector_etf"].isna()]
if not unmapped.empty:
    print("\n⚠️ Sectores sin mapping a ETF:")
    print(unmapped["gics_sector"].drop_duplicates().tolist())

print("\nPreview:")
print(ticker_to_etf.head(10))


# ==========================================================
# 13.8 EXPORTACIÓN
# ==========================================================

OUT_PARQ = RAW / "ticker_to_etf.parquet"
OUT_CSV  = RAW / "ticker_to_etf.csv"

ticker_to_etf.to_parquet(OUT_PARQ, index=False)
ticker_to_etf.to_csv(OUT_CSV, index=False)

print("\n✅ ticker_to_etf guardado en:")
print("-", OUT_PARQ)
print("-", OUT_CSV)

# 14. Merge master del panel

En esta sección se consolida el panel mensual firma–fecha a partir de un backbone único definido a nivel `issuer-date`, incorporando secuencialmente los distintos bloques de información construidos en etapas previas del pipeline.

La reorganización del merge master en sub-bloques tiene cuatro objetivos centrales:

- mejorar la trazabilidad del proceso;
- facilitar la detección de errores de integración;
- auditar la cobertura efectiva de cada familia de variables;
- y verificar que el panel final preserve estrictamente la estructura del backbone original.

En cada sub-bloque se implementan controles explícitos sobre:

- cantidad de filas antes y después del merge;
- duplicados en la llave `issuer-date`;
- porcentaje de valores faltantes en las nuevas variables incorporadas;
- y calidad del matching entre datasets.

La regla central de esta etapa es que **ningún merge puede alterar la estructura base del panel**. En consecuencia, luego de cada bloque se verifica que:

1. la cantidad de observaciones permanezca constante; y  
2. no aparezcan duplicados en la combinación `issuer-date`.

La consolidación se divide en cuatro partes:

- **14A. Base + mapping + equity**
- **14B. Fundamentals + market power**
- **14C. CRC + factores agregados + macro controls + ivol sectorial**
- **14D. OAS auxiliar + CDS + QA final + guardado del panel master**

Esta estructura permite identificar con precisión la etapa en la que se deteriora la cobertura de una variable y evita que un problema de integración quede oculto dentro de una única celda monolítica.

---

## 14A. Base del panel, mapping de identificadores y variables de equity

### Objetivo

Este primer sub-bloque construye la base operativa del merge master. Se parte del panel backbone a nivel `issuer-date`, generado a partir de la agregación mensual firm-level de spreads de bonos, y se incorporan dos insumos esenciales: el mapping de identificadores y las variables mensuales de riesgo accionario.

El objetivo específico de esta subsección es asegurar que el panel tenga resuelta de forma consistente la correspondencia entre emisor, ticker y variables de mercado accionario antes de incorporar capas adicionales de información.

---

### Insumos

Se utilizan los siguientes datasets:

- `bonds_monthly_spreads_firmlevel.parquet` como backbone del panel;
- `issuer_ticker_sector.parquet` o su equivalente `.csv` como mapping entre `issuer`, `ticker` y, cuando está disponible, `sector`;
- `equity_features_monthly.parquet` o su equivalente `.csv`, que contiene variables mensuales de equity.

---

### Metodología de construcción

En primer lugar, se carga el backbone del panel y se normaliza la fecha a fin de mes. La llave estructural del merge master queda definida por la combinación `issuer-date`, que debe ser única desde el inicio del proceso.

Luego se incorpora un mapping de identificadores que vincula cada `issuer` con su `ticker` y, cuando la información está disponible, con su clasificación sectorial. Si la base de mapping presenta múltiples observaciones por `issuer`, se conserva la última fila disponible para asegurar unicidad en el merge. Sobre el `ticker` resultante se construye adicionalmente la variable `ticker_base`, eliminando sufijos posteriores a `"."`, con el fin de homogeneizar identificadores cuando existen variantes instrumentales del mismo ticker.

Formalmente, este paso puede resumirse como:

$$
\text{panel}_{1}
=
\text{panel}_{0}
\;\underset{\text{issuer}}{\text{LEFT JOIN}}\;
\text{issuer\_map}
$$

A continuación, se incorporan las variables de equity mensual a partir del dataset de `equity_features_monthly`. El merge se realiza utilizando la llave `ticker_base-date`, luego de:

- normalizar `ticker` y `date`;
- construir `ticker_base`;
- deduplicar observaciones en la combinación `ticker_base-date`.

Las variables incorporadas en esta etapa son:

- `beta_252`
- `ivol_252`

El merge correspondiente puede expresarse como:

$$
\text{panel}_{2}
=
\text{panel}_{1}
\;\underset{(\text{ticker\_base},\,\text{date})}{\text{LEFT JOIN}}\;
\text{equity\_features}
$$

---

### Validaciones y controles

Durante este sub-bloque se auditan de forma explícita:

- consistencia inicial del backbone (`issuer-date` único);
- cobertura del mapping `issuer \rightarrow ticker`;
- duplicados por `issuer` dentro del mapping;
- tasa de matching entre el panel y la base de equity;
- duplicados en la clave `ticker_base-date` de equity;
- porcentaje de valores faltantes en:
  - `ticker`
  - `ticker_base`
  - `sector`
  - `beta_252`
  - `ivol_252`

Asimismo, al cierre del bloque se verifican dos condiciones estructurales:

1. que la cantidad de filas del panel coincida exactamente con la del backbone original;  
2. que no existan duplicados en `issuer-date`.

---

### Alcance interpretativo

Este sub-bloque no modifica la definición económica de la variable dependiente ni altera la estructura del panel original. Su función es exclusivamente integradora: resolver la vinculación entre emisor, ticker y variables de riesgo accionario mensual.

En consecuencia, cualquier pérdida de cobertura en `beta_252` o `ivol_252` debe interpretarse como una limitación de matching entre identificadores o de disponibilidad de información de equity, y no como una alteración en la construcción del backbone de spreads.

In [ ]:
# ==========================================================
# 14A. MERGE MASTER — BLOQUE 1
# Base + mapping + equity
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ==========================================================
# 14A.1 HELPERS
# ==========================================================

def to_eom(s: pd.Series) -> pd.Series:
    """
    Normaliza una serie temporal a fin de mes.
    """
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")


def clean_str(s: pd.Series) -> pd.Series:
    """
    Estandariza strings en mayúsculas y sin espacios sobrantes.
    """
    return s.astype("string").str.strip().str.upper()


def base_ticker(s: pd.Series) -> pd.Series:
    """
    Construye ticker base eliminando sufijos posteriores a '.'.
    """
    s = clean_str(s)
    return s.str.replace(r"\..*$", "", regex=True)


def load_first_existing(candidates):
    """
    Lee el primer archivo existente entre múltiples rutas candidatas.
    """
    tried = []

    for p in candidates:
        tried.append(str(p))
        if p.exists():
            print(f"📂 Leyendo: {p}")
            if p.suffix == ".parquet":
                return pd.read_parquet(p).copy(), p
            elif p.suffix == ".csv":
                return pd.read_csv(p).copy(), p

    raise FileNotFoundError("No se encontró ninguno de estos archivos:\n" + "\n".join(tried))


def na_report(df: pd.DataFrame, cols: list[str], label: str):
    """
    Reporta porcentaje de missing en columnas seleccionadas.
    """
    cols = [c for c in cols if c in df.columns]

    if not cols:
        print(f"\n[{label}] sin columnas")
        return

    rep = (df[cols].isna().mean() * 100).round(2).sort_values(ascending=False)
    print(f"\n[{label}] NA%:")
    print(rep)


# ==========================================================
# 14A.2 BASE DEL PANEL = BACKBONE issuer-date
# ==========================================================

BASE = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"

if not BASE.exists():
    raise FileNotFoundError(f"No encontré {BASE}")

panel = pd.read_parquet(BASE).copy()
panel["date"] = to_eom(panel["date"])

if "issuer" not in panel.columns:
    raise KeyError("La base debe tener columna 'issuer'")

panel["issuer"] = clean_str(panel["issuer"])

dup_base = panel.duplicated(["issuer", "date"]).sum()
if dup_base > 0:
    raise ValueError(f"La base tiene {dup_base} duplicados issuer-date")

n0 = len(panel)

print("\n=== BLOQUE 1 | BASE PANEL ===")
print("Shape:", panel.shape)
print("Issuers:", panel["issuer"].nunique())
print("Rango:", panel["date"].min(), "->", panel["date"].max())
print("Duplicados issuer-date:", dup_base)

base_cols = [
    c for c in [
        "issuer",
        "date",
        "n_bonds",
        "spread_mean_bps",
        "ytm_mean",
        "residual_maturity_mean",
    ]
    if c in panel.columns
]
print("\nColumnas base presentes:", base_cols)


# ==========================================================
# 14A.3 MAPPING issuer -> ticker -> sector
# ==========================================================

MAP_PARQ = CLEAN / "issuer_ticker_sector.parquet"
MAP_CSV  = CLEAN / "issuer_ticker_sector.csv"

issuer_map, issuer_map_path = load_first_existing([MAP_PARQ, MAP_CSV])
issuer_map.columns = [str(c).strip() for c in issuer_map.columns]

need_map = {"issuer", "ticker"}
miss_map = need_map - set(issuer_map.columns)

if miss_map:
    raise KeyError(f"Faltan columnas en issuer_ticker_sector: {miss_map}")

issuer_map["issuer"] = clean_str(issuer_map["issuer"])
issuer_map["ticker"] = clean_str(issuer_map["ticker"])

if "sector" in issuer_map.columns:
    issuer_map["sector"] = issuer_map["sector"].astype("string").str.strip()

print("\n=== BLOQUE 1 | AUDITORÍA issuer_map ===")
print("Shape original issuer_map:", issuer_map.shape)
print("Issuers únicos:", issuer_map["issuer"].nunique())
print("Duplicados por issuer:", int(issuer_map.duplicated(["issuer"]).sum()))
print("Ticker NA%:", round(issuer_map["ticker"].isna().mean() * 100, 2))
if "sector" in issuer_map.columns:
    print("Sector NA%:", round(issuer_map["sector"].isna().mean() * 100, 2))

# Si hay más de una fila por issuer, se conserva la última
issuer_map = issuer_map.drop_duplicates(["issuer"], keep="last").copy()

panel = panel.merge(
    issuer_map,
    on="issuer",
    how="left",
    validate="m:1"
)

panel["ticker_base"] = base_ticker(panel["ticker"])

print("\n=== BLOQUE 1 | POST issuer_map ===")
print("Shape:", panel.shape)
print("Δ filas vs base:", len(panel) - n0)
print("Duplicados issuer-date:", int(panel.duplicated(["issuer", "date"]).sum()))
print("Issuer match rate:", round((panel["ticker"].notna().mean()) * 100, 2), "%")
print("Ticker NA%:", round(panel["ticker"].isna().mean() * 100, 2))
print("Ticker_base NA%:", round(panel["ticker_base"].isna().mean() * 100, 2))
if "sector" in panel.columns:
    print("Sector NA%:", round(panel["sector"].isna().mean() * 100, 2))

unmatched_issuer = (
    panel.loc[panel["ticker"].isna(), ["issuer"]]
    .drop_duplicates()
    .sort_values("issuer")
)

print("Issuers sin ticker:", len(unmatched_issuer))
if len(unmatched_issuer) > 0:
    print(unmatched_issuer.head(20))


# ==========================================================
# 14A.4 EQUITY FEATURES (MERGE POR ticker_base-date)
# ==========================================================

EQ_PARQ = INTERIM / "equity_features_monthly.parquet"
EQ_CSV  = INTERIM / "equity_features_monthly.csv"

equity, eq_path = load_first_existing([EQ_PARQ, EQ_CSV])
equity.columns = [str(c).strip() for c in equity.columns]

if "date" not in equity.columns or "ticker" not in equity.columns:
    raise KeyError("equity_features_monthly debe tener 'date' y 'ticker'")

equity["date"] = to_eom(equity["date"])
equity["ticker"] = clean_str(equity["ticker"])
equity["ticker_base"] = base_ticker(equity["ticker"])

print("\n=== BLOQUE 1 | AUDITORÍA equity raw ===")
print("Shape original equity:", equity.shape)
print("Ticker-date duplicados:", int(equity.duplicated(["ticker_base", "date"]).sum()))
print("Tickers únicos:", equity["ticker_base"].nunique())
print("Rango equity:", equity["date"].min(), "->", equity["date"].max())

eq_keep = ["ticker_base", "date", "beta_252", "ivol_252"]
eq_keep = [c for c in eq_keep if c in equity.columns]

missing_eq = {"ticker_base", "date"} - set(eq_keep)
if missing_eq:
    raise KeyError(f"Faltan columnas mínimas en equity: {missing_eq}")

if not any(c in equity.columns for c in ["beta_252", "ivol_252"]):
    raise KeyError("equity_features_monthly no tiene beta_252 ni ivol_252")

equity = (
    equity[eq_keep]
    .drop_duplicates(["ticker_base", "date"], keep="last")
    .copy()
)

print("\n=== BLOQUE 1 | AUDITORÍA equity clean ===")
print("Shape clean equity:", equity.shape)
na_report(equity, ["beta_252", "ivol_252"], "Equity clean")

# Diagnóstico de matching antes del merge
panel_keys = panel[["ticker_base", "date"]].drop_duplicates().copy()
eq_keys = equity[["ticker_base", "date"]].drop_duplicates().copy()

match_probe = panel_keys.merge(
    eq_keys.assign(in_equity=1),
    on=["ticker_base", "date"],
    how="left"
)

print("\n=== BLOQUE 1 | MATCH DIAGNOSTICS equity ===")
print("Panel keys:", len(panel_keys))
print("Equity keys:", len(eq_keys))
print("Match rate panel->equity:", round(match_probe["in_equity"].notna().mean() * 100, 2), "%")

panel = panel.merge(
    equity,
    on=["ticker_base", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 1 | POST equity ===")
print("Shape:", panel.shape)
print("Δ filas vs base:", len(panel) - n0)
print("Duplicados issuer-date:", int(panel.duplicated(["issuer", "date"]).sum()))

na_report(panel, ["ticker", "ticker_base", "sector", "beta_252", "ivol_252"], "Bloque 1 final")


# ==========================================================
# 14A.5 QA DE CIERRE DEL BLOQUE 1
# ==========================================================

assert len(panel) == n0, "El bloque 1 cambió la cantidad de filas del backbone"
assert panel.duplicated(["issuer", "date"]).sum() == 0, "Hay duplicados issuer-date después del bloque 1"

print("\n✅ BLOQUE 1 OK")
print("Shape final bloque 1:", panel.shape)
print("Issuers:", panel["issuer"].nunique())
print("Rango:", panel["date"].min(), "->", panel["date"].max())


# ==========================================================
# 14A.6 GUARDADO INTERMEDIO OPCIONAL
# ==========================================================

PANEL_B1 = INTERIM / "panel_master_block1_base_mapping_equity.parquet"
panel.to_parquet(PANEL_B1, index=False)

print("💾 Guardado bloque 1 en:", PANEL_B1)

## 14B. Variables contables y poder de mercado

En este bloque se incorporan al panel las variables firm-level asociadas a estructura financiera y posición competitiva, utilizando como llave operativa la combinación `ticker_base-date`.

El objetivo de esta etapa es enriquecer el backbone del panel con controles contables y proxies de market power, preservando en todo momento la estructura original definida a nivel `issuer-date`.

---

### 1. Fundamentals

A partir del archivo mensual de fundamentals se incorporan al panel variables derivadas de información contable cruda a nivel firma. Previamente, se homogeneizan identificadores y fechas, normalizando `ticker`, construyendo `ticker_base` y llevando la fecha a fin de mes.

Sobre esa base se incorporan, entre otras, las siguientes variables:

- `log_assets`: logaritmo de activos totales;
- `leverage`: deuda total sobre activos totales;
- `cash_to_assets`: efectivo e inversiones de corto plazo sobre activos totales;
- `current_ratio`: activos corrientes sobre pasivos corrientes;
- `interest_coverage`: EBITDA sobre gasto por intereses;
- `ltdebt_share`: deuda de largo plazo sobre deuda total.

Formalmente, si $X_{i,t}$ representa una variable contable mensualizada para la firma $i$ en el período $t$, el merge se realiza como:

$$
\text{panel}_{2A}
=
\text{panel}_{1}
\;\underset{(\text{ticker\_base},\,\text{date})}{\text{LEFT JOIN}}\;
\text{fundamentals}_{i,t}
$$

Estas variables capturan dimensiones centrales de la estructura financiera de la firma y funcionan como controles fundamentales en las especificaciones empíricas del modelo.

---

### 2. Poder de mercado

En esta misma etapa se incorporan las medidas de market share previamente construidas, también mediante un merge a nivel `ticker_base-date`.

Las variables preservadas incluyen distintas variantes de participación en ventas, entre ellas:

- `market_share_raw`
- `market_share_w`
- `market_share_sector`
- `market_share_industry_group`
- `market_share_industry`
- `market_share_subindustry`

El merge correspondiente puede resumirse como:

$$
\text{panel}_{2B}
=
\text{panel}_{2A}
\;\underset{(\text{ticker\_base},\,\text{date})}{\text{LEFT JOIN}}\;
\text{market\_power}_{i,t}
$$

Durante la auditoría de este bloque se detectó que el archivo de market power presenta cobertura correcta en términos de identificadores, pero limitada en frecuencia temporal, ya que contiene esencialmente una observación anual por ticker en lugar de una expansión mensual completa. En consecuencia, el merge es técnicamente correcto, pero la cobertura final de estas variables queda restringida por la construcción upstream del insumo.

---

### 3. Auditoría del bloque

En este sub-bloque se validan explícitamente los siguientes aspectos:

- consistencia de la llave `ticker_base-date`;
- duplicados en las bases de fundamentals y market power;
- tasa de matching contra el panel backbone;
- porcentaje de valores faltantes en los ratios contables;
- porcentaje de valores faltantes en las variables de market share;
- preservación de la estructura original del panel.

El objetivo de esta auditoría es distinguir entre dos tipos de problemas distintos:

1. **problemas de integración**, asociados a errores de merge o inconsistencias de identificadores;  
2. **problemas de cobertura**, asociados a limitaciones de frecuencia o disponibilidad en la fuente original de datos.

---

### Alcance interpretativo

Este bloque no altera la definición del panel backbone ni modifica la variable dependiente. Su función es incorporar controles firm-level relevantes para explicar heterogeneidad en spreads, distinguiendo entre fundamentos contables de la firma y medidas alternativas de posición competitiva.

La interpretación de los valores faltantes debe hacerse con cuidado. En el caso de fundamentals, los faltantes pueden reflejar ausencia de información contable o problemas de matching. En el caso de market power, una parte importante de la falta de cobertura responde a la frecuencia temporal incompleta del archivo fuente, más que a errores de integración dentro del merge master.

In [ ]:
# ==========================================================
# 14B. MERGE MASTER — BLOQUE 2
# Fundamentals + Market Power
# ==========================================================


# ==========================================================
# 14B.1 LOAD PANEL DESDE BLOQUE 1
# ==========================================================

PANEL_B1 = INTERIM / "panel_master_block1_base_mapping_equity.parquet"

if not PANEL_B1.exists():
    raise FileNotFoundError("No encontré panel del bloque 1")

panel = pd.read_parquet(PANEL_B1).copy()
panel["date"] = to_eom(panel["date"])

n0 = len(panel)

print("\n=== BLOQUE 2 | BASE ===")
print("Shape:", panel.shape)
print("Duplicados issuer-date:", panel.duplicated(["issuer", "date"]).sum())


# ==========================================================
# 14B.2 FUNDAMENTALS (MERGE POR ticker_base-date)
# ==========================================================

FUND_PARQ = INTERIM / "fundamentals_monthly_enriched.parquet"
FUND_CSV  = INTERIM / "fundamentals_monthly_enriched.csv"

fund, fund_path = load_first_existing([FUND_PARQ, FUND_CSV])
fund.columns = [str(c).strip() for c in fund.columns]

if "date" not in fund.columns or "ticker" not in fund.columns:
    raise KeyError("fundamentals debe tener 'date' y 'ticker'")

fund["date"] = to_eom(fund["date"])
fund["ticker"] = clean_str(fund["ticker"])
fund["ticker_base"] = base_ticker(fund["ticker"])

print("\n=== BLOQUE 2 | AUDITORÍA fundamentals raw ===")
print("Shape:", fund.shape)
print("Ticker únicos:", fund["ticker_base"].nunique())
print("Rango:", fund["date"].min(), "->", fund["date"].max())
print("Duplicados ticker-date:", fund.duplicated(["ticker_base", "date"]).sum())


# ==========================================================
# 14B.2.1 RENOMBRAR COLUMNAS CONTABLES
# ==========================================================

rename_map = {
    "total assets": "total_assets",
    "total current assets": "current_assets",
    "total current liabilities": "current_liabilities",
    "total debt - actual": "total_debt",
    "cash & short term investments": "cash_st",
    "debt - long-term - total": "long_term_debt",
    "earnings before interest taxes depreciation & amortization": "ebitda",
    "interest expense - broker estimate": "interest_expense",
}

fund = fund.rename(columns={k: v for k, v in rename_map.items() if k in fund.columns})


# ==========================================================
# 14B.2.2 CONSTRUCCIÓN DE VARIABLES ECONÓMICAS
# ==========================================================

# Conversión numérica centralizada
ta = pd.to_numeric(fund["total_assets"], errors="coerce")
td = pd.to_numeric(fund["total_debt"], errors="coerce")
cash = pd.to_numeric(fund["cash_st"], errors="coerce")
ca = pd.to_numeric(fund["current_assets"], errors="coerce")
cl = pd.to_numeric(fund["current_liabilities"], errors="coerce")
ebitda = pd.to_numeric(fund["ebitda"], errors="coerce")
ie = pd.to_numeric(fund["interest_expense"], errors="coerce")

# log_assets
fund["log_assets"] = np.log(ta.where(ta > 0))

# Ratios principales
fund["leverage"] = td / ta
fund["cash_to_assets"] = cash / ta
fund["current_ratio"] = ca / cl
fund["interest_coverage"] = ebitda / ie

# Participación de deuda de largo plazo
if "long_term_debt" in fund.columns:
    ltd = pd.to_numeric(fund["long_term_debt"], errors="coerce")
    fund["ltdebt_share"] = ltd / td

# Limpieza de infinitos
fund_vars = [
    "log_assets",
    "leverage",
    "cash_to_assets",
    "current_ratio",
    "interest_coverage",
    "ltdebt_share",
    "rollover_12m",
]
fund_vars = [c for c in fund_vars if c in fund.columns]

for c in fund_vars:
    fund[c] = pd.to_numeric(fund[c], errors="coerce")
    fund[c] = fund[c].replace([np.inf, -np.inf], np.nan)

na_report(fund, fund_vars, "Fundamentals construidos")


# ==========================================================
# 14B.2.3 DEDUPLICACIÓN Y MATCH DIAGNOSTICS
# ==========================================================

fund = fund.drop_duplicates(["ticker_base", "date"], keep="last").copy()

panel_keys = panel[["ticker_base", "date"]].drop_duplicates()
fund_keys = fund[["ticker_base", "date"]].drop_duplicates()

match_probe = panel_keys.merge(
    fund_keys.assign(in_fund=1),
    on=["ticker_base", "date"],
    how="left"
)

print("\n=== BLOQUE 2 | MATCH fundamentals ===")
print("Panel keys:", len(panel_keys))
print("Fund keys:", len(fund_keys))
print("Match rate:", round(match_probe["in_fund"].notna().mean() * 100, 2), "%")


# ==========================================================
# 14B.2.4 MERGE DE FUNDAMENTALS
# ==========================================================

fund_keep = ["ticker_base", "date"] + fund_vars

panel = panel.merge(
    fund[fund_keep],
    on=["ticker_base", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 2 | POST fundamentals ===")
print("Shape:", panel.shape)
print("Δ filas:", len(panel) - n0)
print("Duplicados:", panel.duplicated(["issuer", "date"]).sum())

na_report(panel, fund_vars, "Fundamentals en panel")


# ==========================================================
# 14B.3 MARKET POWER (MERGE POR ticker_base-date)
# ==========================================================

MP_PARQ = INTERIM / "market_power_monthly.parquet"
MP_CSV  = INTERIM / "market_power_monthly.csv"

mp, mp_path = load_first_existing([MP_PARQ, MP_CSV])
mp.columns = [str(c).strip() for c in mp.columns]

if "date" not in mp.columns or "ticker" not in mp.columns:
    raise KeyError("market_power debe tener 'date' y 'ticker'")

mp["date"] = to_eom(mp["date"])
mp["ticker"] = clean_str(mp["ticker"])
mp["ticker_base"] = base_ticker(mp["ticker"])

print("\n=== BLOQUE 2 | AUDITORÍA market power raw ===")
print("Shape:", mp.shape)
print("Ticker únicos:", mp["ticker_base"].nunique())
print("Rango:", mp["date"].min(), "->", mp["date"].max())
print("Duplicados ticker-date:", mp.duplicated(["ticker_base", "date"]).sum())

mp = mp.drop_duplicates(["ticker_base", "date"], keep="last").copy()

mp_vars = [
    "market_share_raw",
    "market_share_w",
    "market_share_sector",
    "market_share_industry_group",
    "market_share_industry",
    "market_share_subindustry",
]

na_report(mp, mp_vars, "Market power raw")


# ==========================================================
# 14B.3.1 MATCH DIAGNOSTICS MARKET POWER
# ==========================================================

mp_keys = mp[["ticker_base", "date"]].drop_duplicates()

match_probe_mp = panel_keys.merge(
    mp_keys.assign(in_mp=1),
    on=["ticker_base", "date"],
    how="left"
)

print("\n=== BLOQUE 2 | MATCH market power ===")
print("Match rate:", round(match_probe_mp["in_mp"].notna().mean() * 100, 2), "%")


# ==========================================================
# 14B.3.2 MERGE DE MARKET POWER
# ==========================================================

mp_keep = ["ticker_base", "date"] + [c for c in mp_vars if c in mp.columns]
mp = mp[mp_keep].copy()

panel = panel.merge(
    mp,
    on=["ticker_base", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 2 | POST market power ===")
print("Shape:", panel.shape)
print("Δ filas:", len(panel) - n0)

na_report(panel, mp_vars, "Market power en panel")


# ==========================================================
# 14B.4 QA DE CIERRE DEL BLOQUE 2
# ==========================================================

assert len(panel) == n0, "Bloque 2 cambió cantidad de filas"
assert panel.duplicated(["issuer", "date"]).sum() == 0, "Duplicados en bloque 2"

print("\n✅ BLOQUE 2 OK")
print("Shape final:", panel.shape)


# ==========================================================
# 14B.5 GUARDADO INTERMEDIO
# ==========================================================

PANEL_B2 = INTERIM / "panel_master_block2_fund_mp.parquet"
panel.to_parquet(PANEL_B2, index=False)

print("💾 Guardado en:", PANEL_B2)

## 14C. Riesgo crediticio agregado, factores macro y riesgo sectorial

En este bloque se incorporan al panel las variables agregadas y semagregadas que capturan exposición a shocks comunes de mercado y crédito, manteniendo la estructura del backbone original a nivel `issuer-date`.

La lógica de esta etapa es complementar los controles firm-level incorporados en bloques anteriores con variables que reflejan condiciones agregadas del mercado financiero, del entorno macroeconómico y del riesgo sectorial.

---

### 1. Exposición al factor de crédito agregado

En primer lugar, se agregan al panel las métricas de exposición diferencial al riesgo crediticio agregado estimadas a nivel `issuer-date`, provenientes del bloque CRC alternativo basado en índices iBoxx. Las variables incorporadas incluyen:

- `crc_level_beta`
- `crc_slope_beta`
- `crc_r2`
- `crc_nobs`

Estas variables buscan capturar heterogeneidad firm-level en la sensibilidad de los spreads a shocks agregados del mercado de crédito, en línea con la reformulación estructural del trabajo.

Dado que estas métricas ya están definidas a nivel `issuer-date`, el merge se realiza directamente sobre la llave backbone del panel:

$$
\text{panel}_{3A}
=
\text{panel}_{2}
\;\underset{(\text{issuer},\,\text{date})}{\text{LEFT JOIN}}\;
\text{crc}_{i,t}
$$

---

### 2. Factores agregados observables

En una segunda etapa se incorporan factores macro-financieros observables que son comunes a todas las firmas en cada fecha. Entre ellos se incluyen:

- `mkt_ret`: retorno agregado mensual del mercado;
- `mkt_vol_60d`: volatilidad agregada del mercado;
- `credit_level`: shock agregado de nivel en crédito;
- `credit_slope`: shock agregado de pendiente en crédito.

Además, cuando el archivo ampliado de factores está disponible, se preservan variables adicionales derivadas de la estructura iBoxx, tales como:

- `credit_tightening_shock`
- `credit_curve_shock`
- `credit_market_level_log`
- `credit_curve_level`
- `credit_curve_change`
- y sus respectivos rezagos.

Estas variables permiten modelar explícitamente shocks comunes del entorno financiero, evitando interpretar coeficientes firm-level como si resumieran mecánicamente movimientos agregados.

Como estas series varían únicamente en el tiempo, el merge se realiza por fecha:

$$
\text{panel}_{3B}
=
\text{panel}_{3A}
\;\underset{\text{date}}{\text{LEFT JOIN}}\;
\text{factors}_{t}
$$

---

### 3. Controles macroeconómicos

A continuación, se incorporan controles macro mensuales, también comunes a todas las firmas en cada período, junto con una dummy para el período COVID.

La dummy `covid_dummy` toma valor 1 entre marzo de 2020 y diciembre de 2021, y 0 en el resto de la muestra. Su inclusión busca capturar un episodio extraordinario de disrupción financiera y macroeconómica que afectó simultáneamente al mercado de crédito corporativo.

En términos operativos, estos controles se agregan por fecha:

$$
\text{panel}_{3C}
=
\text{panel}_{3B}
\;\underset{\text{date}}{\text{LEFT JOIN}}\;
\text{macro}_{t}
$$

---

### 4. Riesgo sectorial

Finalmente, se incorpora `ivol_sector`, una proxy de volatilidad sectorial agregada construida a partir del ETF sectorial asociado a cada firma.

Para ello, primero se vincula cada `ticker_base` con un `sector_ric` mediante el archivo auxiliar `ticker_to_etf`. Luego, se mergea la base mensual de volatilidad sectorial a nivel `sector_ric-date`.

Formalmente, esta etapa puede resumirse en dos pasos:

$$
\text{panel}_{3D}
=
\text{panel}_{3C}
\;\underset{\text{ticker\_base}}{\text{LEFT JOIN}}\;
\text{ticker\_to\_etf}
$$

y luego:

$$
\text{panel}_{3E}
=
\text{panel}_{3D}
\;\underset{(\text{sector\_ric},\,\text{date})}{\text{LEFT JOIN}}\;
\text{ivol\_sector}_{s,t}
$$

La variable `ivol_sector` aproxima la volatilidad agregada del sector al que pertenece cada firma, permitiendo controlar por shocks sectoriales comunes que no quedan reflejados plenamente en factores de mercado agregados.

---

### 5. Auditoría del bloque

Como en los bloques anteriores, en esta etapa se verifica explícitamente:

- consistencia de las llaves de merge;
- cobertura de cada familia de variables;
- duplicados en las bases intermedias;
- ausencia de duplicados `issuer-date` en el panel resultante;
- y preservación exacta de la estructura del backbone original.

En particular, se audita la cobertura de:

- variables CRC;
- factores agregados observables;
- controles macroeconómicos;
- `sector_ric`;
- e `ivol_sector`.

---

### Alcance interpretativo

Este bloque incorpora variables que no reflejan características idiosincráticas puras de la firma, sino exposición o sensibilidad a condiciones comunes del entorno financiero, crediticio, macroeconómico y sectorial.

Por lo tanto, su función principal dentro de la estrategia empírica es separar movimientos agregados de mercado de la heterogeneidad firm-level en spreads. La presencia de valores faltantes en estas variables debe interpretarse según el caso: en algunos casos puede responder a problemas de cobertura temporal de la fuente; en otros, a limitaciones de matching entre firma, ETF sectorial y fecha.

In [ ]:
# ==========================================================
# 14C. MERGE MASTER — BLOQUE 3
# CRC + factores agregados + macro controls + ivol sectorial
# ==========================================================


# ==========================================================
# 14C.1 LOAD PANEL DESDE BLOQUE 2
# ==========================================================

PANEL_B2 = INTERIM / "panel_master_block2_fund_mp.parquet"

if not PANEL_B2.exists():
    raise FileNotFoundError("No encontré panel del bloque 2")

panel = pd.read_parquet(PANEL_B2).copy()
panel["date"] = to_eom(panel["date"])

n0 = len(panel)

print("\n=== BLOQUE 3 | BASE ===")
print("Shape:", panel.shape)
print("Duplicados issuer-date:", panel.duplicated(["issuer", "date"]).sum())


# ==========================================================
# 14C.2 CRC ALTERNATIVA (MERGE POR issuer-date)
# ==========================================================

CRC_PARQ = INTERIM / "crc_alt_iboxx_betas_monthly.parquet"
CRC_CSV  = INTERIM / "crc_alt_iboxx_betas_monthly.csv"

crc, crc_path = load_first_existing([CRC_PARQ, CRC_CSV])
crc.columns = [str(c).strip() for c in crc.columns]

need_crc = {"issuer", "date", "crc_level_beta", "crc_slope_beta"}
miss_crc = need_crc - set(crc.columns)

if miss_crc:
    raise KeyError(f"Faltan columnas en CRC: {miss_crc}")

crc["issuer"] = clean_str(crc["issuer"])
crc["date"] = to_eom(crc["date"])

print("\n=== BLOQUE 3 | AUDITORÍA CRC raw ===")
print("Shape:", crc.shape)
print("Issuers únicos:", crc["issuer"].nunique())
print("Rango:", crc["date"].min(), "->", crc["date"].max())
print("Duplicados issuer-date:", crc.duplicated(["issuer", "date"]).sum())

crc_keep = ["issuer", "date", "crc_level_beta", "crc_slope_beta", "crc_r2", "crc_nobs"]
crc_keep = [c for c in crc_keep if c in crc.columns]

crc = crc[crc_keep].drop_duplicates(["issuer", "date"], keep="last").copy()

crc_keys = crc[["issuer", "date"]].drop_duplicates()
panel_keys_crc = panel[["issuer", "date"]].drop_duplicates()

match_probe_crc = panel_keys_crc.merge(
    crc_keys.assign(in_crc=1),
    on=["issuer", "date"],
    how="left"
)

print("\n=== BLOQUE 3 | MATCH CRC ===")
print("Panel keys:", len(panel_keys_crc))
print("CRC keys:", len(crc_keys))
print("Match rate:", round(match_probe_crc["in_crc"].notna().mean() * 100, 2), "%")

panel = panel.merge(
    crc,
    on=["issuer", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 3 | POST CRC ===")
print("Shape:", panel.shape)
print("Δ filas:", len(panel) - n0)
print("Duplicados:", panel.duplicated(["issuer", "date"]).sum())

na_report(panel, ["crc_level_beta", "crc_slope_beta", "crc_r2", "crc_nobs"], "CRC en panel")


# ==========================================================
# 14C.3 FACTORES AGREGADOS OBSERVABLES
#   - mkt_ret
#   - mkt_vol_60d
#   - factores de crédito ampliados
# ==========================================================

# -------------------------------------------------
# 14C.3A mkt_ret mensual
# -------------------------------------------------

mkt_daily, mkt_daily_path = load_first_existing([
    RAW / "market_prices_daily.parquet",
    RAW / "market_prices_daily.csv",
    CLEAN / "market_prices_daily.parquet",
    CLEAN / "market_prices_daily.csv",
    INTERIM / "market_prices_daily.parquet",
    INTERIM / "market_prices_daily.csv",
    RAW / "market_sp500.parquet",
    RAW / "market_sp500.csv",
    CLEAN / "market_sp500.parquet",
    CLEAN / "market_sp500.csv",
    INTERIM / "market_sp500.parquet",
    INTERIM / "market_sp500.csv",
])

mkt_daily.columns = [str(c).strip() for c in mkt_daily.columns]
mkt_daily["date"] = pd.to_datetime(mkt_daily["date"], errors="coerce")

ret_candidates = ["mkt_ret", "mkt_ret_log", "ret", "return", "sp500_ret", "daily_ret", "ret_log"]
price_candidates = ["price", "close", "mkt_close", "px_last", "index", "spx", "value"]

ret_col = next((c for c in ret_candidates if c in mkt_daily.columns), None)
px_col = next((c for c in price_candidates if c in mkt_daily.columns), None)

if ret_col is not None:
    mkt_daily["mkt_ret_d"] = pd.to_numeric(mkt_daily[ret_col], errors="coerce")
elif px_col is not None:
    mkt_daily["px"] = pd.to_numeric(mkt_daily[px_col], errors="coerce")
    mkt_daily = mkt_daily.sort_values("date")
    mkt_daily["mkt_ret_d"] = np.log(mkt_daily["px"]).diff()
else:
    raise ValueError(f"No encontré retorno ni precio válido en mercado: {mkt_daily.columns.tolist()}")

mkt_daily = mkt_daily.dropna(subset=["mkt_ret_d"]).copy()
mkt_daily["date_eom"] = to_eom(mkt_daily["date"])

mkt_ret_m = (
    mkt_daily.groupby("date_eom", as_index=False)["mkt_ret_d"]
    .sum()
    .rename(columns={"date_eom": "date", "mkt_ret_d": "mkt_ret"})
)

panel = panel.merge(
    mkt_ret_m,
    on="date",
    how="left",
    validate="m:1"
)

# -------------------------------------------------
# 14C.3B mkt_vol_60d
# -------------------------------------------------

MV_PARQ = INTERIM / "mkt_vol_monthly_60d.parquet"
MV_CSV  = INTERIM / "mkt_vol_monthly_60d.csv"

mkt_vol, mv_path = load_first_existing([MV_PARQ, MV_CSV])
mkt_vol.columns = [str(c).strip() for c in mkt_vol.columns]
mkt_vol["date"] = to_eom(mkt_vol["date"])

mv_col = next((c for c in mkt_vol.columns if "mkt_vol" in c.lower()), None)
if mv_col is None:
    raise KeyError(f"No encontré columna mkt_vol en {mv_path}")

mkt_vol = mkt_vol.rename(columns={mv_col: "mkt_vol_60d"})[["date", "mkt_vol_60d"]].copy()

panel = panel.merge(
    mkt_vol,
    on="date",
    how="left",
    validate="m:1"
)

# -------------------------------------------------
# 14C.3C factores de crédito ampliados
# -------------------------------------------------

CREDIT_EXT_PARQ = INTERIM / "credit_factors_iboxx_extended.parquet"
CREDIT_EXT_CSV  = INTERIM / "credit_factors_iboxx_extended.csv"

credit_ext_exists = CREDIT_EXT_PARQ.exists() or CREDIT_EXT_CSV.exists()

if credit_ext_exists:
    credit, credit_path = load_first_existing([CREDIT_EXT_PARQ, CREDIT_EXT_CSV])
    credit.columns = [str(c).strip() for c in credit.columns]
    credit["date"] = to_eom(credit["date"])

    print("\n=== BLOQUE 3 | CREDIT FACTORS EXTENDED ===")
    print("Fuente:", credit_path)
    print("Shape raw:", credit.shape)
    print("Rango:", credit["date"].min(), "->", credit["date"].max())

    credit_keep = [
        "date",
        # legacy
        "credit_level",
        "credit_slope",
        # nombres alternativos descriptivos
        "credit_tightening_shock",
        "credit_curve_shock",
        # componentes / alternativas nuevas
        "credit_market_return",
        "credit_short_return",
        "credit_long_return",
        "credit_market_level_log",
        "credit_curve_level",
        "credit_curve_change",
        # rezagos
        "credit_tightening_shock_l1",
        "credit_curve_shock_l1",
        "credit_market_level_log_l1",
        "credit_curve_level_l1",
        "credit_curve_change_l1",
    ]
    credit_keep = [c for c in credit_keep if c in credit.columns]

    if "credit_level" not in credit_keep or "credit_slope" not in credit_keep:
        raise KeyError(
            "El archivo credit_factors_iboxx_extended debe incluir al menos "
            "'credit_level' y 'credit_slope' para mantener compatibilidad."
        )

    credit = (
        credit[credit_keep]
        .drop_duplicates(["date"], keep="last")
        .sort_values("date")
        .copy()
    )

else:
    print("\n⚠ No encontré credit_factors_iboxx_extended. Fallback a construcción legacy desde iBoxx.")

    iboxx, iboxx_path = load_first_existing([
        RAW / "iboxx_credit_indices_monthly.parquet",
        RAW / "iboxx_credit_indices_monthly.csv",
        INTERIM / "iboxx_credit_indices_monthly.parquet",
        INTERIM / "iboxx_credit_indices_monthly.csv",
        CLEAN / "iboxx_credit_indices_monthly.parquet",
        CLEAN / "iboxx_credit_indices_monthly.csv",
    ])

    iboxx.columns = [str(c).strip() for c in iboxx.columns]
    iboxx["date"] = to_eom(iboxx["date"])

    if "value" not in iboxx.columns and "close" in iboxx.columns:
        iboxx = iboxx.rename(columns={"close": "value"})

    req_cols = {"date", "series", "value"}
    miss = req_cols - set(iboxx.columns)
    if miss:
        raise ValueError(f"Faltan columnas en iBoxx: {miss}")

    wide = iboxx.pivot(index="date", columns="series", values="value").sort_index()

    need = ["iboxx_ig_liquid", "iboxx_overall_1_5y", "iboxx_overall_10y_plus"]
    missing = [c for c in need if c not in wide.columns]
    if missing:
        raise ValueError(f"Faltan series iBoxx requeridas: {missing}")

    if (wide[need] <= 0).any().any():
        raise ValueError("Hay valores no positivos en iBoxx; no puedo construir logs.")

    log_wide = np.log(wide[need])
    ret = log_wide.diff()

    credit_market_return = ret["iboxx_ig_liquid"]
    credit_tightening_shock = -credit_market_return
    credit_short_return = ret["iboxx_overall_1_5y"]
    credit_long_return = ret["iboxx_overall_10y_plus"]
    credit_curve_shock = (-credit_long_return) - (-credit_short_return)

    credit_market_level_log = log_wide["iboxx_ig_liquid"]
    credit_curve_level = (
        log_wide["iboxx_overall_10y_plus"] - log_wide["iboxx_overall_1_5y"]
    )
    credit_curve_change = credit_curve_level.diff()

    credit = pd.DataFrame({
        "date": log_wide.index,

        # legacy
        "credit_level": credit_tightening_shock.values,
        "credit_slope": credit_curve_shock.values,

        # nombres alternativos descriptivos
        "credit_tightening_shock": credit_tightening_shock.values,
        "credit_curve_shock": credit_curve_shock.values,

        # componentes / alternativas
        "credit_market_return": credit_market_return.values,
        "credit_short_return": credit_short_return.values,
        "credit_long_return": credit_long_return.values,
        "credit_market_level_log": credit_market_level_log.values,
        "credit_curve_level": credit_curve_level.values,
        "credit_curve_change": credit_curve_change.values,

        # rezagos
        "credit_tightening_shock_l1": credit_tightening_shock.shift(1).values,
        "credit_curve_shock_l1": credit_curve_shock.shift(1).values,
        "credit_market_level_log_l1": credit_market_level_log.shift(1).values,
        "credit_curve_level_l1": credit_curve_level.shift(1).values,
        "credit_curve_change_l1": credit_curve_change.shift(1).values,
    }).sort_values("date").drop_duplicates(["date"], keep="last")

panel = panel.merge(
    credit,
    on="date",
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 3 | POST factores agregados ===")
factor_check = [
    "mkt_ret",
    "mkt_vol_60d",
    "credit_level",
    "credit_slope",
    "credit_tightening_shock",
    "credit_curve_shock",
    "credit_market_level_log",
    "credit_curve_level",
    "credit_curve_change",
    "credit_tightening_shock_l1",
    "credit_curve_shock_l1",
    "credit_market_level_log_l1",
    "credit_curve_level_l1",
    "credit_curve_change_l1",
]
factor_check = [c for c in factor_check if c in panel.columns]
na_report(panel, factor_check, "Factores agregados en panel")


# ==========================================================
# 14C.4 MACRO CONTROLS + COVID DUMMY
# ==========================================================

macro_controls, macro_controls_path = load_first_existing([
    CLEAN / "macro_controls_monthly.parquet",
    INTERIM / "macro_controls_monthly.parquet",
    RAW / "macro_controls_monthly.parquet",
])

macro_controls.columns = [str(c).strip() for c in macro_controls.columns]
macro_controls["date"] = to_eom(macro_controls["date"])

dup_macro = macro_controls.duplicated(["date"]).sum()
if dup_macro > 0:
    num_cols = [c for c in macro_controls.columns if c != "date"]
    macro_controls = macro_controls.groupby("date", as_index=False)[num_cols].mean()

panel = panel.merge(
    macro_controls,
    on="date",
    how="left",
    validate="m:1"
)

covid_start = pd.Timestamp("2020-03-31")
covid_end   = pd.Timestamp("2021-12-31")

panel["covid_dummy"] = (
    (panel["date"] >= covid_start) &
    (panel["date"] <= covid_end)
).astype("int8")

print("\n=== BLOQUE 3 | POST macro controls ===")
macro_check = [
    c for c in [
        "policy_rate",
        "term_spread_10y_2y",
        "indpro_yoy",
        "cpi_yoy",
        "vix_eom",
        "move_eom",
        "covid_dummy",
    ]
    if c in panel.columns
]
na_report(panel, macro_check, "Macro controls en panel")


# ==========================================================
# 14C.5 IVOL SECTORIAL
# ==========================================================

ticker_to_etf, tte_path = load_first_existing([
    RAW / "ticker_to_etf.parquet",
    RAW / "ticker_to_etf.csv",
])

ticker_to_etf.columns = [str(c).strip() for c in ticker_to_etf.columns]

ticker_col = next((c for c in ticker_to_etf.columns if c.lower() == "ticker"), None)
sec_col = next((c for c in ticker_to_etf.columns if "sector_ric" in c.lower() or c.lower() == "sector_ric"), None)

if ticker_col is None or sec_col is None:
    raise KeyError(f"ticker_to_etf debe tener ticker y sector_ric. Tiene: {ticker_to_etf.columns.tolist()}")

ticker_to_etf = ticker_to_etf.rename(columns={ticker_col: "ticker", sec_col: "sector_ric"}).copy()
ticker_to_etf["ticker"] = clean_str(ticker_to_etf["ticker"])
ticker_to_etf["ticker_base"] = base_ticker(ticker_to_etf["ticker"])
ticker_to_etf["sector_ric"] = clean_str(ticker_to_etf["sector_ric"])

ticker_to_etf = ticker_to_etf.drop_duplicates(["ticker_base"], keep="last").copy()

panel = panel.merge(
    ticker_to_etf[["ticker_base", "sector_ric"]],
    on="ticker_base",
    how="left",
    validate="m:1"
)

ivs, ivs_path = load_first_existing([
    INTERIM / "ivol_sector_monthly_60d.parquet",
    INTERIM / "ivol_sector_monthly_60d.csv",
])

ivs.columns = [str(c).strip() for c in ivs.columns]
ivs["date"] = to_eom(ivs["date"])

if "ivol_sector" not in ivs.columns:
    cand = next((c for c in ivs.columns if "ivol" in c.lower()), None)
    if cand is None:
        raise KeyError(f"No encontré ivol_sector en {ivs_path}")
    ivs = ivs.rename(columns={cand: "ivol_sector"})

ivs["sector_ric"] = clean_str(ivs["sector_ric"])
ivs = ivs[["sector_ric", "date", "ivol_sector"]].drop_duplicates(["sector_ric", "date"], keep="last")

panel = panel.merge(
    ivs,
    on=["sector_ric", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 3 | POST ivol sectorial ===")
na_report(panel, ["sector_ric", "ivol_sector"], "IVOL sectorial en panel")


# ==========================================================
# 14C.6 QA CIERRE BLOQUE 3
# ==========================================================

assert len(panel) == n0, "Bloque 3 cambió cantidad de filas"
assert panel.duplicated(["issuer", "date"]).sum() == 0, "Duplicados en bloque 3"

print("\n✅ BLOQUE 3 OK")
print("Shape final:", panel.shape)


# ==========================================================
# 14C.7 SAVE
# ==========================================================

PANEL_B3 = INTERIM / "panel_master_block3_crc_macro_sector.parquet"
panel.to_parquet(PANEL_B3, index=False)

print("💾 Guardado en:", PANEL_B3)

## 14D. Spreads auxiliares, CDS y consolidación final del panel master

En este último bloque se incorporan al panel medidas adicionales de spread y riesgo crediticio observadas directamente en instrumentos de deuda, y luego se realiza la auditoría final del panel consolidado.

El objetivo de esta etapa es complementar la variable dependiente principal con proxies auxiliares derivadas de bonos y CDS, y cerrar el pipeline verificando que la estructura del backbone original permanezca intacta.

---

### 1. OAS auxiliar

A partir de la base mensual de bonos se construyen medidas agregadas firm-level de spread adicional a nivel `issuer-date`. En particular, se incorporan:

- `oas_mean`
- `oas_median`
- `n_bonds_oas`

Estas variables se obtienen agregando, dentro de cada combinación `issuer-date`, la medida de spread disponible en la base mensual de bonos, utilizando como fuente `spread_bps` o `oas_bps`, según la columna presente en el archivo.

Formalmente, si $s_{b,i,t}$ representa el spread observado para el bono $b$ perteneciente al emisor $i$ en el período $t$, entonces:

$$
\text{oas\_mean}_{i,t} = \frac{1}{N_{i,t}} \sum_{b=1}^{N_{i,t}} s_{b,i,t}
$$

$$
\text{oas\_median}_{i,t} = \text{mediana}\left(s_{b,i,t}\right)
$$

donde $N_{i,t}$ corresponde al número de bonos con información de spread para el emisor $i$ en la fecha $t$, y se almacena como `n_bonds_oas`.

Estas variables funcionan como alternativa o complemento a la medida principal de spread del panel.

---

### 2. CDS firm-level

En una segunda etapa se incorporan medidas mensuales de CDS agregadas por emisor, construidas a partir de datos diarios. Las variables incorporadas son:

- `cds_5y_mean`
- `cds_5y_eom`
- `n_obs_cds`

La integración requiere mapear correctamente los datos diarios de CDS al identificador `issuer`, utilizando la mejor llave disponible según cobertura del archivo fuente. El procedimiento sigue un criterio jerárquico:

1. uso directo de `issuer`, cuando el match con el panel es suficientemente alto;
2. fallback a `ticker`, utilizando el mapping `issuer_ticker_sector`;
3. fallback a `ric`, utilizando correspondencia con `bond_ric` en la base mensual de bonos.

Una vez resuelto el identificador, la serie diaria se agrega a frecuencia mensual a nivel `issuer-date`, calculando:

$$
\text{cds\_5y\_mean}_{i,t} = \frac{1}{M_{i,t}} \sum_{d=1}^{M_{i,t}} \text{CDS}_{i,d}
$$

$$
\text{cds\_5y\_eom}_{i,t} = \text{última observación diaria disponible del mes}
$$

donde $M_{i,t}$ es la cantidad de observaciones diarias disponibles de CDS para el emisor $i$ en el mes $t$, y se registra como `n_obs_cds`.

Estas variables aportan una medida observada de riesgo crediticio derivada del mercado de CDS, complementaria a los spreads de bonos.

---

### 3. QA final

Una vez incorporadas todas las capas de información, se realiza una auditoría final del panel master, verificando:

- cantidad final de observaciones;
- variación en el número de filas respecto del backbone;
- duplicados en `issuer-date`;
- cobertura de variables clave;
- rango temporal del panel consolidado;
- y preservación exacta de la estructura original del backbone.

El principio central se mantiene: **ningún merge puede modificar la cantidad de observaciones ni generar duplicados en la combinación `issuer-date`**.

---

### 4. Guardado del panel consolidado

Finalmente, el panel consolidado se exporta como archivo maestro para su uso posterior en el runner econométrico y en la construcción de tablas finales del trabajo.

El archivo de salida es:

- `CLEAN/panel_master.parquet`

---

### Alcance interpretativo

Las variables agregadas en este bloque no redefinen la variable dependiente principal del trabajo, sino que aportan medidas complementarias de riesgo crediticio observadas directamente en instrumentos de deuda.

En particular:

- `oas_mean` y `oas_median` permiten contrastar o complementar la medida principal de spread;
- `cds_5y_mean` y `cds_5y_eom` aportan información adicional proveniente del mercado de derivados de crédito.

Su utilidad empírica reside en enriquecer el análisis de robustez y ampliar la caracterización del riesgo crediticio firm-level, sin alterar la estructura del panel backbone consolidado.

In [ ]:
# ==========================================================
# 14D. MERGE MASTER — BLOQUE 4
# OAS auxiliar + CDS + QA final + guardado panel master
# ==========================================================


# ==========================================================
# 14D.1 LOAD PANEL DESDE BLOQUE 3
# ==========================================================

PANEL_B3 = INTERIM / "panel_master_block3_crc_macro_sector.parquet"

if not PANEL_B3.exists():
    raise FileNotFoundError("No encontré panel del bloque 3")

panel = pd.read_parquet(PANEL_B3).copy()
panel["date"] = to_eom(panel["date"])

n0 = len(panel)

print("\n=== BLOQUE 4 | BASE ===")
print("Shape:", panel.shape)
print("Duplicados issuer-date:", panel.duplicated(["issuer", "date"]).sum())


# ==========================================================
# 14D.2 OAS AUXILIAR DESDE bonds_monthly_spreads
# ==========================================================

bm, bm_path = load_first_existing([
    INTERIM / "bonds_monthly_spreads.parquet",
    INTERIM / "bonds_monthly_spreads.csv",
    CLEAN / "bonds_monthly_spreads.parquet",
    CLEAN / "bonds_monthly_spreads.csv",
])

bm.columns = [str(c).strip() for c in bm.columns]
bm["date"] = to_eom(bm["date"])
bm["issuer"] = clean_str(bm["issuer"])

print("\n=== BLOQUE 4 | AUDITORÍA bonds_monthly_spreads ===")
print("Shape:", bm.shape)
print("Issuers únicos:", bm["issuer"].nunique())
print("Rango:", bm["date"].min(), "->", bm["date"].max())

spread_candidates = [c for c in ["spread_bps", "oas_bps"] if c in bm.columns]
if not spread_candidates:
    raise ValueError("bonds_monthly_spreads no tiene spread_bps ni oas_bps")

spread_col = spread_candidates[0]
print("Columna usada para OAS auxiliar:", spread_col)

oas_aux = (
    bm.groupby(["issuer", "date"], as_index=False)
      .agg(
          oas_mean=(spread_col, "mean"),
          oas_median=(spread_col, "median"),
          n_bonds_oas=(spread_col, "count"),
      )
)

oas_keys = oas_aux[["issuer", "date"]].drop_duplicates()
panel_keys = panel[["issuer", "date"]].drop_duplicates()

match_probe_oas = panel_keys.merge(
    oas_keys.assign(in_oas=1),
    on=["issuer", "date"],
    how="left"
)

print("\n=== BLOQUE 4 | MATCH OAS auxiliar ===")
print("Panel keys:", len(panel_keys))
print("OAS keys:", len(oas_keys))
print("Match rate:", round(match_probe_oas["in_oas"].notna().mean() * 100, 2), "%")

panel = panel.merge(
    oas_aux,
    on=["issuer", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 4 | POST OAS auxiliar ===")
print("Shape:", panel.shape)
print("Δ filas:", len(panel) - n0)
print("Duplicados:", panel.duplicated(["issuer", "date"]).sum())

na_report(panel, ["oas_mean", "oas_median", "n_bonds_oas"], "OAS auxiliar en panel")


# ==========================================================
# 14D.3 CDS DIARIO -> issuer-month
# ==========================================================

cds, cds_path = load_first_existing([
    RAW / "cds_spreads_daily.parquet",
    CLEAN / "cds_spreads_daily.parquet",
    INTERIM / "cds_spreads_daily.parquet",
])

cds.columns = [str(c).strip().lower() for c in cds.columns]
cds["date"] = pd.to_datetime(cds["date"], errors="coerce")

print("\n=== BLOQUE 4 | AUDITORÍA CDS raw ===")
print("Shape:", cds.shape)
print("Columnas:", cds.columns.tolist())

preferred_value_cols = ["cds_spread_bps", "cds_5y_bps", "cds_mid_bps", "spread_bps"]
value_col = next((c for c in preferred_value_cols if c in cds.columns), None)

if value_col is None:
    fuzzy = [c for c in cds.columns if (("cds" in c) and ("bps" in c)) or ("spread" in c)]
    if not fuzzy:
        raise ValueError("No se encontró columna numérica de CDS")
    value_col = fuzzy[0]

print("Columna usada para CDS:", value_col)
cds[value_col] = pd.to_numeric(cds[value_col], errors="coerce")


# ==========================================================
# 14D.3.1 MAPPING A issuer
# ==========================================================

panel_issuer_set = set(panel["issuer"].dropna().astype(str).str.strip().unique())

if "issuer" in cds.columns:
    cds["issuer"] = cds["issuer"].astype("string").str.strip().str.upper()
    direct_match = cds["issuer"].isin(panel_issuer_set).mean()
    print("Direct issuer match:", round(direct_match * 100, 2), "%")

    if direct_match < 0.10:
        mapped = False

        if "ticker" in cds.columns:
            cds["ticker"] = clean_str(cds["ticker"])

            issuer_map, _ = load_first_existing([
                CLEAN / "issuer_ticker_sector.parquet",
                CLEAN / "issuer_ticker_sector.csv",
            ])
            issuer_map.columns = [str(c).strip() for c in issuer_map.columns]
            issuer_map["issuer"] = clean_str(issuer_map["issuer"])
            issuer_map["ticker"] = clean_str(issuer_map["ticker"])

            ticker_map = issuer_map[["issuer", "ticker"]].dropna().drop_duplicates()

            cds = cds.drop(columns=["issuer"], errors="ignore").merge(
                ticker_map,
                on="ticker",
                how="left"
            )
            mapped = True

        if (not mapped or cds["issuer"].isna().mean() > 0.90) and "ric" in cds.columns:
            cds["ric"] = clean_str(cds["ric"])

            if "bond_ric" not in bm.columns:
                raise ValueError("No existe bond_ric en bonds_monthly_spreads para mapear CDS por ric")

            ric_map = (
                bm.assign(ric=clean_str(bm["bond_ric"]))[["issuer", "ric"]]
                .dropna()
                .drop_duplicates()
            )

            cds = cds.drop(columns=["issuer"], errors="ignore").merge(
                ric_map,
                on="ric",
                how="left"
            )

elif "ticker" in cds.columns:
    cds["ticker"] = clean_str(cds["ticker"])

    issuer_map, _ = load_first_existing([
        CLEAN / "issuer_ticker_sector.parquet",
        CLEAN / "issuer_ticker_sector.csv",
    ])
    issuer_map.columns = [str(c).strip() for c in issuer_map.columns]
    issuer_map["issuer"] = clean_str(issuer_map["issuer"])
    issuer_map["ticker"] = clean_str(issuer_map["ticker"])

    ticker_map = issuer_map[["issuer", "ticker"]].dropna().drop_duplicates()
    cds = cds.merge(ticker_map, on="ticker", how="left")

elif "ric" in cds.columns:
    cds["ric"] = clean_str(cds["ric"])

    if "bond_ric" not in bm.columns:
        raise ValueError("No existe bond_ric en bonds_monthly_spreads para mapear CDS por ric")

    ric_map = (
        bm.assign(ric=clean_str(bm["bond_ric"]))[["issuer", "ric"]]
        .dropna()
        .drop_duplicates()
    )

    cds = cds.merge(ric_map, on="ric", how="left")

else:
    raise ValueError("No hay forma de mapear CDS a issuer")

if "issuer" not in cds.columns:
    raise ValueError("No se logró construir issuer en CDS")

cds["issuer"] = clean_str(cds["issuer"])
cds = cds.dropna(subset=["issuer", "date"]).copy()
cds["date"] = to_eom(cds["date"])
cds = cds.sort_values(["issuer", "date"]).copy()


# ==========================================================
# 14D.3.2 AGREGACIÓN MENSUAL DE CDS
# ==========================================================

cds_agg = (
    cds.groupby(["issuer", "date"], as_index=False)
       .agg(
           cds_5y_mean=(value_col, "mean"),
           cds_5y_eom=(value_col, "last"),
           n_obs_cds=(value_col, "count"),
       )
)

cds_keys = cds_agg[["issuer", "date"]].drop_duplicates()

match_probe_cds = panel_keys.merge(
    cds_keys.assign(in_cds=1),
    on=["issuer", "date"],
    how="left"
)

print("\n=== BLOQUE 4 | MATCH CDS ===")
print("Panel keys:", len(panel_keys))
print("CDS keys:", len(cds_keys))
print("Match rate:", round(match_probe_cds["in_cds"].notna().mean() * 100, 2), "%")

panel = panel.merge(
    cds_agg,
    on=["issuer", "date"],
    how="left",
    validate="m:1"
)

print("\n=== BLOQUE 4 | POST CDS ===")
print("Shape:", panel.shape)
print("Δ filas:", len(panel) - n0)
print("Duplicados:", panel.duplicated(["issuer", "date"]).sum())

na_report(panel, ["cds_5y_mean", "cds_5y_eom", "n_obs_cds"], "CDS en panel")


# ==========================================================
# 14D.4 QA FINAL PANEL MASTER
# ==========================================================

print("\n=== QA FINAL PANEL MASTER ===")
print("Shape final:", panel.shape)
print("Δ filas vs base:", len(panel) - n0)
print("Duplicados issuer-date:", int(panel.duplicated(["issuer", "date"]).sum()))
print("Issuers:", panel["issuer"].nunique())
print("Rango:", panel["date"].min(), "->", panel["date"].max())

key_vars = [
    c for c in [
        "spread_mean_bps",
        "ytm_mean",
        "residual_maturity_mean",
        "beta_252",
        "ivol_252",
        "ivol_sector",
        "crc_level_beta",
        "crc_slope_beta",
        "mkt_ret",
        "credit_level",
        "credit_slope",
        "log_assets",
        "leverage",
        "cash_to_assets",
        "market_share_raw",
        "oas_mean",
        "cds_5y_mean",
    ]
    if c in panel.columns
]

na_report(panel, key_vars, "QA final variables clave")

assert len(panel) == n0, "El bloque 4 cambió la cantidad de filas del backbone"
assert panel.duplicated(["issuer", "date"]).sum() == 0, "Hay duplicados issuer-date en panel final"


# ==========================================================
# 14D.5 GUARDADO FINAL
# ==========================================================

PANEL_FINAL = CLEAN / "panel_master.parquet"
panel.to_parquet(PANEL_FINAL, index=False)

print("\n✅ PANEL MASTER FINAL guardado en:", PANEL_FINAL)

# 15) QA DEL PANEL — VALIDACIÓN ESTRUCTURAL Y DE COBERTURA

## Objetivo

Esta sección tiene como objetivo realizar una validación integral del panel final construido, 
con el fin de garantizar su consistencia estructural, calidad de datos y cobertura suficiente 
para la estimación de los modelos econométricos.

A diferencia de instancias preliminares de control de calidad, este bloque no se limita a la 
inspección en consola, sino que produce outputs estructurados y exportables. Estos outputs 
constituyen insumos directos para la caracterización formal de la muestra en la sección de 
Metodología del trabajo.

---

## Alcance del QA

El análisis se organiza en cinco bloques complementarios:

### 1. Estructura del panel

Se valida la estructura básica del panel de datos, considerando:

- Número total de observaciones
- Número de firmas únicas
- Número de períodos
- Rango temporal efectivo
- Existencia de duplicados a nivel firma–fecha

Este bloque permite verificar que la unidad de observación (firma–mes) se encuentra correctamente 
definida y detectar inconsistencias estructurales críticas.

---

### 2. Distribución del panel

Se analiza la distribución de observaciones a lo largo de las dimensiones del panel:

- Número de observaciones por firma
- Número de observaciones por período

Este análisis permite evaluar el grado de desbalanceo del panel, así como identificar posibles 
problemas de concentración temporal o heterogeneidad excesiva entre emisores.

---

### 3. Cobertura conjunta de variables clave

Se construye una muestra analítica restringida a observaciones con información completa en un 
conjunto de variables fundamentales:

- Variable dependiente (credit spreads medidos a través de OAS)
- Shock agregado de mercado
- Shock agregado de crédito
- Medida de riesgo idiosincrático (volatilidad idiosincrática)
- Variables firm-level relevantes (por ejemplo, leverage)

El objetivo es cuantificar la pérdida de observaciones al exigir disponibilidad conjunta de 
información, lo cual resulta determinante para la estimación econométrica.

---

### 4. Muestras por modelo (aproximación)

Se definen muestras analíticas específicas para cada modelo econométrico (M0–M6), en función 
de la disponibilidad conjunta de las variables incluidas en cada especificación.

Para cada modelo se reporta:

- Número de observaciones
- Número de firmas
- Número de períodos

Este bloque permite anticipar diferencias en el tamaño muestral entre especificaciones, lo cual 
es fundamental para la correcta interpretación de los resultados.

---

### 5. Missingness global

Se calcula el porcentaje de valores faltantes para cada variable del panel.

Este análisis permite:

- Identificar variables con baja cobertura
- Detectar posibles problemas en la construcción de datos
- Evaluar la viabilidad empírica de cada bloque de variables

---

## Outputs generados

El proceso de QA genera los siguientes archivos en la carpeta `outputs/qa/`:

- `panel_structure_summary.csv`  
  → Resumen estructural del panel

- `panel_distribution.csv`  
  → Distribución de observaciones por firma y período

- `key_sample_summary.csv`  
  → Caracterización de la muestra analítica basada en variables clave

- `model_samples_summary.csv`  
  → Tamaño muestral por modelo (M0–M6)

- `panel_missingness_full.csv`  
  → Porcentaje de valores faltantes por variable

Estos outputs son utilizados en la sección de Metodología para documentar de manera transparente 
la construcción del panel y la calidad de los datos.

---

## Nota metodológica

La muestra utilizada en cada modelo econométrico se define como la intersección de observaciones 
no faltantes en todas las variables requeridas por dicha especificación.

En consecuencia, las diferencias en el tamaño muestral entre modelos no responden a cambios en 
el universo de firmas, sino a restricciones en la disponibilidad de datos.

Este punto resulta clave para asegurar una interpretación correcta de las comparaciones entre 
especificaciones econométricas.

In [ ]:
# ==========================================================
# 20.X QA PANEL — VALIDACIÓN ESTRUCTURAL Y OUTPUTS EXPORTABLES
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------------------------
# Definición de path de outputs QA
# -------------------------------------------------
OUT_QA = OUTPUTS / "qa"
OUT_QA.mkdir(parents=True, exist_ok=True)

# ==========================================================
# 1) RESUMEN ESTRUCTURAL DEL PANEL
# ==========================================================

# Métricas básicas del panel
n_obs = len(panel)                              # número total de observaciones
n_firms = panel["issuer"].nunique()             # número de firmas únicas
n_periods = panel["date"].nunique()             # número de períodos

# Rango temporal
date_min = panel["date"].min()
date_max = panel["date"].max()

# Duplicados a nivel firma–fecha
dup = panel.duplicated(["issuer", "date"]).sum()

# Construcción del resumen estructural
panel_structure = pd.DataFrame({
    "metric": [
        "n_obs",
        "n_firms",
        "n_periods",
        "date_min",
        "date_max",
        "duplicate_issuer_date"
    ],
    "value": [
        n_obs,
        n_firms,
        n_periods,
        date_min,
        date_max,
        dup
    ]
})

# Export
panel_structure.to_csv(OUT_QA / "panel_structure_summary.csv", index=False)

# ==========================================================
# 2) DISTRIBUCIÓN DEL PANEL
# ==========================================================

# ------------------------------
# Observaciones por firma
# ------------------------------
obs_per_firm = panel.groupby("issuer").size()
dist_firm = obs_per_firm.describe()

# ------------------------------
# Observaciones por período
# ------------------------------
obs_per_date = panel.groupby("date").size()
dist_date = obs_per_date.describe()

# Consolidación
panel_distribution = pd.DataFrame({
    "metric": dist_firm.index,
    "obs_per_firm": dist_firm.values,
    "obs_per_date": dist_date.values
})

# Export
panel_distribution.to_csv(OUT_QA / "panel_distribution.csv", index=False)

# ==========================================================
# 3) COBERTURA CONJUNTA — VARIABLES CLAVE
# ==========================================================

# Variables clave para muestra analítica
key_vars = [
    "oas_mean",       # variable dependiente (credit spreads)
    "mkt_ret",        # shock de mercado
    "credit_level",   # nivel de crédito agregado
    "ivol_252",       # volatilidad idiosincrática
    "leverage"        # variable firm-level
]

# Filtrar variables existentes en el panel
key_vars = [c for c in key_vars if c in panel.columns]

# Submuestra con datos completos
panel_key_sample = panel.dropna(subset=key_vars)

# Resumen de la muestra analítica
key_sample_summary = pd.DataFrame({
    "metric": ["n_obs", "n_firms", "n_periods"],
    "value": [
        len(panel_key_sample),
        panel_key_sample["issuer"].nunique(),
        panel_key_sample["date"].nunique()
    ]
})

# Export
key_sample_summary.to_csv(OUT_QA / "key_sample_summary.csv", index=False)

# ==========================================================
# 4) MUESTRAS POR MODELO (APROXIMACIÓN)
# ==========================================================

MODEL_SPECS_QA = {
    "M0": ["oas_mean", "leverage", "log_assets"],
    "M1": ["oas_mean", "policy_rate", "term_spread_10y_2y", "move_eom"],
    "M2": ["oas_mean", "mkt_ret"],
    "M3": ["oas_mean", "mkt_ret", "leverage"],
    "M4": ["oas_mean", "credit_level", "credit_slope"],
    "M5": ["oas_mean", "credit_level", "credit_slope", "rollover_12m"],
    "M6": ["oas_mean", "mkt_ret", "credit_level", "leverage"]
}

rows = []

for model, vars_list in MODEL_SPECS_QA.items():
    
    # Filtrar variables disponibles en el panel
    vars_list = [v for v in vars_list if v in panel.columns]
    
    # Submuestra con datos completos para el modelo
    df_model = panel.dropna(subset=vars_list)

    # Métricas de muestra
    rows.append({
        "model": model,
        "n_obs": len(df_model),
        "n_firms": df_model["issuer"].nunique(),
        "n_periods": df_model["date"].nunique()
    })

# Consolidación
model_samples = pd.DataFrame(rows)

# Export
model_samples.to_csv(OUT_QA / "model_samples_summary.csv", index=False)

# ==========================================================
# 5) MISSINGNESS GLOBAL DEL PANEL
# ==========================================================

# Porcentaje de valores faltantes por variable
na_all = (panel.isna().mean() * 100).round(2)

# Formato tabla
na_table = na_all.reset_index()
na_table.columns = ["variable", "na_pct"]

# Export
na_table.to_csv(OUT_QA / "panel_missingness_full.csv", index=False)

# ==========================================================
# CONFIRMACIÓN DE EXPORT
# ==========================================================
print("✔ QA estructural del panel exportado correctamente en:", OUT_QA)

## 16. FIGURAS DESCRIPTIVAS PARA TESIS

## Objetivo

Esta sección construye un conjunto de figuras descriptivas destinadas a complementar 
los resultados econométricos del trabajo. El objetivo es proveer evidencia visual que 
permita caracterizar de manera intuitiva:

- La dinámica temporal de los credit spreads corporativos
- Su co-movimiento con shocks agregados de mercado y de crédito
- La existencia de heterogeneidad en la exposición a dichos shocks según características firm-level

En línea con la estrategia empírica del trabajo, el foco está puesto en la identificación 
de componentes agregados comunes y en la exposición diferencial de las firmas a estos shocks.

---

## 16.1 Evolución temporal de spreads y shocks agregados

Se construye una figura de series temporales mensuales que muestra la evolución conjunta de:

- El spread promedio del panel
- El shock agregado de mercado
- El shock agregado de crédito

Con el objetivo de hacer comparables las magnitudes, las series son previamente winsorizadas 
para mitigar la influencia de outliers y luego estandarizadas en términos de z-scores:

\[
z_{t} = \frac{x_{t} - \mu_x}{\sigma_x}
\]

Esta figura permite identificar visualmente la presencia de factores agregados comunes en la 
dinámica de los spreads, así como episodios de estrés sistémico (por ejemplo, COVID-19).

---

## 16.2 Relación descriptiva entre spreads y shocks agregados

Se analizan las relaciones descriptivas entre el spread promedio mensual y los shocks agregados.

Dado que los shocks agregados son comunes a todas las firmas en cada período, el análisis se 
realiza a nivel agregado mensual. Para cada relación se construye:

- Una nube de puntos mensual
- Un binscatter por cuantiles del shock agregado
- Una recta de ajuste lineal (OLS) a nivel descriptivo

El objetivo de esta figura no es establecer causalidad, sino documentar la asociación empírica 
entre shocks agregados y spreads, en línea con la hipótesis de que los spreads reflejan 
condiciones macro-financieras comunes.

Se prioriza la relación con el shock de mercado como figura principal, mientras que la relación 
con el shock de crédito se presenta como diagnóstico complementario.

---

## 16.3 Heterogeneidad por poder de mercado

Se analiza la heterogeneidad en la relación entre spreads y shocks agregados en función del 
poder de mercado de las firmas.

Para ello:

1. Se calcula el poder de mercado promedio por firma a lo largo de la muestra
2. Se clasifican las firmas en terciles (bajo, medio y alto poder de mercado)
3. Se construyen binscatters separados por grupo

Este ejercicio permite evaluar si la sensibilidad de los spreads a shocks agregados difiere 
sistemáticamente entre firmas, lo cual constituye evidencia descriptiva de exposición 
heterogénea.

Es importante destacar que esta figura tiene un carácter puramente ilustrativo: la 
heterogeneidad se formaliza posteriormente en las especificaciones econométricas mediante 
términos de interacción entre shocks agregados y características firm-level.

---

## Outputs

Las figuras generadas se almacenan en la carpeta `outputs/figures/`, junto con las bases 
agregadas utilizadas para su construcción.

En particular, se exportan:

- Series mensuales agregadas (raw y procesadas)
- Datos utilizados en binscatters
- Bases intermedias para análisis de heterogeneidad

Esto garantiza la trazabilidad de los resultados y facilita su reutilización en la redacción 
de la tesis y en eventuales revisiones.

---

## Nota metodológica

Las transformaciones aplicadas a las series (winsorización y estandarización) tienen un 
propósito exclusivamente descriptivo y no afectan la estimación de los modelos econométricos.

Asimismo, el análisis agregado implica una pérdida de variación cross-section, por lo que 
los patrones observados deben interpretarse como evidencia complementaria y no como 
resultados causales.

In [ ]:
# ==========================================================
# 16. FIGURAS DESCRIPTIVAS PARA TESIS — VERSIÓN FINAL LIMPIA
# Winsorización + agregación mensual + heterogeneidad estructural
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------------------------------
# 0) PARÁMETROS
# -------------------------------------------------
WINSOR_P = 0.05                  # winsorización (5%–95%)
N_BINS = 8                       # número de bins para binscatter
MIN_ISSUERS_MONTH = 8            # mínimo de emisores por mes (figuras agregadas)
MIN_ISSUERS_TERCILE_MONTH = 3    # mínimo por tercil y mes (heterogeneidad)
EXPORT_DIAGNOSTIC_CREDIT = True  # exportar figura secundaria de crédito

# -------------------------------------------------
# 1) PATHS Y CARGA DE DATOS
# -------------------------------------------------
FIGURES = OUTPUTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

PANEL_FINAL = CLEAN / "panel_master.parquet"

if not PANEL_FINAL.exists():
    raise FileNotFoundError(f"No encontré {PANEL_FINAL}")

panel = pd.read_parquet(PANEL_FINAL).copy()

# Asegurar frecuencia mensual (end-of-month)
panel["date"] = (
    pd.to_datetime(panel["date"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp("M")
)

print("✔ Panel cargado correctamente:", PANEL_FINAL)
print("Shape del panel:", panel.shape)

# -------------------------------------------------
# 2) FUNCIONES AUXILIARES
# -------------------------------------------------

def pick_first_existing(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    """Selecciona la primera columna disponible en el panel."""
    for c in candidates:
        if c in df.columns:
            print(f"✔ {label}: usando '{c}'")
            return c
    raise ValueError(f"No encontré columna válida para {label}: {candidates}")


def to_num(s: pd.Series) -> pd.Series:
    """Convierte a numérico y limpia infinitos."""
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)


def winsorize_series(s: pd.Series, p: float = 0.05) -> pd.Series:
    """Winsorización simple por cuantiles."""
    s = to_num(s)
    if s.dropna().empty:
        return s
    lo = s.quantile(p)
    hi = s.quantile(1 - p)
    return s.clip(lower=lo, upper=hi)


def zscore(s: pd.Series) -> pd.Series:
    """Estandarización (z-score)."""
    s = to_num(s)
    std = s.std()
    if pd.isna(std) or std == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / std


def add_covid_shading(ax, df_time: pd.DataFrame, covid_col: str | None):
    """Sombreado de períodos COVID (si existe dummy)."""
    if covid_col is None or covid_col not in df_time.columns:
        return

    covid_dates = df_time.loc[df_time[covid_col] == 1, "date"].sort_values().tolist()
    if len(covid_dates) == 0:
        return

    start = covid_dates[0]
    prev = covid_dates[0]

    for d in covid_dates[1:]:
        if (d.to_period("M") - prev.to_period("M")).n != 1:
            ax.axvspan(start, prev, alpha=0.12)
            start = d
        prev = d

    ax.axvspan(start, prev, alpha=0.12)


def build_binscatter(df: pd.DataFrame, x: str, y: str, n_bins: int = 8) -> pd.DataFrame:
    """Construcción de binscatter por cuantiles."""
    tmp = df[[x, y]].copy()
    tmp[x] = to_num(tmp[x])
    tmp[y] = to_num(tmp[y])
    tmp = tmp.dropna()

    if tmp.empty or tmp[x].nunique() < 2:
        return pd.DataFrame(columns=[x, y, "n_obs"])

    n_bins = int(min(n_bins, max(3, tmp[x].nunique())))

    try:
        tmp["bin"] = pd.qcut(tmp[x], q=n_bins, duplicates="drop")
    except ValueError:
        tmp["bin"] = pd.cut(tmp[x], bins=min(5, tmp[x].nunique()))

    out = (
        tmp.groupby("bin", observed=False)
           .agg(
               **{
                   x: (x, "mean"),
                   y: (y, "mean"),
                   "n_obs": (y, "size")
               }
           )
           .reset_index(drop=True)
           .sort_values(x)
    )
    return out


def fit_line(x: pd.Series, y: pd.Series):
    """Ajuste lineal simple (OLS) para visualización."""
    tmp = pd.DataFrame({"x": x, "y": y}).copy()
    tmp["x"] = to_num(tmp["x"])
    tmp["y"] = to_num(tmp["y"])
    tmp = tmp.dropna()

    if len(tmp) < 3 or tmp["x"].nunique() < 2:
        return None

    x_vals = tmp["x"].astype(float).to_numpy()
    y_vals = tmp["y"].astype(float).to_numpy()

    b1, b0 = np.polyfit(x_vals, y_vals, deg=1)
    xs = np.linspace(x_vals.min(), x_vals.max(), 100)
    ys = b1 * xs + b0

    return xs, ys, b1, b0

# -------------------------------------------------
# 3) SELECCIÓN DE VARIABLES
# -------------------------------------------------

spread_var = pick_first_existing(panel, ["oas_mean", "spread_mean_bps"], "spread")
market_shock_var = pick_first_existing(panel, ["mkt_ret"], "shock de mercado")
credit_shock_var = pick_first_existing(panel, ["credit_level", "crc_level_beta"], "shock de crédito")

market_power_var = pick_first_existing(
    panel,
    [
        "market_share_industry_group",
        "market_share_industry",
        "market_share_sector",
        "market_share_subindustry",
        "market_share_w",
        "market_share_raw",
    ],
    "poder de mercado"
)

covid_var = "covid_dummy" if "covid_dummy" in panel.columns else None

if "issuer" not in panel.columns:
    raise ValueError("Falta columna 'issuer' en el panel.")

# -------------------------------------------------
# 4) CONSTRUCCIÓN BASE MENSUAL AGREGADA
# -------------------------------------------------

monthly = (
    panel.groupby("date", as_index=False)
         .agg(
             spread_avg=(spread_var, "mean"),
             mkt_ret=(market_shock_var, "first"),
             credit_level=(credit_shock_var, "first"),
             covid_dummy=(covid_var, "max") if covid_var else (spread_var, lambda x: np.nan),
             n_issuers=("issuer", "nunique")
         )
         .sort_values("date")
         .reset_index(drop=True)
)

# Limpieza numérica
for c in ["spread_avg", "mkt_ret", "credit_level", "covid_dummy", "n_issuers"]:
    if c in monthly.columns:
        monthly[c] = to_num(monthly[c])

# Filtro por cobertura mínima
monthly["keep_for_plots"] = (monthly["n_issuers"] >= MIN_ISSUERS_MONTH).astype(int)

# Winsorización
monthly["spread_avg_w"] = winsorize_series(monthly["spread_avg"], WINSOR_P)
monthly["mkt_ret_w"] = winsorize_series(monthly["mkt_ret"], WINSOR_P)
monthly["credit_level_w"] = winsorize_series(monthly["credit_level"], WINSOR_P)

# Estandarización
monthly["spread_z"] = zscore(monthly["spread_avg_w"])
monthly["mkt_ret_z"] = zscore(monthly["mkt_ret_w"])
monthly["credit_level_z"] = zscore(monthly["credit_level_w"])

# QA informativo
print("\n=== BASE MENSUAL ===")
print(monthly.head())
print("\nProporción de missing:")
print(monthly[["spread_avg", "mkt_ret", "credit_level"]].isna().mean().round(4))

# Export base
monthly.to_csv(FIGURES / "fig_16_monthly_aggregated_data_raw_and_clean.csv", index=False)

# Base final para gráficos
monthly_plot = monthly.loc[monthly["keep_for_plots"] == 1].copy()

## 17) Outputs documentales para la Sección de Datos — panel final, embudo muestral y cobertura analítica

Esta sección consolida un conjunto de outputs documentales diseñados específicamente 
para la redacción de la Sección 3 (Datos) del trabajo.

El objetivo es garantizar la trazabilidad completa del proceso de construcción del panel, 
desde el universo inicial de datos hasta la muestra final utilizada en la estimación 
econométrica, así como documentar de manera transparente la cobertura de las variables 
relevantes.

---

## Objetivos

Los outputs generados en esta sección cumplen cuatro funciones principales:

1. **Documentar el embudo muestral**

   Se reconstruye el proceso de filtrado de datos, desde las bases iniciales hasta el panel 
   final, permitiendo cuantificar la pérdida de observaciones en cada etapa.

2. **Medir la cobertura por bloques de variables**

   Se evalúa la disponibilidad conjunta de variables clave (spreads, shocks agregados, 
   variables firm-level), identificando restricciones empíricas relevantes para la estimación.

3. **Caracterizar la estructura del panel**

   Se resumen las dimensiones fundamentales del panel firma–mes:

   - Número de observaciones
   - Número de firmas
   - Número de instrumentos (bonos)
   - Número de períodos
   - Rango temporal efectivo

4. **Generar tablas reproducibles para la tesis**

   Se construyen tablas exportables que serán utilizadas directamente en la redacción de la 
   subsección de datos, en particular en lo relativo a:

   - Cobertura de la muestra
   - Filtros aplicados
   - Limitaciones empíricas

---

## Enfoque metodológico

El enfoque adoptado en esta sección es deliberadamente documental y no inferencial. 
El objetivo no es realizar análisis estadístico, sino garantizar:

- Reproducibilidad del pipeline de datos  
- Transparencia en la construcción de la muestra  
- Consistencia entre código, outputs y documento final  

En particular, cada etapa del proceso se resume mediante estadísticas descriptivas 
estandarizadas, permitiendo reconstruir el denominado *embudo muestral* del trabajo.

---

## Outputs generados

Los outputs se almacenan en las siguientes rutas:

- `outputs/tables/data_section/`  
  → Tablas finales utilizadas en la tesis

- `outputs/qa/data_section/`  
  → Outputs intermedios de validación y diagnóstico

Entre los principales outputs se incluyen:

- Resumen por etapa del pipeline de datos  
- Métricas de cobertura por bloque de variables  
- Caracterización estructural del panel final  

Estos archivos permiten asegurar la coherencia entre el pipeline de datos y la 
documentación presentada en la tesis.

---

## Nota metodológica

Las métricas reportadas en esta sección deben interpretarse como descripciones del proceso 
de construcción de datos y no como estadísticas poblacionales.

Asimismo, la pérdida de observaciones a lo largo del pipeline responde a restricciones de 
disponibilidad de datos y no a decisiones de muestreo discrecionales.

Este punto es fundamental para contextualizar las limitaciones empíricas del análisis.

In [ ]:
# ==========================================================
# 17A. CONFIG + HELPERS | OUTPUTS DOCUMENTALES SECCIÓN DATOS
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------------------------------
# Definición de directorios de salida
# -------------------------------------------------

DATASEC_DIR = OUTPUTS / "tables" / "data_section"   # tablas finales para tesis
DATASEC_QA_DIR = QA / "data_section"                # outputs de QA intermedios

# Crear directorios si no existen
for p in [DATASEC_DIR, DATASEC_QA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# HELPERS DE CARGA
# -------------------------------------------------

def load_first_existing(paths):
    """
    Carga el primer archivo existente de una lista de paths.
    Soporta formatos parquet y csv.
    """
    for p in paths:
        p = Path(p)

        if p.exists():
            if p.suffix == ".parquet":
                return pd.read_parquet(p), p

            elif p.suffix == ".csv":
                return pd.read_csv(p), p

    return None, None


# -------------------------------------------------
# HELPERS DE IDENTIFICACIÓN DE COLUMNAS
# -------------------------------------------------

def first_existing_col(df: pd.DataFrame, candidates: list[str]):
    """
    Devuelve la primera columna existente en el DataFrame
    a partir de una lista de nombres candidatos (case-insensitive).
    """
    cols = {str(c).strip().lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]

    return None


def to_eom_safe(s):
    """
    Convierte una serie de fechas a end-of-month (EOM),
    manejando errores de parsing.
    """
    return pd.to_datetime(s, errors="coerce") \
             .dt.to_period("M") \
             .dt.to_timestamp("M")


# -------------------------------------------------
# FUNCIÓN PRINCIPAL — RESUMEN POR ETAPA
# -------------------------------------------------

def summarize_stage(stage, df, path=None):
    """
    Construye un resumen estructural de una etapa del pipeline de datos.
    
    Métricas:
    - Número de filas
    - Número de firmas
    - Número de bonos
    - Número de períodos
    - Rango temporal
    """

    # Caso: dataset inexistente
    if df is None:
        return {
            "stage": stage,
            "path": None,
            "n_rows": np.nan,
            "n_firms": np.nan,
            "n_bonds": np.nan,
            "n_periods": np.nan,
            "date_min": pd.NaT,
            "date_max": pd.NaT,
        }

    # Identificación flexible de columnas clave
    date_col = first_existing_col(df, ["date", "Date", "period_end"])
    issuer_col = first_existing_col(df, ["issuer", "Issuer", "ticker", "Ticker", "ric", "RIC"])
    bond_col = first_existing_col(df, ["bond_ric", "Preferred RIC", "RIC"])

    # Inicialización del output
    out = {
        "stage": stage,
        "path": str(path) if path is not None else None,
        "n_rows": len(df),
        "n_firms": df[issuer_col].nunique(dropna=True) if issuer_col else np.nan,
        "n_bonds": df[bond_col].nunique(dropna=True) if bond_col else np.nan,
        "n_periods": np.nan,
        "date_min": pd.NaT,
        "date_max": pd.NaT,
    }

    # Cálculo de métricas temporales
    if date_col:
        d = to_eom_safe(df[date_col])
        out["n_periods"] = d.nunique(dropna=True)
        out["date_min"] = d.min()
        out["date_max"] = d.max()

    return out


# -------------------------------------------------
# CONFIRMACIÓN DE ENTORNO
# -------------------------------------------------

print("✔ Directorio de outputs documentales:", DATASEC_DIR)

In [ ]:
# ==========================================================
# 17B. EMBUDO MUESTRAL / PIPELINE DEL PANEL
# ==========================================================

stage_files = {
    "01_universe_firms": [
        CLEAN / "universe_firms.parquet",
        CLEAN / "universe_firms.csv",
        PROJECT_ROOT / "data" / "inputs" / "empresas_tickers_test.csv",
    ],
    "02_bonds_universe_full": [
        CLEAN / "bonds_universe_full.parquet",
        CLEAN / "bonds_universe_full.csv",
    ],
    "03_bond_month_clean": [
        INTERIM / "bonds_monthly_spreads.parquet",
        INTERIM / "bonds_monthly_spreads.csv",
        CLEAN / "bonds_monthly_spreads.parquet",
        CLEAN / "bonds_monthly_spreads.csv",
    ],
    "04_firm_month_spreads": [
        INTERIM / "bonds_monthly_spreads_firmlevel.parquet",
        INTERIM / "bonds_monthly_spreads_firmlevel.csv",
    ],
    "05_panel_block1_base_mapping_equity": [
        INTERIM / "panel_master_block1_base_mapping_equity.parquet",
    ],
    "06_panel_block2_fund_mp": [
        INTERIM / "panel_master_block2_fund_mp.parquet",
    ],
    "07_panel_block3_crc_macro_sector": [
        INTERIM / "panel_master_block3_crc_macro_sector.parquet",
    ],
    "08_panel_master_final": [
        CLEAN / "panel_master.parquet",
    ],
}

stage_rows = []
stage_dfs = {}

for stage, candidates in stage_files.items():
    df, path = load_first_existing(candidates)
    stage_dfs[stage] = (df, path)
    stage_rows.append(summarize_stage(stage, df, path))

stage_funnel_df = pd.DataFrame(stage_rows)

stage_funnel_path = DATASEC_DIR / "02_panel_stage_funnel.csv"
stage_funnel_df.to_csv(stage_funnel_path, index=False)

print("✔ Stage funnel guardado en:", stage_funnel_path)
display(stage_funnel_df)

In [ ]:
# ==========================================================
# 17C. COBERTURA POR BLOQUE E INTERSECCIONES ANALÍTICAS
# ==========================================================

panel, panel_path = stage_dfs["08_panel_master_final"]

if panel is None:
    raise FileNotFoundError("No encontré panel_master para construir outputs documentales.")

panel = panel.copy()
panel["date"] = to_eom_safe(panel["date"])

# -------------------------------------------------
# Variable dependiente (fallback robusto)
# -------------------------------------------------
dep_candidates = ["oas_mean", "spread_mean_bps", "oas_median"]
dep_var = next((c for c in dep_candidates if c in panel.columns), None)

if dep_var is None:
    raise KeyError(
        "No encontré una variable dependiente candidata entre: "
        "oas_mean, spread_mean_bps, oas_median"
    )

# -------------------------------------------------
# Definición de bloques analíticos
# -------------------------------------------------
block_candidates = {
    "dependiente": [dep_var, "n_bonds", "n_bonds_oas"],
    "equity": ["beta_252", "ivol_252"],
    "mercado_macro": ["mkt_ret", "mkt_vol_60d", "policy_rate", "term_spread_10y_2y", "indpro_yoy"],
    "credito_agregado": ["credit_level", "credit_slope", "crc_level_beta", "crc_slope_beta"],
    "fundamentals_core": ["leverage", "cash_to_assets", "log_assets", "rollover_12m", "residual_maturity_mean"],
    "market_power": ["market_share_w"],
    "cds_aux": ["cds_5y_mean"],
}

# Mantener únicamente columnas existentes en el panel
block_cols = {
    block: [c for c in cols if c in panel.columns]
    for block, cols in block_candidates.items()
}

# -------------------------------------------------
# Coverage a nivel variable
# -------------------------------------------------
var_rows = []

for block, cols in block_cols.items():
    for c in cols:
        s = panel[c]

        var_rows.append({
            "block": block,
            "variable": c,
            "non_null": int(s.notna().sum()),
            "null": int(s.isna().sum()),
            "null_pct": round(s.isna().mean() * 100, 2),
            "n_firms_with_data": panel.loc[s.notna(), "issuer"].nunique() if "issuer" in panel.columns else np.nan,
            "n_periods_with_data": panel.loc[s.notna(), "date"].nunique(),
        })

coverage_var_df = pd.DataFrame(var_rows).sort_values(["block", "null_pct", "variable"])

coverage_var_path = DATASEC_DIR / "02_block_coverage_variable_level.csv"
coverage_var_df.to_csv(coverage_var_path, index=False)

# -------------------------------------------------
# Coverage a nivel bloque
# -------------------------------------------------
block_rows = []

for block, cols in block_cols.items():
    if not cols:
        continue

    mask_any = panel[cols].notna().any(axis=1)
    mask_all = panel[cols].notna().all(axis=1)

    block_rows.append({
        "block": block,
        "n_variables": len(cols),
        "n_obs_any": int(mask_any.sum()),
        "n_obs_all": int(mask_all.sum()),
        "pct_obs_any": round(mask_any.mean() * 100, 2),
        "pct_obs_all": round(mask_all.mean() * 100, 2),
        "n_firms_any": panel.loc[mask_any, "issuer"].nunique() if "issuer" in panel.columns else np.nan,
        "n_firms_all": panel.loc[mask_all, "issuer"].nunique() if "issuer" in panel.columns else np.nan,
        "n_periods_any": panel.loc[mask_any, "date"].nunique(),
        "n_periods_all": panel.loc[mask_all, "date"].nunique(),
        "variables": ", ".join(cols),
    })

coverage_block_df = pd.DataFrame(block_rows).sort_values("block")

coverage_block_path = DATASEC_DIR / "02_block_coverage_block_level.csv"
coverage_block_df.to_csv(coverage_block_path, index=False)

print("✔ Coverage variable-level:", coverage_var_path)
print("✔ Coverage block-level:", coverage_block_path)
display(coverage_block_df)

# -------------------------------------------------
# Embudo secuencial para redacción de la sección Datos
# -------------------------------------------------
seq_defs = [
    ("panel_base", []),
    ("dep_only", block_cols["dependiente"]),
    ("dep_plus_equity", block_cols["dependiente"] + block_cols["equity"]),
    ("dep_plus_equity_macro", block_cols["dependiente"] + block_cols["equity"] + block_cols["mercado_macro"]),
    (
        "dep_plus_equity_macro_credit",
        block_cols["dependiente"] + block_cols["equity"] + block_cols["mercado_macro"] + block_cols["credito_agregado"],
    ),
    (
        "core_without_market_power",
        block_cols["dependiente"] + block_cols["equity"] + block_cols["mercado_macro"] + block_cols["credito_agregado"] + block_cols["fundamentals_core"],
    ),
    (
        "core_with_market_power",
        block_cols["dependiente"] + block_cols["equity"] + block_cols["mercado_macro"] + block_cols["credito_agregado"] + block_cols["fundamentals_core"] + block_cols["market_power"],
    ),
]

seq_rows = []

for stage, cols in seq_defs:
    cols = [c for c in cols if c in panel.columns]

    if len(cols) == 0:
        mask = pd.Series(True, index=panel.index)
    else:
        mask = panel[cols].notna().all(axis=1)

    seq_rows.append({
        "stage": stage,
        "n_required_vars": len(cols),
        "n_obs": int(mask.sum()),
        "n_firms": panel.loc[mask, "issuer"].nunique() if "issuer" in panel.columns else np.nan,
        "n_periods": panel.loc[mask, "date"].nunique(),
        "required_vars": ", ".join(cols),
    })

sample_funnel_df = pd.DataFrame(seq_rows)

sample_funnel_path = DATASEC_DIR / "02_sample_funnel_sequential.csv"
sample_funnel_df.to_csv(sample_funnel_path, index=False)

print("✔ Sample funnel guardado en:", sample_funnel_path)
display(sample_funnel_df)

In [ ]:
# ==========================================================
# 17D. ESTRUCTURA DEL PANEL: DESBALANCE, COBERTURA POR FIRMA Y POR MES
# ==========================================================

panel = panel.copy()

# -------------------------------------------------
# Estructura por firma
# -------------------------------------------------
firm_panel_df = (
    panel.groupby("issuer", as_index=False)
         .agg(
             n_obs=("date", "size"),
             first_date=("date", "min"),
             last_date=("date", "max"),
             dep_non_null=(dep_var, lambda s: int(s.notna().sum())),
             dep_null_pct=(dep_var, lambda s: round(s.isna().mean() * 100, 2)),
         )
         .sort_values(["n_obs", "dep_non_null"], ascending=[False, False])
)

firm_panel_path = DATASEC_DIR / "02_panel_structure_by_firm.csv"
firm_panel_df.to_csv(firm_panel_path, index=False)

# -------------------------------------------------
# Estructura por mes
# -------------------------------------------------
month_panel_df = (
    panel.groupby("date", as_index=False)
         .agg(
             n_obs=("issuer", "size"),
             n_firms=("issuer", "nunique"),
             dep_non_null=(dep_var, lambda s: int(s.notna().sum())),
             dep_null_pct=(dep_var, lambda s: round(s.isna().mean() * 100, 2)),
         )
         .sort_values("date")
)

month_panel_path = DATASEC_DIR / "02_panel_structure_by_month.csv"
month_panel_df.to_csv(month_panel_path, index=False)

# -------------------------------------------------
# Resumen sintético del panel
# -------------------------------------------------
panel_summary = pd.DataFrame([{
    "panel_path": str(panel_path),
    "n_obs_total": len(panel),
    "n_firms_total": panel["issuer"].nunique() if "issuer" in panel.columns else np.nan,
    "n_periods_total": panel["date"].nunique(),
    "date_min": panel["date"].min(),
    "date_max": panel["date"].max(),
    "dep_var_used": dep_var,
    "dep_non_null_total": int(panel[dep_var].notna().sum()),
    "dep_null_pct_total": round(panel[dep_var].isna().mean() * 100, 2),
    "obs_per_firm_mean": round(firm_panel_df["n_obs"].mean(), 2),
    "obs_per_firm_median": round(firm_panel_df["n_obs"].median(), 2),
    "obs_per_firm_min": int(firm_panel_df["n_obs"].min()),
    "obs_per_firm_max": int(firm_panel_df["n_obs"].max()),
}])

panel_summary_path = DATASEC_DIR / "02_panel_summary.csv"
panel_summary.to_csv(panel_summary_path, index=False)

print("✔ Panel summary:", panel_summary_path)
print("✔ Estructura por firma:", firm_panel_path)
print("✔ Estructura por mes:", month_panel_path)
display(panel_summary)

In [ ]:
# ==========================================================
# 17E. REPORTE CORTO EN MARKDOWN
# ==========================================================

panel_summary_row = panel_summary.iloc[0].to_dict()

report_lines = [
    "# Resumen documental — Notebook 02 (construcción de panel)",
    "",
    "## Panel final",
    f"- Panel final: {panel_summary_row['n_obs_total']} observaciones firma-mes.",
    f"- Firmas: {panel_summary_row['n_firms_total']}.",
    f"- Períodos mensuales: {panel_summary_row['n_periods_total']}.",
    f"- Rango temporal: {panel_summary_row['date_min']} -> {panel_summary_row['date_max']}.",
    f"- Variable dependiente usada en el reporte: `{panel_summary_row['dep_var_used']}`.",
    "",
    "## Archivos principales",
    f"- Funnel de etapas: `{stage_funnel_path.name}`",
    f"- Funnel secuencial de muestra: `{sample_funnel_path.name}`",
    f"- Coverage por variable: `{coverage_var_path.name}`",
    f"- Coverage por bloque: `{coverage_block_path.name}`",
    f"- Estructura por firma: `{firm_panel_path.name}`",
    f"- Estructura por mes: `{month_panel_path.name}`",
    "",
    "## Uso sugerido en la tesis",
    "- 3.6 Integración del panel final y trazabilidad del pipeline: stage funnel + panel summary.",
    "- 3.7 Cobertura efectiva, filtros muestrales y disponibilidad por bloque: sample funnel + coverage block-level.",
    "- 3.8 Limitaciones empíricas: estructura no balanceada + cobertura diferencial por bloque.",
]

report_text = "\n".join(report_lines)

report_path = DATASEC_QA_DIR / "02_panel_report.md"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print("\n✔ Reporte markdown guardado en:", report_path)